In [ ]:
c# AUDIT: Runtime state, CSV files, figures, runtime variables
import os, glob

print("=" * 60)
print("CSV FILES ON DISK")
print("=" * 60)
for f in sorted(glob.glob("*.csv")):
    lines = sum(1 for _ in open(f))
    print(f"  {f}: {lines} lines (including header)")

print("\n" + "=" * 60)
print("IMAGE / PDF FILES")
print("=" * 60)
for f in sorted(glob.glob("*.png") + glob.glob("*.pdf")):
    print(f"  {f}: {os.path.getsize(f)/1024:.1f} KB")

if not glob.glob("*.csv"):
    print("  No CSV files found")
if not glob.glob("*.png") and not glob.glob("*.pdf"):
    print("  No image/PDF files found")

print("\n" + "=" * 60)
print("DATA FILES")
print("=" * 60)
for f in sorted(glob.glob("data/*")):
    print(f"  {f}: {os.path.getsize(f)/1e6:.1f} MB")

print("\n" + "=" * 60)
print("RUNTIME VARIABLES")
print("=" * 60)
import sys
key_vars = ['torch', 'pd', 'np', 'df_all', 'fd11', 'DEVICE',
            'YEAR_FEATURES', 'SmallCNN', 'LargerCNN', 'train_model',
            'run_one_config', 'ClassificationCNN', 'train_classifier']
for v in key_vars:
    present = v in dir() or v in sys.modules
    print(f"  {v}: {'loaded' if present else 'NOT loaded'}")


In [ ]:

# ============================================================
# AUDIT: Current state of all experiments, results, and files
# ============================================================
import os, glob
import pandas as pd

print("="*70)
print("CSV RESULT FILES")
print("="*70)
for f in sorted(glob.glob("*.csv")):
    try:
        df = pd.read_csv(f)
        print(f"\n{f}: {len(df)} rows, {len(df.columns)} columns")
        print(f"  Columns: {list(df.columns)[:8]}...")
        if "experiment" in df.columns:
            print(f"  Experiments: {df['experiment'].value_counts().to_dict()}")
        if "horizon" in df.columns:
            print(f"  Horizons: {sorted(df['horizon'].unique())}")
        if "model" in df.columns:
            print(f"  Models: {sorted(df['model'].unique())}")
        if "features" in df.columns:
            print(f"  Features: {sorted(df['features'].unique())}")
    except Exception as e:
        print(f"\n{f}: ERROR reading - {e}")

print("\n" + "="*70)
print("IMAGE/PDF FILES")
print("="*70)
for ext in ["*.png", "*.pdf"]:
    for f in sorted(glob.glob(ext)):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

print("\n" + "="*70)
print("DATA FILES")
print("="*70)
for f in sorted(glob.glob("data/*")):
    size_mb = os.path.getsize(f) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

print("\n" + "="*70)
print("ALL OTHER FILES")
print("="*70)
for f in sorted(glob.glob("*")):
    if not f.endswith(('.csv', '.png', '.pdf')) and os.path.isfile(f):
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f}: {size_kb:.1f} KB")

# Check for LaTeX files
print("\n" + "="*70)
print("LATEX FILES (searching recursively)")
print("="*70)
tex_files = glob.glob("**/*.tex", recursive=True)
bib_files = glob.glob("**/*.bib", recursive=True)
if tex_files or bib_files:
    for f in tex_files + bib_files:
        print(f"  {f}")
else:
    print("  No .tex or .bib files found in the Colab working directory")


# CNN Soil Moisture Forecasting — TxSON Dataset

Extends single-station, single-year CNN forecasting to multiple years and stations using the Texas Soil Observation Network (TxSON) dataset. Compares architectures, tests input features, and evaluates both regression and 12-class classification approaches.

## Setup

In [ ]:
import torch, random, numpy as np, os, time

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", DEVICE, "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import urllib.request, os

BASE = "https://raw.githubusercontent.com/kiat/tx-soil-moisture/main/datasets/txson33_complete_for_modeling"
files = ["TxSON33_modeling_complete.parquet", "data_quality_summary.csv", "README.md"]

os.makedirs("data", exist_ok=True)
for fn in files:
    dest = f"data/{fn}"
    if not os.path.exists(dest):
        print("downloading", fn, "...")
        urllib.request.urlretrieve(f"{BASE}/{fn}", dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f"{fn}: {size_mb:.1f} MB")

In [ ]:
import pandas as pd

df_all = pd.read_parquet("data/TxSON33_modeling_complete.parquet")
print("full dataset shape:", df_all.shape)

fd11 = df_all[df_all["Station"] == "FD11"].copy()
fd11["Timestamp"] = pd.to_datetime(fd11["Timestamp"])
fd11 = fd11.sort_values("Timestamp").reset_index(drop=True)
print("FD11 rows:", len(fd11), "range:", fd11["Timestamp"].min(), "to", fd11["Timestamp"].max())

fd11["year"] = fd11["Timestamp"].dt.year
rows = []
for yr, g in fd11.groupby("year"):
    expected_hours = 8784 if (yr % 4 == 0 and (yr % 100 != 0 or yr % 400 == 0)) else 8760
    swc_origin = g["SWC_20_Origin"].value_counts(normalize=True) * 100
    ppt_origin = g["Ppt_Origin"].value_counts(normalize=True) * 100
    rows.append({
        "year": yr,
        "n_rows": len(g),
        "expected_hours": expected_hours,
        "coverage_pct": round(len(g) / expected_hours * 100, 1),
        "SWC20_observed_pct": round(swc_origin.get("observed", 0.0), 2),
        "SWC20_pipeline_imputed_pct": round(swc_origin.get("pipeline_imputed", 0.0), 2),
        "SWC20_modeling_completion_pct": round(swc_origin.get("modeling_completion", 0.0), 2),
        "Ppt_observed_pct": round(ppt_origin.get("observed", 0.0), 2),
        "Ppt_pipeline_imputed_pct": round(ppt_origin.get("pipeline_imputed", 0.0), 2),
        "Ppt_modeling_completion_pct": round(ppt_origin.get("modeling_completion", 0.0), 2),
    })

year_summary = pd.DataFrame(rows).set_index("year")
pd.set_option("display.width", 140)
print(year_summary)

In [ ]:
full_years = [y for y in range(2015, 2025)]  # 2014 and 2025 are partial, excluded
annual_ppt = fd11[fd11["year"].isin(full_years)].groupby("year")["Ppt"].sum().round(1)
annual_swc_mean = fd11[fd11["year"].isin(full_years)].groupby("year")["SWC_20"].mean().round(4)
ranking = pd.DataFrame({"annual_Ppt_mm": annual_ppt, "mean_SWC_20": annual_swc_mean}).sort_values("annual_Ppt_mm")
print(ranking)

## Data Selection

Station FD11, using every complete calendar year (2015–2024). Three held-out test years are chosen by annual precipitation to span the wet/dry range:

- **Driest:** 2023 (203 mm)
- **Median:** 2020 (635 mm)
- **Wettest:** 2018 (856 mm)

Validation is fixed at 2019 across all folds. Training uses the remaining 8 years.

In [ ]:
TEST_YEARS = [2023, 2020, 2018]
VAL_YEAR = 2019
ALL_YEARS = list(range(2015, 2025))

def windows_per_year(year, window_size, horizon):
    hours = 8784 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 8760
    n = hours - window_size - horizon + 1
    return max(n, 0)

for window_size in [12, 24, 48]:
    for horizon in [24, 48, 72, 168]:
        print(f"--- window={window_size}h, horizon={horizon}h ---")
        for test_year in TEST_YEARS:
            train_years = [y for y in ALL_YEARS if y not in (test_year, VAL_YEAR)]
            n_train = sum(windows_per_year(y, window_size, horizon) for y in train_years)
            n_val = windows_per_year(VAL_YEAR, window_size, horizon)
            n_test = windows_per_year(test_year, window_size, horizon)
            print(f"  test={test_year}: train={n_train:,} windows ({len(train_years)} years), "
                  f"val={n_val:,} windows, test={n_test:,} windows")

## Input Window Construction

Windows are built independently within each calendar year — no window spans a year boundary, preventing leakage by construction.

In [ ]:
import numpy as np

TARGET_COL = "SWC_20"

def prepare_year_features(df, year):
    g = df[df["year"] == year].copy()
    g = g.sort_values("Timestamp").reset_index(drop=True)
    # defensive clip, same reasoning as the single-year notebook: log1p is undefined below -1
    g["Ppt"] = g["Ppt"].fillna(0.0).clip(lower=0.0)
    g["Ppt_log"] = np.log1p(g["Ppt"])
    return g

def make_windows_for_year(g, feature_cols, window_size, horizon):
    arr = g[feature_cols].to_numpy(dtype=np.float32)
    target = g[TARGET_COL].to_numpy(dtype=np.float32)
    n = len(g) - window_size - horizon + 1
    if n <= 0:
        return np.empty((0, window_size, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.float32)
    X = np.stack([arr[i:i + window_size] for i in range(n)])
    y = np.array([target[i + window_size + horizon - 1] for i in range(n)], dtype=np.float32)
    return X, y

# quick sanity check: confirm target timestamp is exactly `horizon` hours after the window's last hour
g2018 = prepare_year_features(fd11, 2018)
X_demo, y_demo = make_windows_for_year(g2018, ["SWC_20"], window_size=24, horizon=24)
ts = g2018["Timestamp"].to_numpy()
i = 0
gap = (ts[i + 24 + 24 - 1] - ts[i + 24 - 1]).astype("timedelta64[h]")
print("first input hour:", ts[i], " last input hour:", ts[i + 23], " target hour:", ts[i + 24 + 24 - 1])
print("gap from window end to target:", gap)
print("X shape:", X_demo.shape, "y shape:", y_demo.shape)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

YEAR_FEATURES = {y: prepare_year_features(fd11, y) for y in ALL_YEARS}

def build_fold(test_year, feature_cols, window_size, horizon, batch_size, seed=SEED):
    train_years = [y for y in ALL_YEARS if y not in (test_year, VAL_YEAR)]

    # fit scaler on pooled training years only
    scaler = MinMaxScaler()
    scaler.fit(pd.concat([YEAR_FEATURES[y][feature_cols] for y in train_years], axis=0))

    def scaled_windows(years):
        Xs, ys = [], []
        for y in years:
            g = YEAR_FEATURES[y].copy()
            g[feature_cols] = scaler.transform(g[feature_cols])
            Xy, yy = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(Xy):
                Xs.append(Xy); ys.append(yy)
        return np.concatenate(Xs), np.concatenate(ys)

    Xtr, ytr = scaled_windows(train_years)
    Xv, yv = scaled_windows([VAL_YEAR])
    Xte, yte = scaled_windows([test_year])

    g = torch.Generator().manual_seed(seed)
    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g if shuffle else None)

    return loader(Xtr, ytr, True), loader(Xv, yv, False), loader(Xte, yte, False), (Xtr, ytr, Xv, yv, Xte, yte), scaler

# sanity check on one fold
trl, vl, tel, raw, _scaler_check = build_fold(2023, ["SWC_20"], 24, 24, 32)
Xtr, ytr, Xv, yv, Xte, yte = raw
print("train:", Xtr.shape, "val:", Xv.shape, "test:", Xte.shape)
print("train range:", Xtr.min(), Xtr.max(), " val range:", Xv.min(), Xv.max(), " (val outside [0,1] is fine -- proves no refit)")

## Inverse Scaling

In [ ]:
def inverse_transform_target(scaler, feature_cols, values, target_col="SWC_20"):
    """Inverse-transforms only the target column, using this scaler's own per-column
    min_/scale_ rather than assuming column order -- safe regardless of how many other
    features (e.g. Ppt_log) were included when the scaler was fit."""
    idx = feature_cols.index(target_col)
    values = np.asarray(values, dtype=np.float64)
    return (values - scaler.min_[idx]) / scaler.scale_[idx]

# Verify the helper against sklearn's own inverse_transform before trusting it elsewhere.
_feature_cols_check = ["SWC_20", "Ppt_log"]
_trl_v, _vl_v, _tel_v, (_, _, _, _, _Xte_v, _yte_v), _scaler_v = build_fold(
    2023, _feature_cols_check, 24, 24, batch_size=128)

_manual = inverse_transform_target(_scaler_v, _feature_cols_check, _yte_v)
_full_inverse = _scaler_v.inverse_transform(np.stack([_yte_v, np.zeros_like(_yte_v)], axis=1))[:, 0]
print("helper matches scaler.inverse_transform exactly?", np.allclose(_manual, _full_inverse))
print("max abs difference:", np.abs(_manual - _full_inverse).max())

## Baseline Models

In [ ]:
def compute_metrics(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    rse = np.sum((y_pred - y_true) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + eps)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_pred) > 1e-10 else float("nan")
    return {"MSE": mse, "MAE": mae, "SMAPE": smape, "RSE": rse, "CORR": corr}

# verify against known cases before trusting on real data
perfect = compute_metrics([1.0, 2.0, 3.0], [1.0, 2.0, 3.0])
assert abs(perfect["MSE"]) < 1e-9 and abs(perfect["CORR"] - 1.0) < 1e-9
print("metric sanity check (perfect prediction):", perfect)

def persistence_predict(X, target_idx):
    # last observed value in the window, per sample
    return X[:, -1, target_idx]

def moving_average_predict(X, target_idx):
    # mean of the target column across the whole input window
    return X[:, :, target_idx].mean(axis=1)

In [ ]:
# CORR invariance check: Pearson correlation must be unaffected by inverse-scaling,
# since it's invariant to any affine transform of either variable. Uses the same fold
# and scaler already built above.
_preds_scaled_check = persistence_predict(_Xte_v, _feature_cols_check.index("SWC_20"))
_preds_original_check = inverse_transform_target(_scaler_v, _feature_cols_check, _preds_scaled_check)
_corr_scaled = compute_metrics(_yte_v, _preds_scaled_check)["CORR"]
_corr_original = compute_metrics(_manual, _preds_original_check)["CORR"]
print(f"CORR in scaled space:   {_corr_scaled:.10f}")
print(f"CORR in original scale: {_corr_original:.10f}")
print("CORR unchanged up to floating-point precision?", np.isclose(_corr_scaled, _corr_original, atol=1e-8))

## CNN Architecture

In [ ]:
import torch.nn as nn

class SmallCNN(nn.Module):
    def __init__(self, num_features, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv1d(num_features, 8, kernel_size, padding=p)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(8, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

class LargerCNN(nn.Module):
    def __init__(self, num_features, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv1d(num_features, 16, kernel_size, padding=p)
        self.conv2 = nn.Conv1d(16, 32, kernel_size, padding=p)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("SmallCNN params (F=1):", count_params(SmallCNN(1)))
print("LargerCNN params (F=1):", count_params(LargerCNN(1)))

## Mathematical Formulation

## Training Procedure

In [ ]:
def train_model(model, train_loader, val_loader, epochs=40, patience=8, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []
    best_val, best_state, bad_epochs, stopped_epoch = float("inf"), None, 0, epochs

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_losses.append(running / len(train_loader.dataset))

        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                running += criterion(model(xb), yb).item() * len(xb)
        val_loss = running / len(val_loader.dataset)
        val_losses.append(val_loss)

        if val_loss < best_val - 1e-7:
            best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                stopped_epoch = epoch + 1
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses, stopped_epoch

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

# quick real-run smoke test: one small CNN fold, timed
t0 = time.time()
model = SmallCNN(1)
model, trl_hist, vll_hist, stopped = train_model(model, trl, vl, epochs=40, patience=8)
preds, trues = evaluate_model(model, tel)
print(f"trained in {time.time()-t0:.1f}s, stopped at epoch {stopped}")
print(compute_metrics(trues, preds))

In [ ]:
WINDOW_SIZES = [12, 24, 48]
HORIZONS = [24, 48, 72, 168]
FEATURE_SETS = {"SWC_20 only": ["SWC_20"], "SWC_20 + Ppt_log": ["SWC_20", "Ppt_log"]}
MODEL_CLASSES = {"small_cnn": SmallCNN, "larger_cnn": LargerCNN}
RESULTS_CSV = "results.csv"

def run_one_config(test_year, window_size, horizon, feat_name, feature_cols, model_name, seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    target_idx = feature_cols.index("SWC_20")
    trl, vl, tel, (Xtr, ytr, Xv, yv, Xte, yte), scaler = build_fold(
        test_year, feature_cols, window_size, horizon, batch_size=128, seed=seed)

    row = {"test_year": test_year, "window_size": window_size, "horizon": horizon,
           "features": feat_name, "model": model_name, "seed": seed,
           "n_train": len(ytr), "n_val": len(yv), "n_test": len(yte)}

    if model_name == "persistence":
        preds_scaled = persistence_predict(Xte, target_idx)
        row["epochs_run"] = 0
    elif model_name == "moving_average":
        preds_scaled = moving_average_predict(Xte, target_idx)
        row["epochs_run"] = 0
    else:
        t0 = time.time()
        model = MODEL_CLASSES[model_name](num_features=len(feature_cols))
        model, trh, vlh, stopped = train_model(model, trl, vl, epochs=30, patience=6)
        preds_out, trues_check = evaluate_model(model, tel)
        preds_scaled = preds_out.flatten()
        row["epochs_run"] = stopped
        row["train_seconds"] = round(time.time() - t0, 1)
        row["final_train_loss"] = trh[-1]
        row["final_val_loss"] = vlh[-1]
        row["best_val_loss"] = min(vlh)

    yte_original = inverse_transform_target(scaler, feature_cols, yte)
    preds_original = inverse_transform_target(scaler, feature_cols, preds_scaled)

    metrics = compute_metrics(yte_original, preds_original)
    row.update(metrics)
    return row

def run_chunk(test_year):
    rows = []
    t_chunk = time.time()
    for window_size in WINDOW_SIZES:
        for horizon in HORIZONS:
            for feat_name, feature_cols in FEATURE_SETS.items():
                for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
                    r = run_one_config(test_year, window_size, horizon, feat_name, feature_cols, model_name)
                    rows.append(r)
    df_chunk = pd.DataFrame(rows)
    header = not os.path.exists(RESULTS_CSV)
    df_chunk.to_csv(RESULTS_CSV, mode="a", header=header, index=False)
    print(f"test_year={test_year}: {len(rows)} configs done in {time.time()-t_chunk:.1f}s, appended to {RESULTS_CSV}")
    return df_chunk

print("runner defined (metrics on original SWC_20 scale, records both final_val_loss and best_val_loss)")

In [ ]:

# Quick feature extension — adds engineered columns to YEAR_FEATURES
# without running any experiments. Needed before classification cells.
for y in ALL_YEARS:
    g = YEAR_FEATURES[y]
    for col in ["Tair", "T_5", "T_10", "T_20", "T_50",
                "SWC_5", "SWC_10", "SWC_50", "RH", "Srad"]:
        if col in g.columns:
            g[col] = g[col].ffill().bfill().fillna(0.0)
    g["Ppt_3h"]  = g["Ppt"].rolling(3,  min_periods=1).sum()
    g["Ppt_6h"]  = g["Ppt"].rolling(6,  min_periods=1).sum()
    g["Ppt_12h"] = g["Ppt"].rolling(12, min_periods=1).sum()
    g["Ppt_24h"] = g["Ppt"].rolling(24, min_periods=1).sum()
    doy = g["Timestamp"].dt.dayofyear
    g["sin_doy"] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    g["cos_doy"] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

print("YEAR_FEATURES extended with engineered columns")
print(f"Columns now: {list(YEAR_FEATURES[2020].columns[:5])}...{list(YEAR_FEATURES[2020].columns[-5:])}")
print(f"sin_doy range: [{YEAR_FEATURES[2020]['sin_doy'].min():.4f}, {YEAR_FEATURES[2020]['sin_doy'].max():.4f}]")


In [ ]:
import os
if os.path.exists("results.csv"):
    os.remove("results.csv")
    print("removed stale scaled-space results.csv -- rerunning all three folds with original-scale metrics")
else:
    print("no existing results.csv found")

In [ ]:
df_2023 = run_chunk(2023)

In [ ]:
df_2020 = run_chunk(2020)

In [ ]:
df_2018 = run_chunk(2018)

## Single-Station Regression Results

In [ ]:
full = pd.read_csv("results.csv")
baseline_rows = full[full["model"].isin(["persistence", "moving_average"])]
cnn_rows = full[full["model"].isin(["small_cnn", "larger_cnn"])]

# check: are the two feature-set rows for each (test_year, window_size, horizon, model) identical for baselines?
dup_check = baseline_rows.pivot_table(index=["test_year","window_size","horizon","model"],
                                        columns="features", values="MSE")
identical = (dup_check["SWC_20 only"] == dup_check["SWC_20 + Ppt_log"]).all()
print("baseline MSE identical across feature sets for every (year,window,horizon,model)?", identical)
print("max abs difference:", (dup_check["SWC_20 only"] - dup_check["SWC_20 + Ppt_log"]).abs().max())

n_baseline_rows = len(baseline_rows)
n_baseline_unique = len(baseline_rows.drop_duplicates(subset=["test_year","window_size","horizon","model","MSE","MAE","CORR"]))
n_cnn_rows = len(cnn_rows)
print(f"\nbaseline rows stored: {n_baseline_rows}, unique baseline experiments: {n_baseline_unique}")
print(f"CNN rows stored (all unique, features genuinely change the input): {n_cnn_rows}")
print(f"total rows stored: {len(full)}")
print(f"total UNIQUE experiments: {n_baseline_unique + n_cnn_rows}")

### Comparison with Simple Baselines

Averaging across all configurations hides horizon-dependent effects, since persistence is expected
to look artificially strong at short horizons and weak at long ones. Comparison is broken out by
horizon below, using the `SWC_20`-only feature set and `window_size=24` as the common reference
point (the same setting used throughout the single-year notebook), averaged over the 3 test years.

In [ ]:
full = pd.read_csv("results.csv")
print("total rows:", len(full))

ref = full[(full["features"] == "SWC_20 only") & (full["window_size"] == 24)]
baseline_table = ref.groupby(["horizon", "model"])[["MSE", "MAE", "SMAPE", "CORR"]].mean().round(5)
print(baseline_table)

In [ ]:
cnn_only = full[full["model"].isin(["small_cnn", "larger_cnn"])]

print("=== Effect of input-window size (SWC_20 only, averaged over test years) ===")
window_table = cnn_only[cnn_only["features"] == "SWC_20 only"].groupby(
    ["model", "window_size", "horizon"])["MSE"].mean().unstack("window_size").round(5)
print(window_table)

print()
print("=== Effect of adding Ppt_log (window_size=24, averaged over test years) ===")
feat_table = cnn_only[cnn_only["window_size"] == 24].groupby(
    ["model", "horizon", "features"])["MSE"].mean().unstack("features").round(5)
feat_table["Ppt_helps"] = feat_table["SWC_20 + Ppt_log"] < feat_table["SWC_20 only"]
print(feat_table)

In [ ]:
print("=== Effect of test-year wetness (SWC_20 only, window=24) ===")
year_table = full[(full["features"] == "SWC_20 only") & (full["window_size"] == 24) &
                   (full["model"].isin(["small_cnn", "larger_cnn"]))].groupby(
    ["model", "horizon", "test_year"])["MSE"].mean().unstack("test_year").round(5)
year_table.columns = [f"{y} ({'driest' if y==2023 else 'median' if y==2020 else 'wettest'})" for y in year_table.columns]
print(year_table)

In [ ]:
corr_year_table = full[(full["features"] == "SWC_20 only") & (full["window_size"] == 24) &
                        (full["model"].isin(["small_cnn", "larger_cnn"]))].groupby(
    ["model", "horizon", "test_year"])["CORR"].mean().unstack("test_year").round(4)
corr_year_table.columns = [f"{y} ({'driest' if y==2023 else 'median' if y==2020 else 'wettest'})" for y in corr_year_table.columns]
print(corr_year_table)

In [ ]:
loss_curves = {}
for model_name in ["small_cnn", "larger_cnn"]:
    for test_year in TEST_YEARS:
        for horizon in [24, 168]:
            random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
            trl_, vl_, tel_, _, _ = build_fold(test_year, ["SWC_20"], 24, horizon, batch_size=128)
            model = MODEL_CLASSES[model_name](num_features=1)
            model, trh, vlh, stopped = train_model(model, trl_, vl_, epochs=30, patience=6)
            loss_curves[(model_name, test_year, horizon)] = (trh, vlh, stopped)
            print(f"{model_name} test={test_year} horizon={horizon}: stopped at {stopped}, "
                  f"final train={trh[-1]:.5f} val={vlh[-1]:.5f}")

## Overfitting Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(22, 7), sharey=False)
col = 0
for model_name in ["small_cnn", "larger_cnn"]:
    for test_year in TEST_YEARS:
        for horizon in [24, 168]:
            trh, vlh, stopped = loss_curves[(model_name, test_year, horizon)]
            ax = axes[0 if model_name == "small_cnn" else 1][col % 6]
            ax.plot(trh, label="train")
            ax.plot(vlh, label="val")
            ax.axvline(stopped - 1, color="gray", linestyle=":", linewidth=1)
            ax.set_title(f"{model_name}\ntest={test_year} h={horizon}", fontsize=9)
            ax.set_xlabel("epoch")
            if col % 6 == 0:
                ax.set_ylabel("MSE loss (scaled)")
                ax.legend(fontsize=7)
            col += 1
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=100)
plt.show()
print("saved loss_curves.png")

In [ ]:
pred_examples = {}
for horizon in [24, 168]:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    trl_, vl_, tel_, (_, _, _, _, Xte_, yte_), scaler_ = build_fold(2020, ["SWC_20"], 24, horizon, batch_size=128)
    model = LargerCNN(num_features=1)
    model, trh, vlh, stopped = train_model(model, trl_, vl_, epochs=30, patience=6)
    preds, trues = evaluate_model(model, tel_)
    trues_original = inverse_transform_target(scaler_, ["SWC_20"], trues.flatten())
    preds_original = inverse_transform_target(scaler_, ["SWC_20"], preds.flatten())
    pred_examples[horizon] = (trues_original, preds_original)
    print(f"horizon={horizon}: test points={len(trues)}, "
          f"MSE (original SWC_20 scale)={compute_metrics(trues_original, preds_original)['MSE']:.6f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
for ax, h in zip(axes, [24, 168]):
    trues, preds = pred_examples[h]
    ax.plot(trues, label="actual", linewidth=0.9)
    ax.plot(preds, label="predicted", linewidth=0.9, alpha=0.85)
    ax.set_title(f"test year 2020, horizon={h}h, larger_cnn, window=24, SWC_20 only")
    ax.set_ylabel("SWC_20")
    ax.legend(fontsize=8)
axes[1].set_xlabel("test-set hour index")
plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=100)
print("saved actual_vs_predicted.png")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
    sub = baseline_table.xs(model_name, level="model")
    ax.plot(sub.index, sub["MSE"], marker="o", label=model_name)
ax.set_xlabel("forecast horizon (hours)")
ax.set_ylabel("test MSE (SWC_20 only, window=24, averaged over 3 test years)")
ax.set_title("Baseline vs. CNN across horizons")
ax.legend()
plt.tight_layout()
plt.savefig("baseline_vs_cnn.png", dpi=100)
plt.close(fig)
print("saved baseline_vs_cnn.png")

## Extended Training

In [ ]:
EXT_EPOCHS = 100
extended_curves = {}
for model_name in ["small_cnn", "larger_cnn"]:
    for horizon in [24, 168]:
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        trl_e, vl_e, tel_e, _, _ = build_fold(2020, ["SWC_20"], 24, horizon, batch_size=128)
        model = MODEL_CLASSES[model_name](num_features=1)
        # patience > epochs disables early stopping entirely -- the full budget always runs
        model, trh, vlh, stopped = train_model(model, trl_e, vl_e, epochs=EXT_EPOCHS, patience=EXT_EPOCHS + 1)
        best_epoch = int(np.argmin(vlh)) + 1
        extended_curves[(model_name, horizon)] = (trh, vlh, best_epoch)
        print(f"{model_name} h={horizon}: ran all {len(trh)} epochs, best val epoch={best_epoch} "
              f"(val={vlh[best_epoch-1]:.6f}), final (epoch {len(vlh)}) val={vlh[-1]:.6f}, "
              f"final train={trh[-1]:.6f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for row, model_name in enumerate(["small_cnn", "larger_cnn"]):
    for col, horizon in enumerate([24, 168]):
        trh, vlh, best_epoch = extended_curves[(model_name, horizon)]
        ax = axes[row][col]
        ax.plot(range(1, len(trh) + 1), trh, label="train")
        ax.plot(range(1, len(vlh) + 1), vlh, label="val")
        ax.axvline(best_epoch, color="gray", linestyle=":", linewidth=1, label=f"best val epoch ({best_epoch})")
        ax.set_title(f"{model_name}, horizon={horizon}h")
        ax.set_xlabel("epoch")
        ax.set_ylabel("MSE loss (scaled)")
        ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("extended_training_curves.png", dpi=100)
plt.close(fig)
print("saved extended_training_curves.png")

Mild overfitting at H=168, more pronounced for LargerCNN (+10.1% val loss rise) than SmallCNN (+4.7%). Early stopping is necessary for the larger model.

## Discussion (Regression)

## References

---

In [ ]:

# ============================================================
# Multi-station data quality survey
# Goal: find which stations have enough clean, complete years
# of SWC_20 data (>= 95% observed, full calendar-year coverage)
# to be useful for a multi-station generalization experiment.
# ============================================================
import pandas as pd, numpy as np

df_all = pd.read_parquet("data/TxSON33_modeling_complete.parquet")
df_all["Timestamp"] = pd.to_datetime(df_all["Timestamp"])
df_all["year"] = df_all["Timestamp"].dt.year

stations = sorted(df_all["Station"].unique())
print(f"Total stations in dataset: {len(stations)}")

rows = []
for station in stations:
    sdf = df_all[df_all["Station"] == station]
    for yr in sorted(sdf["year"].unique()):
        g = sdf[sdf["year"] == yr]
        expected = 8784 if (yr % 4 == 0 and (yr % 100 != 0 or yr % 400 == 0)) else 8760
        n = len(g)
        coverage = n / expected * 100

        swc_obs = (g["SWC_20_Origin"] == "observed").sum() / max(n, 1) * 100
        swc_mc = (g["SWC_20_Origin"] == "modeling_completion").sum() / max(n, 1) * 100
        ppt_obs = (g["Ppt_Origin"] == "observed").sum() / max(n, 1) * 100

        rows.append({
            "station": station, "year": yr, "n_rows": n,
            "expected": expected, "coverage_pct": round(coverage, 1),
            "swc20_obs_pct": round(swc_obs, 1),
            "swc20_mc_pct": round(swc_mc, 1),
            "ppt_obs_pct": round(ppt_obs, 1),
        })

quality = pd.DataFrame(rows)

# Filter: full calendar year (>=99% coverage), high SWC_20 observed (>=95%),
# no modeling_completion in SWC_20, and good precipitation (>=90% observed)
clean = quality[
    (quality["coverage_pct"] >= 99.0) &
    (quality["swc20_obs_pct"] >= 95.0) &
    (quality["swc20_mc_pct"] == 0.0) &
    (quality["ppt_obs_pct"] >= 90.0)
].copy()

print(f"\nClean station-years (>=99% coverage, >=95% SWC_20 observed, 0% modeling_completion, >=90% Ppt observed):")
print(f"  {len(clean)} station-years across {clean['station'].nunique()} stations")

# Summary per station: how many clean years?
station_summary = clean.groupby("station").agg(
    n_clean_years=("year", "count"),
    years=("year", lambda x: sorted(x.tolist())),
    min_swc_obs=("swc20_obs_pct", "min"),
).sort_values("n_clean_years", ascending=False)

print("\nStations with >= 5 clean years:")
for _, row in station_summary[station_summary["n_clean_years"] >= 5].iterrows():
    print(f"  {row.name}: {row['n_clean_years']} years {row['years']}  (min SWC_20 obs: {row['min_swc_obs']}%)")


## Multi-Station Regression (Early Exploration)

Initial multi-station generalization experiments before the full pipeline was built.

## Feature Engineering

In [ ]:

# ============================================================
# Extend YEAR_FEATURES with candidate features
# ============================================================

for y in ALL_YEARS:
    g = YEAR_FEATURES[y]

    for col in ["Tair", "T_5", "T_10", "T_20", "T_50",
                "SWC_5", "SWC_10", "SWC_50", "RH", "Srad"]:
        if col in g.columns:
            g[col] = g[col].ffill().bfill().fillna(0.0)

    g["Ppt_3h"]  = g["Ppt"].rolling(3,  min_periods=1).sum()
    g["Ppt_6h"]  = g["Ppt"].rolling(6,  min_periods=1).sum()
    g["Ppt_12h"] = g["Ppt"].rolling(12, min_periods=1).sum()
    g["Ppt_24h"] = g["Ppt"].rolling(24, min_periods=1).sum()

    doy = g["Timestamp"].dt.dayofyear
    g["sin_doy"] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
    g["cos_doy"] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

print("Extended YEAR_FEATURES with engineered features")
eng_cols = ["Ppt_3h", "Ppt_6h", "Ppt_12h", "Ppt_24h", "sin_doy", "cos_doy"]
print(YEAR_FEATURES[2020][eng_cols].describe().round(4).to_string())

# ============================================================
# Individual feature selection — W=24, H=24, test_year=2023
# ============================================================
FS_TEST_YEAR = 2023
FS_WINDOW = 24
FS_HORIZON = 24
FEAT_SELECT_CSV = "feature_selection_results.csv"

INDIVIDUAL_FEATURE_SETS = [
    ("SWC_20 only",   ["SWC_20"]),
    ("+ Ppt_log",     ["SWC_20", "Ppt_log"]),
    ("+ Tair",        ["SWC_20", "Tair"]),
    ("+ T_20",        ["SWC_20", "T_20"]),
    ("+ SWC depths",  ["SWC_20", "SWC_5", "SWC_10", "SWC_50"]),
    ("+ Rain accum",  ["SWC_20", "Ppt_log", "Ppt_3h", "Ppt_6h", "Ppt_12h", "Ppt_24h"]),
    ("+ Seasonal",    ["SWC_20", "sin_doy", "cos_doy"]),
]

fs_rows = []

# Baselines (instant, only use SWC_20 — identical across feature sets)
for bl_name in ["persistence", "moving_average"]:
    r = run_one_config(FS_TEST_YEAR, FS_WINDOW, FS_HORIZON,
                       "SWC_20 only", ["SWC_20"], bl_name)
    fs_rows.append(r)
    print(f"Baseline {bl_name}: MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f}")

# CNN models for each feature set
t_total = time.time()
for feat_name, feature_cols in INDIVIDUAL_FEATURE_SETS:
    for model_name in ["small_cnn", "larger_cnn"]:
        t0 = time.time()
        r = run_one_config(FS_TEST_YEAR, FS_WINDOW, FS_HORIZON,
                           feat_name, feature_cols, model_name)
        fs_rows.append(r)
        pd.DataFrame(fs_rows).to_csv(FEAT_SELECT_CSV, index=False)
        vl = r.get("final_val_loss", None)
        vl_str = f"{vl:.6f}" if vl else "N/A"
        print(f"  {feat_name:16s} {model_name:12s}: val={vl_str}, "
              f"MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} ({time.time()-t0:.0f}s)")

print(f"\nAll individual additions done: {len(fs_rows)} rows in {time.time()-t_total:.0f}s")
print(f"Saved to {FEAT_SELECT_CSV}")


In [ ]:

# ============================================================
# Combined feature set + Summary table
# ============================================================
fs_df = pd.read_csv(FEAT_SELECT_CSV)
print(f"Loaded {len(fs_df)} feature selection results")

# Determine which additions improved validation loss (using LargerCNN)
larger = fs_df[fs_df["model"] == "larger_cnn"].copy()
baseline_val = larger[larger["features"] == "SWC_20 only"]["final_val_loss"].values[0]
baseline_test_mse = larger[larger["features"] == "SWC_20 only"]["MSE"].values[0]

FEATURE_ADDITIONS = {
    "+ Ppt_log":    ["Ppt_log"],
    "+ Tair":       ["Tair"],
    "+ T_20":       ["T_20"],
    "+ SWC depths": ["SWC_5", "SWC_10", "SWC_50"],
    "+ Rain accum": ["Ppt_log", "Ppt_3h", "Ppt_6h", "Ppt_12h", "Ppt_24h"],
    "+ Seasonal":   ["sin_doy", "cos_doy"],
}

print(f"\nLargerCNN baseline — val_loss: {baseline_val:.6f}, test MSE: {baseline_test_mse:.6f}")
print(f"\nIndividual additions (LargerCNN):")
val_improved = []
test_improved = []
for _, row in larger.iterrows():
    fn = row["features"]
    if fn == "SWC_20 only" or fn not in FEATURE_ADDITIONS:
        continue
    vl = row["final_val_loss"]
    tm = row["MSE"]
    vl_delta = (vl - baseline_val) / baseline_val * 100
    tm_delta = (tm - baseline_test_mse) / baseline_test_mse * 100
    v_status = "BETTER" if vl < baseline_val else "worse"
    t_status = "BETTER" if tm < baseline_test_mse else "worse"
    print(f"  {fn:16s}: val_loss={vl:.6f} ({vl_delta:+.1f}% {v_status}), "
          f"test_MSE={tm:.6f} ({tm_delta:+.1f}% {t_status})")
    if vl < baseline_val:
        val_improved.append(fn)
    if tm < baseline_test_mse:
        test_improved.append(fn)

# Build combined set
if val_improved:
    print(f"\nFeatures that improved validation: {val_improved}")
    combined_features = ["SWC_20"]
    for fn in val_improved:
        combined_features.extend(FEATURE_ADDITIONS[fn])
    combined_features = list(dict.fromkeys(combined_features))
else:
    print("\nNo individual addition improved validation loss.")
    print("This means SWC_20 alone is the strongest single-feature set for this model/config.")
    if test_improved:
        print(f"However, these improved test MSE: {test_improved}")
        print("Building a combined set from test-improving features for comparison")
        print("(NOTE: this is informational — feature selection was done on validation)")
        combined_features = ["SWC_20"]
        for fn in test_improved:
            combined_features.extend(FEATURE_ADDITIONS[fn])
        combined_features = list(dict.fromkeys(combined_features))
    else:
        combined_features = ["SWC_20"]

print(f"\nCombined feature set ({len(combined_features)} features): {combined_features}")

# Run combined set if it differs from baseline
new_rows = []
if len(combined_features) > 1:
    t0 = time.time()
    for model_name in ["small_cnn", "larger_cnn"]:
        r = run_one_config(FS_TEST_YEAR, FS_WINDOW, FS_HORIZON,
                           "Combined best", combined_features, model_name)
        new_rows.append(r)
        vl = r.get("final_val_loss", None)
        print(f"  Combined best {model_name:12s}: val={vl:.6f}, "
              f"MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} ({time.time()-t0:.0f}s)")
    fs_df = pd.concat([fs_df, pd.DataFrame(new_rows)], ignore_index=True)
    fs_df.to_csv(FEAT_SELECT_CSV, index=False)
else:
    print("Combined set = baseline (SWC_20 only), skipping redundant run.")

# ============================================================
# Summary table
# ============================================================
print("\n" + "="*100)
print("FEATURE SELECTION SUMMARY -- W=24, H=24, test_year=2023 (val=2019)")
print("="*100)
print(f"{'Features':20s} {'Model':14s} {'Val Loss':>10s} {'Test MSE':>10s} "
      f"{'RMSE':>8s} {'MAE':>8s} {'CORR':>8s} {'Ep':>4s}")
print("-"*100)

display_order = ["SWC_20 only", "+ Ppt_log", "+ Tair", "+ T_20",
                 "+ SWC depths", "+ Rain accum", "+ Seasonal"]
if len(combined_features) > 1:
    display_order.append("Combined best")
model_order = ["persistence", "moving_average", "small_cnn", "larger_cnn"]

for feat_name in display_order:
    sub = fs_df[fs_df["features"] == feat_name]
    for mn in model_order:
        row = sub[sub["model"] == mn]
        if row.empty:
            continue
        row = row.iloc[0]
        vl = row.get("final_val_loss", None)
        vl_str = f"{vl:.6f}" if pd.notna(vl) and vl else "     --"
        rmse = np.sqrt(row["MSE"])
        ep = int(row["epochs_run"]) if pd.notna(row.get("epochs_run")) else 0
        print(f"{feat_name:20s} {mn:14s} {vl_str:>10s} {row['MSE']:10.6f} "
              f"{rmse:8.4f} {row['MAE']:8.4f} {row['CORR']:8.4f} {ep:4d}")
    print()

# Key observations
print("="*100)
print("KEY OBSERVATIONS:")
print("- Baselines: persistence MSE=0.000299, moving_average MSE=0.000356")

best_larger = fs_df[(fs_df["model"]=="larger_cnn")].sort_values("MSE").iloc[0]
best_small = fs_df[(fs_df["model"]=="small_cnn")].sort_values("MSE").iloc[0]
print(f"- Best LargerCNN test MSE: {best_larger['MSE']:.6f} (features: {best_larger['features']})")
print(f"- Best SmallCNN test MSE:  {best_small['MSE']:.6f} (features: {best_small['features']})")
print(f"- LargerCNN consistently outperforms SmallCNN across all feature sets")

# Check if any CNN beats persistence
cnn_rows = fs_df[fs_df["model"].isin(["small_cnn", "larger_cnn"])]
persistence_mse = fs_df[fs_df["model"]=="persistence"]["MSE"].values[0]
beats_persistence = cnn_rows[cnn_rows["MSE"] < persistence_mse]
print(f"- {len(beats_persistence)} of {len(cnn_rows)} CNN runs beat persistence baseline")


## Architecture Search & Feature–Horizon Interaction

In [ ]:

# ============================================================
# Part B: Diagnose LargerCNN collapse with combined features
# Train with more epochs/patience to see full learning curves
# ============================================================
import matplotlib.pyplot as plt

DIAG_CONFIGS = {
    "SWC_20 only":       ["SWC_20"],
    "+ SWC depths":      ["SWC_20", "SWC_5", "SWC_10", "SWC_50"],
    "Combined (5 feat)": ["SWC_20", "Tair", "SWC_5", "SWC_10", "SWC_50"],
}

diag_curves = {}
diag_metrics = {}

for feat_name, feature_cols in DIAG_CONFIGS.items():
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    trl, vl, tel, (Xtr, ytr, Xv, yv, Xte, yte), scaler = build_fold(
        2023, feature_cols, 24, 24, batch_size=128, seed=SEED)

    model = LargerCNN(num_features=len(feature_cols))
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model, trh, vlh, stopped = train_model(model, trl, vl, epochs=60, patience=20)

    preds, _ = evaluate_model(model, tel)
    yte_orig = inverse_transform_target(scaler, feature_cols, yte)
    preds_orig = inverse_transform_target(scaler, feature_cols, preds.flatten())
    metrics = compute_metrics(yte_orig, preds_orig)

    diag_curves[feat_name] = (trh, vlh)
    diag_metrics[feat_name] = {**metrics, "stopped": stopped, "params": n_params}

    print(f"{feat_name:25s}: {n_params:5d} params, stopped ep {stopped:2d}, "
          f"best_val={min(vlh):.6f}, MSE={metrics['MSE']:.6f}, CORR={metrics['CORR']:.4f}")

# Plot loss curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
colors = {"SWC_20 only": "#1f77b4", "+ SWC depths": "#ff7f0e", "Combined (5 feat)": "#2ca02c"}

for feat_name in DIAG_CONFIGS:
    trh, vlh = diag_curves[feat_name]
    c = colors[feat_name]
    epochs = range(1, len(trh)+1)

    axes[0].plot(epochs, trh, color=c, label=feat_name)
    axes[1].plot(epochs, vlh, color=c, label=feat_name)
    ratio = [v/t if t > 1e-10 else 0 for t, v in zip(trh, vlh)]
    axes[2].plot(epochs, ratio, color=c, label=feat_name)

axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE (scaled)")
axes[1].set_title("Validation Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MSE (scaled)")
axes[2].set_title("Val/Train Ratio (>1 = generalization gap)"); axes[2].set_xlabel("Epoch")
axes[2].axhline(1.0, color='gray', linestyle='--', alpha=0.5)

for ax in axes:
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.suptitle("LargerCNN Loss Curves — Diagnosing Combined Feature Collapse (W=24, H=24, test=2023)")
plt.tight_layout()
plt.savefig("largercnn_diagnosis.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSaved largercnn_diagnosis.png")

# Additional diagnostic: check if the issue is train/val distribution mismatch
print("\n--- Feature value distributions (train vs val) ---")
for feat_name, feature_cols in DIAG_CONFIGS.items():
    trl_d, vl_d, tel_d, (Xtr_d, ytr_d, Xv_d, yv_d, Xte_d, yte_d), sc = build_fold(
        2023, feature_cols, 24, 24, batch_size=128, seed=SEED)
    print(f"\n{feat_name} ({len(feature_cols)} features):")
    for i, col in enumerate(feature_cols):
        tr_mean, tr_std = Xtr_d[:,:,i].mean(), Xtr_d[:,:,i].std()
        va_mean, va_std = Xv_d[:,:,i].mean(), Xv_d[:,:,i].std()
        te_mean, te_std = Xte_d[:,:,i].mean(), Xte_d[:,:,i].std()
        print(f"  {col:8s}: train={tr_mean:.3f}±{tr_std:.3f}, val={va_mean:.3f}±{va_std:.3f}, test={te_mean:.3f}±{te_std:.3f}")


### Feature Experiments at Longer Horizons

In [ ]:

# ============================================================
# Part A: Feature experiments at longer horizons (H=48, 72, 168)
# Question: do features that didn't help at H=24 become useful
# at longer horizons where persistence degrades?
# ============================================================
HORIZON_FEATURE_SETS = {
    "SWC_20 only":   ["SWC_20"],
    "+ Ppt_log":     ["SWC_20", "Ppt_log"],
    "+ Tair":        ["SWC_20", "Tair"],
    "+ SWC depths":  ["SWC_20", "SWC_5", "SWC_10", "SWC_50"],
    "+ Rain accum":  ["SWC_20", "Ppt_log", "Ppt_3h", "Ppt_6h", "Ppt_12h", "Ppt_24h"],
    "+ Seasonal":    ["SWC_20", "sin_doy", "cos_doy"],
}

LONGER_HORIZONS = [48, 72, 168]
FS_WINDOW = 24
FS_TEST_YEAR = 2023
HORIZON_CSV = "feature_horizon_results.csv"

rows = []
t_start = time.time()

for horizon in LONGER_HORIZONS:
    # Baselines once per horizon
    for bl_name in ["persistence", "moving_average"]:
        r = run_one_config(FS_TEST_YEAR, FS_WINDOW, horizon,
                           "SWC_20 only", ["SWC_20"], bl_name)
        rows.append(r)
        print(f"  H={horizon:3d} {bl_name:16s}: MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f}")

    # CNN runs for each feature set
    for feat_name, feature_cols in HORIZON_FEATURE_SETS.items():
        for model_name in ["small_cnn", "larger_cnn"]:
            t0 = time.time()
            r = run_one_config(FS_TEST_YEAR, FS_WINDOW, horizon,
                               feat_name, feature_cols, model_name)
            rows.append(r)
            pd.DataFrame(rows).to_csv(HORIZON_CSV, index=False)
            vl = r.get("final_val_loss", None)
            print(f"  H={horizon:3d} {feat_name:16s} {model_name:12s}: "
                  f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} ({time.time()-t0:.0f}s)")

    elapsed = time.time() - t_start
    print(f"  --- H={horizon} complete ({elapsed:.0f}s elapsed) ---\n")

pd.DataFrame(rows).to_csv(HORIZON_CSV, index=False)
print(f"\nAll horizon experiments done: {len(rows)} rows in {time.time()-t_start:.0f}s")
print(f"Saved to {HORIZON_CSV}")


In [ ]:

# ============================================================
# Part A continued: Summary table + horizon comparison plots
# Combine H=24 results from Stage 1 with H=48/72/168 results
# ============================================================
import matplotlib.pyplot as plt

# Load both CSVs
fs24 = pd.read_csv("feature_selection_results.csv")
fs_long = pd.read_csv("feature_horizon_results.csv")

# Add H=24 baseline and individual feature CNN results from Stage 1
# (exclude "Combined best" since it's not in the longer-horizon set)
fs24_clean = fs24[fs24["features"] != "Combined best"].copy()

# Merge all horizons
all_fs = pd.concat([fs24_clean, fs_long], ignore_index=True)
all_fs.to_csv("feature_all_horizons.csv", index=False)

# ---- Summary table: LargerCNN val_loss and test MSE by feature × horizon ----
print("="*110)
print("FEATURE × HORIZON SUMMARY — LargerCNN, W=24, test=2023 (val=2019)")
print("="*110)
print(f"{'Features':16s} | {'H=24':>22s} | {'H=48':>22s} | {'H=72':>22s} | {'H=168':>22s}")
print(f"{'':16s} | {'val_loss   test_MSE':>22s} | {'val_loss   test_MSE':>22s} | {'val_loss   test_MSE':>22s} | {'val_loss   test_MSE':>22s}")
print("-"*110)

feat_order = ["SWC_20 only", "+ Ppt_log", "+ Tair", "+ SWC depths", "+ Rain accum", "+ Seasonal"]
horizons = [24, 48, 72, 168]
larger = all_fs[all_fs["model"] == "larger_cnn"]

for fn in feat_order:
    parts = []
    for h in horizons:
        row = larger[(larger["features"] == fn) & (larger["horizon"] == h)]
        if row.empty:
            parts.append(f"{'--':>22s}")
        else:
            row = row.iloc[0]
            vl = row.get("final_val_loss", float('nan'))
            mse = row["MSE"]
            parts.append(f"{vl:.6f}  {mse:.6f}")
    print(f"{fn:16s} | {' | '.join(parts)}")

# Baselines for reference
print("-"*110)
for bl in ["persistence", "moving_average"]:
    parts = []
    for h in horizons:
        row = all_fs[(all_fs["model"] == bl) & (all_fs["horizon"] == h)]
        if row.empty:
            parts.append(f"{'--':>22s}")
        else:
            parts.append(f"{'--':>10s}  {row.iloc[0]['MSE']:.6f}")
    print(f"{bl:16s} | {' | '.join(parts)}")

# ---- Identify features that help at longer horizons ----
print("\n\nFEATURE SELECTION BY HORIZON (LargerCNN val_loss improvement over SWC_20-only):")
for h in horizons:
    baseline_row = larger[(larger["features"] == "SWC_20 only") & (larger["horizon"] == h)]
    if baseline_row.empty:
        continue
    baseline_vl = baseline_row.iloc[0]["final_val_loss"]
    print(f"\n  H={h}: baseline val_loss = {baseline_vl:.6f}")
    for fn in feat_order[1:]:
        row = larger[(larger["features"] == fn) & (larger["horizon"] == h)]
        if row.empty:
            continue
        vl = row.iloc[0]["final_val_loss"]
        delta = (vl - baseline_vl) / baseline_vl * 100
        status = "BETTER" if vl < baseline_vl else "worse"
        print(f"    {fn:16s}: val={vl:.6f} ({delta:+.1f}% {status})")

# ---- Plot: MSE by horizon for LargerCNN feature sets ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_feat = {"SWC_20 only": "#1f77b4", "+ Ppt_log": "#ff7f0e", "+ Tair": "#2ca02c",
               "+ SWC depths": "#d62728", "+ Rain accum": "#9467bd", "+ Seasonal": "#8c564b"}

# LargerCNN test MSE
ax = axes[0]
for fn in feat_order:
    mses = []
    for h in horizons:
        row = larger[(larger["features"] == fn) & (larger["horizon"] == h)]
        mses.append(row.iloc[0]["MSE"] if not row.empty else float('nan'))
    ax.plot(horizons, mses, 'o-', color=colors_feat[fn], label=fn, markersize=5)

# Add persistence baseline
pers_mses = []
for h in horizons:
    row = all_fs[(all_fs["model"] == "persistence") & (all_fs["horizon"] == h)]
    pers_mses.append(row.iloc[0]["MSE"] if not row.empty else float('nan'))
ax.plot(horizons, pers_mses, 's--', color='gray', label='persistence', markersize=5)

ax.set_xlabel("Horizon (hours)"); ax.set_ylabel("Test MSE")
ax.set_title("LargerCNN Test MSE by Feature Set and Horizon")
ax.legend(fontsize=7, loc='upper left'); ax.grid(alpha=0.3)
ax.set_xticks(horizons)

# LargerCNN validation loss
ax = axes[1]
for fn in feat_order:
    vls = []
    for h in horizons:
        row = larger[(larger["features"] == fn) & (larger["horizon"] == h)]
        if not row.empty:
            vls.append(row.iloc[0]["final_val_loss"])
        else:
            vls.append(float('nan'))
    ax.plot(horizons, vls, 'o-', color=colors_feat[fn], label=fn, markersize=5)
ax.set_xlabel("Horizon (hours)"); ax.set_ylabel("Validation Loss")
ax.set_title("LargerCNN Validation Loss by Feature Set and Horizon")
ax.legend(fontsize=7, loc='upper left'); ax.grid(alpha=0.3)
ax.set_xticks(horizons)

plt.tight_layout()
plt.savefig("feature_horizon_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved feature_horizon_comparison.png")


### Architecture Search

In [ ]:

# ============================================================
# Part C: CNN Architecture Search
# Flexible architecture class + systematic comparison
# ============================================================

class FlexCNN(nn.Module):
    def __init__(self, num_features, n_layers=2, filters=None, kernel_size=3,
                 pool_type='avg', dropout=0.0):
        super().__init__()
        if filters is None:
            filters = [16, 32][:n_layers]
        p = kernel_size // 2
        layers = []
        in_ch = num_features
        for f in filters:
            layers.append(nn.Conv1d(in_ch, f, kernel_size, padding=p))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = f
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)
        hidden = max(filters[-1] // 2, 4)
        self.fc1 = nn.Linear(filters[-1], hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.convs(x)
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Architecture configs to test
ARCH_CONFIGS = {
    "SmallCNN (K=3, 8)":           {"n_layers": 1, "filters": [8], "kernel_size": 3},
    "LargerCNN (K=3, 16/32)":      {"n_layers": 2, "filters": [16, 32], "kernel_size": 3},
    "K=5, 16/32":                   {"n_layers": 2, "filters": [16, 32], "kernel_size": 5},
    "K=7, 16/32":                   {"n_layers": 2, "filters": [16, 32], "kernel_size": 7},
    "K=3, 32/64":                   {"n_layers": 2, "filters": [32, 64], "kernel_size": 3},
    "K=3, 16/32/64 (3 layers)":     {"n_layers": 3, "filters": [16, 32, 64], "kernel_size": 3},
    "K=5, 32/64":                   {"n_layers": 2, "filters": [32, 64], "kernel_size": 5},
    "K=3, 16/32, maxpool":          {"n_layers": 2, "filters": [16, 32], "kernel_size": 3, "pool_type": "max"},
    "K=3, 16/32, dropout=0.2":      {"n_layers": 2, "filters": [16, 32], "kernel_size": 3, "dropout": 0.2},
    "K=3, 16/32, dropout=0.3":      {"n_layers": 2, "filters": [16, 32], "kernel_size": 3, "dropout": 0.3},
}

# Print param counts
print("Architecture parameter counts (F=1 input):")
for name, cfg in ARCH_CONFIGS.items():
    m = FlexCNN(num_features=1, **cfg)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {name:35s}: {n_params:6d} params")

# Test each architecture at TWO horizons: H=24 (persistence-strong) and H=168 (CNN-strong)
ARCH_TEST_HORIZONS = [24, 168]
ARCH_CSV = "architecture_results.csv"
arch_rows = []
t_start = time.time()

for horizon in ARCH_TEST_HORIZONS:
    # Baselines
    for bl in ["persistence", "moving_average"]:
        r = run_one_config(2023, 24, horizon, "SWC_20 only", ["SWC_20"], bl)
        r["architecture"] = bl
        arch_rows.append(r)

    for arch_name, cfg in ARCH_CONFIGS.items():
        random.seed(SEED); np.random.seed(SEED)
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

        feature_cols = ["SWC_20"]
        trl, vl, tel, (Xtr, ytr, Xv, yv, Xte, yte), scaler = build_fold(
            2023, feature_cols, 24, horizon, batch_size=128, seed=SEED)

        model = FlexCNN(num_features=1, **cfg)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        t0 = time.time()
        model, trh, vlh, stopped = train_model(model, trl, vl, epochs=30, patience=6)
        preds, _ = evaluate_model(model, tel)

        yte_orig = inverse_transform_target(scaler, feature_cols, yte)
        preds_orig = inverse_transform_target(scaler, feature_cols, preds.flatten())
        metrics = compute_metrics(yte_orig, preds_orig)

        row = {"test_year": 2023, "window_size": 24, "horizon": horizon,
               "features": "SWC_20 only", "model": "flex_cnn", "architecture": arch_name,
               "n_params": n_params, "epochs_run": stopped, "train_seconds": round(time.time()-t0, 1),
               "final_train_loss": trh[-1], "final_val_loss": vlh[-1], "seed": SEED,
               "n_train": len(ytr), "n_val": len(yv), "n_test": len(yte)}
        row.update(metrics)
        arch_rows.append(row)
        pd.DataFrame(arch_rows).to_csv(ARCH_CSV, index=False)

        print(f"  H={horizon:3d} {arch_name:35s}: {n_params:5d}p, ep={stopped:2d}, "
              f"val={vlh[-1]:.6f}, MSE={metrics['MSE']:.6f}, CORR={metrics['CORR']:.4f} "
              f"({time.time()-t0:.0f}s)")
    print(f"  --- H={horizon} done ({time.time()-t_start:.0f}s) ---\n")

pd.DataFrame(arch_rows).to_csv(ARCH_CSV, index=False)
print(f"Architecture search done: {len(arch_rows)} rows in {time.time()-t_start:.0f}s")


In [ ]:

# ============================================================
# Part C continued: Architecture results summary
# ============================================================
import matplotlib.pyplot as plt

arch_df = pd.read_csv("architecture_results.csv")

print("="*100)
print("ARCHITECTURE COMPARISON — SWC_20 only, W=24, test=2023")
print("="*100)

for horizon in [24, 168]:
    sub = arch_df[arch_df["horizon"] == horizon].copy()
    print(f"\n--- Horizon = {horizon}h ---")
    print(f"{'Architecture':37s} {'Params':>7s} {'Epochs':>7s} {'Val Loss':>10s} "
          f"{'Test MSE':>10s} {'RMSE':>8s} {'CORR':>7s}")
    print("-"*95)

    # Baselines first
    for bl in ["persistence", "moving_average"]:
        row = sub[sub["architecture"] == bl]
        if not row.empty:
            row = row.iloc[0]
            rmse = np.sqrt(row["MSE"])
            print(f"{bl:37s} {'--':>7s} {'--':>7s} {'--':>10s} "
                  f"{row['MSE']:10.6f} {rmse:8.4f} {row['CORR']:7.4f}")

    # CNN architectures sorted by val_loss
    cnns = sub[sub["architecture"].isin(list(ARCH_CONFIGS.keys()))].sort_values("final_val_loss")
    for _, row in cnns.iterrows():
        rmse = np.sqrt(row["MSE"])
        best_marker = " *" if row["final_val_loss"] == cnns["final_val_loss"].min() else ""
        print(f"{row['architecture']:37s} {int(row['n_params']):7d} {int(row['epochs_run']):7d} "
              f"{row['final_val_loss']:10.6f} {row['MSE']:10.6f} {rmse:8.4f} "
              f"{row['CORR']:7.4f}{best_marker}")

# Bar chart: MSE comparison at H=168
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, h in zip(axes, [24, 168]):
    sub = arch_df[arch_df["horizon"] == h]
    cnns = sub[sub["architecture"].isin(list(ARCH_CONFIGS.keys()))].sort_values("MSE")

    names = cnns["architecture"].tolist()
    mses = cnns["MSE"].tolist()
    colors = ['#2ca02c' if mse == min(mses) else '#4c72b0' for mse in mses]

    # Add baselines
    for bl in ["persistence", "moving_average"]:
        row = sub[sub["architecture"] == bl]
        if not row.empty:
            names.append(bl)
            mses.append(row.iloc[0]["MSE"])
            colors.append('#999999')

    y_pos = range(len(names))
    ax.barh(y_pos, mses, color=colors, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_xlabel("Test MSE")
    ax.set_title(f"Architecture Comparison — H={h}")
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig("architecture_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved architecture_comparison.png")

# Print the top 3 at each horizon
print("\nTOP 3 ARCHITECTURES:")
for h in [24, 168]:
    sub = arch_df[(arch_df["horizon"] == h) & arch_df["architecture"].isin(list(ARCH_CONFIGS.keys()))]
    top3 = sub.sort_values("final_val_loss").head(3)
    print(f"\n  H={h} (by validation loss):")
    for _, row in top3.iterrows():
        print(f"    {row['architecture']:35s}: val={row['final_val_loss']:.6f}, MSE={row['MSE']:.6f}")


In [ ]:

# Multi-station data loading and pre-computation
SELECTED_STATIONS = ["FD11", "FD02", "CB06", "CB25"]

df_multi = df_all[df_all["Station"].isin(SELECTED_STATIONS)].copy()
df_multi["Timestamp"] = pd.to_datetime(df_multi["Timestamp"])
df_multi = df_multi.sort_values(["Station", "Timestamp"]).reset_index(drop=True)
df_multi["year"] = df_multi["Timestamp"].dt.year

# Keep only full calendar years 2015-2024
df_multi = df_multi[df_multi["year"].isin(ALL_YEARS)]
print(f"Multi-station dataset: {len(df_multi):,} rows, {df_multi['Station'].nunique()} stations, years {df_multi['year'].min()}-{df_multi['year'].max()}")

for station in SELECTED_STATIONS:
    sdf = df_multi[df_multi["Station"] == station]
    print(f"  {station}: {len(sdf):,} rows, {sdf['year'].nunique()} years")

# Pre-compute per-(station, year) DataFrames with Ppt_log added
def prepare_station_year(df, station, year):
    g = df[(df["Station"] == station) & (df["year"] == year)].copy()
    g = g.sort_values("Timestamp").reset_index(drop=True)
    g["Ppt"] = g["Ppt"].fillna(0.0).clip(lower=0.0)
    g["Ppt_log"] = np.log1p(g["Ppt"])
    return g

STATION_YEAR_DATA = {}
for station in SELECTED_STATIONS:
    for year in ALL_YEARS:
        STATION_YEAR_DATA[(station, year)] = prepare_station_year(df_multi, station, year)

print(f"\nPre-computed {len(STATION_YEAR_DATA)} station-year DataFrames")
print(f"Example — FD11/2020: {len(STATION_YEAR_DATA[('FD11', 2020)])} rows")
print(f"Example — CB06/2023: {len(STATION_YEAR_DATA[('CB06', 2023)])} rows")


In [ ]:

# Generalized fold builder for multi-station experiments

def get_fold_keys(experiment_type, test_value):
    """
    Returns (train_keys, val_keys, test_keys) as lists of (station, year).
    experiment_type: "hold_out_year" or "hold_out_station"
    test_value: a year (int) or station name (str)
    """
    if experiment_type == "hold_out_year":
        test_year = test_value
        train_keys = [(s, y) for s in SELECTED_STATIONS for y in ALL_YEARS
                      if y != test_year and y != VAL_YEAR]
        val_keys = [(s, VAL_YEAR) for s in SELECTED_STATIONS]
        test_keys = [(s, test_year) for s in SELECTED_STATIONS]

    elif experiment_type == "hold_out_station":
        test_station = test_value
        train_stations = [s for s in SELECTED_STATIONS if s != test_station]
        train_keys = [(s, y) for s in train_stations for y in ALL_YEARS if y != VAL_YEAR]
        val_keys = [(s, VAL_YEAR) for s in train_stations]
        test_keys = [(test_station, y) for y in ALL_YEARS if y != VAL_YEAR]

    else:
        raise ValueError(f"Unknown experiment_type: {experiment_type}")

    return train_keys, val_keys, test_keys


def build_multi_fold(train_keys, val_keys, test_keys, feature_cols, window_size, horizon, batch_size, seed=SEED):
    """
    Build train/val/test loaders from lists of (station, year) keys.
    Scaler is fit on training data only.
    """
    scaler = MinMaxScaler()
    train_dfs = [STATION_YEAR_DATA[k][feature_cols] for k in train_keys]
    scaler.fit(pd.concat(train_dfs, axis=0))

    def collect_windows(keys):
        Xs, ys = [], []
        for (station, year) in keys:
            g = STATION_YEAR_DATA[(station, year)].copy()
            g[feature_cols] = scaler.transform(g[feature_cols])
            X, y = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X):
                Xs.append(X); ys.append(y)
        if not Xs:
            return np.empty((0, window_size, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.float32)
        return np.concatenate(Xs), np.concatenate(ys)

    Xtr, ytr = collect_windows(train_keys)
    Xv, yv = collect_windows(val_keys)
    Xte, yte = collect_windows(test_keys)

    g = torch.Generator().manual_seed(seed)
    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y).unsqueeze(1))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g if shuffle else None)

    return (loader(Xtr, ytr, True), loader(Xv, yv, False), loader(Xte, yte, False),
            (Xtr, ytr, Xv, yv, Xte, yte), scaler)


# Sanity check
tr, va, te = get_fold_keys("hold_out_year", 2023)
print(f"Hold-out-year 2023: train={len(tr)} station-years, val={len(va)}, test={len(te)}")
tr, va, te = get_fold_keys("hold_out_station", "CB06")
print(f"Hold-out-station CB06: train={len(tr)} station-years, val={len(va)}, test={len(te)}")

# Quick fold build test
trl_m, vl_m, tel_m, (Xtr_m, ytr_m, Xv_m, yv_m, Xte_m, yte_m), sc_m = build_multi_fold(
    *get_fold_keys("hold_out_year", 2023), ["SWC_20"], 24, 24, 128)
print(f"\nHold-out-year 2023 fold: train={Xtr_m.shape}, val={Xv_m.shape}, test={Xte_m.shape}")

trl_m2, vl_m2, tel_m2, (Xtr_m2, ytr_m2, Xv_m2, yv_m2, Xte_m2, yte_m2), sc_m2 = build_multi_fold(
    *get_fold_keys("hold_out_station", "CB06"), ["SWC_20"], 24, 24, 128)
print(f"Hold-out-station CB06 fold: train={Xtr_m2.shape}, val={Xv_m2.shape}, test={Xte_m2.shape}")


In [ ]:

# Updated compute_metrics with RMSE, and multi-station run_one_config

def compute_metrics_v2(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=np.float64).flatten()
    y_pred = np.asarray(y_pred, dtype=np.float64).flatten()
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    smape = np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100
    rse = np.sum((y_pred - y_true) ** 2) / (np.sum((y_true - y_true.mean()) ** 2) + eps)
    corr = np.corrcoef(y_true, y_pred)[0, 1] if np.std(y_pred) > 1e-10 else float("nan")
    return {"MSE": mse, "RMSE": rmse, "MAE": mae, "SMAPE": smape, "RSE": rse, "CORR": corr}


MULTI_RESULTS_CSV = "multi_station_results.csv"

def run_one_config_multi(experiment_type, test_value, window_size, horizon,
                         feat_name, feature_cols, model_name, seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    target_idx = feature_cols.index("SWC_20")

    train_keys, val_keys, test_keys = get_fold_keys(experiment_type, test_value)
    trl, vl, tel, (Xtr, ytr, Xv, yv, Xte, yte), scaler = build_multi_fold(
        train_keys, val_keys, test_keys, feature_cols, window_size, horizon, batch_size=128, seed=seed)

    row = {
        "experiment": experiment_type, "test_value": test_value,
        "window_size": window_size, "horizon": horizon,
        "features": feat_name, "model": model_name, "seed": seed,
        "n_train": len(ytr), "n_val": len(yv), "n_test": len(yte),
        "n_train_station_years": len(train_keys),
        "n_val_station_years": len(val_keys),
        "n_test_station_years": len(test_keys),
    }

    if model_name == "persistence":
        preds_scaled = persistence_predict(Xte, target_idx)
        row["epochs_run"] = 0
    elif model_name == "moving_average":
        preds_scaled = moving_average_predict(Xte, target_idx)
        row["epochs_run"] = 0
    else:
        t0 = time.time()
        model = MODEL_CLASSES[model_name](num_features=len(feature_cols))
        model, trh, vlh, stopped = train_model(model, trl, vl, epochs=30, patience=6)
        preds_out, _ = evaluate_model(model, tel)
        preds_scaled = preds_out.flatten()
        row["epochs_run"] = stopped
        row["train_seconds"] = round(time.time() - t0, 1)
        row["final_train_loss"] = trh[-1]
        row["final_val_loss"] = vlh[-1]
        row["best_val_loss"] = min(vlh)

    yte_original = inverse_transform_target(scaler, feature_cols, yte)
    preds_original = inverse_transform_target(scaler, feature_cols, preds_scaled)
    metrics = compute_metrics_v2(yte_original, preds_original)
    row.update(metrics)
    return row


# Smoke test: one config from each experiment type
t0 = time.time()
r1 = run_one_config_multi("hold_out_year", 2023, 24, 24, "SWC_20 only", ["SWC_20"], "small_cnn")
print(f"Hold-out-year smoke test: RMSE={r1['RMSE']:.4f}, CORR={r1['CORR']:.4f}, {r1['train_seconds']}s")

r2 = run_one_config_multi("hold_out_station", "CB06", 24, 24, "SWC_20 only", ["SWC_20"], "small_cnn")
print(f"Hold-out-station smoke test: RMSE={r2['RMSE']:.4f}, CORR={r2['CORR']:.4f}, {r2['train_seconds']}s")
print(f"Total smoke test time: {time.time()-t0:.1f}s")


In [ ]:

# ============================================================
# Experiment A: Hold out a year (test on all 4 stations for that year)
# Test years: 2023 (driest), 2020 (near-median), 2018 (wettest)
# ============================================================
MULTI_TEST_YEARS = [2023, 2020, 2018]

rows_a = []
t_start = time.time()
total_configs = len(MULTI_TEST_YEARS) * len(WINDOW_SIZES) * len(HORIZONS) * len(FEATURE_SETS) * 4
done = 0

for test_year in MULTI_TEST_YEARS:
    for window_size in WINDOW_SIZES:
        for horizon in HORIZONS:
            for feat_name, feature_cols in FEATURE_SETS.items():
                for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
                    r = run_one_config_multi("hold_out_year", test_year,
                                             window_size, horizon, feat_name, feature_cols, model_name)
                    rows_a.append(r)
                    done += 1

    elapsed = time.time() - t_start
    print(f"  Year {test_year} done — {done}/{total_configs} configs, {elapsed:.0f}s elapsed")

df_a = pd.DataFrame(rows_a)
header = not os.path.exists(MULTI_RESULTS_CSV)
df_a.to_csv(MULTI_RESULTS_CSV, mode="a", header=header, index=False)
print(f"\nExperiment A complete: {len(rows_a)} configs in {time.time()-t_start:.0f}s")
print(f"Saved to {MULTI_RESULTS_CSV}")


In [ ]:

# ============================================================
# Experiment B: Hold out a station (test on all years of that station)
# ============================================================

rows_b = []
t_start = time.time()
total_configs = len(SELECTED_STATIONS) * len(WINDOW_SIZES) * len(HORIZONS) * len(FEATURE_SETS) * 4
done = 0

for test_station in SELECTED_STATIONS:
    for window_size in WINDOW_SIZES:
        for horizon in HORIZONS:
            for feat_name, feature_cols in FEATURE_SETS.items():
                for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
                    r = run_one_config_multi("hold_out_station", test_station,
                                             window_size, horizon, feat_name, feature_cols, model_name)
                    rows_b.append(r)
                    done += 1

    elapsed = time.time() - t_start
    print(f"  Station {test_station} done — {done}/{total_configs} configs, {elapsed:.0f}s elapsed")

df_b = pd.DataFrame(rows_b)
df_b.to_csv(MULTI_RESULTS_CSV, mode="a", header=False, index=False)
print(f"\nExperiment B complete: {len(rows_b)} configs in {time.time()-t_start:.0f}s")
print(f"Appended to {MULTI_RESULTS_CSV}")


In [ ]:

# ============================================================
# Extend multi-station data with Tair + add maxpool model
# ============================================================

# Extend STATION_YEAR_DATA with Tair (ffill/bfill for gaps)
for key, g in STATION_YEAR_DATA.items():
    for col in ["Tair", "SWC_5", "SWC_10", "SWC_50", "T_5", "T_10", "T_20", "T_50"]:
        if col in g.columns:
            g[col] = g[col].ffill().bfill().fillna(0.0)

# Check Tair coverage across stations
print("Tair coverage by station (year 2020 sample):")
for station in SELECTED_STATIONS:
    g = STATION_YEAR_DATA[(station, 2020)]
    tair_nz = (g["Tair"].abs() > 0.01).sum()
    print(f"  {station}: {tair_nz}/{len(g)} non-zero ({tair_nz/len(g)*100:.1f}%), "
          f"range=[{g['Tair'].min():.1f}, {g['Tair'].max():.1f}]")

# Check SWC_20 quality across stations
print("\nSWC_20 coverage by station (all years):")
for station in SELECTED_STATIONS:
    total = sum(len(STATION_YEAR_DATA[(station, y)]) for y in ALL_YEARS)
    nz = sum((STATION_YEAR_DATA[(station, y)]["SWC_20"].abs() > 0.001).sum() for y in ALL_YEARS)
    print(f"  {station}: {nz}/{total} non-zero ({nz/total*100:.1f}%)")

# Add maxpool CNN to MODEL_CLASSES
def MaxpoolCNN(num_features, kernel_size=3):
    return FlexCNN(num_features, n_layers=2, filters=[16, 32], kernel_size=3, pool_type='max')

MODEL_CLASSES["maxpool_cnn"] = MaxpoolCNN
print(f"\nMODEL_CLASSES: {list(MODEL_CLASSES.keys())}")
print(f"MaxpoolCNN params (F=1): {sum(p.numel() for p in MaxpoolCNN(1).parameters() if p.requires_grad)}")
print(f"MaxpoolCNN params (F=2): {sum(p.numel() for p in MaxpoolCNN(2).parameters() if p.requires_grad)}")


## Multi-Station Regression

### Experiment A: Hold-Out Year

In [ ]:

# ============================================================
# Stage 3, Experiment A: Hold-out year — H=24
# ============================================================
MULTI_TEST_YEARS = [2023, 2020, 2018]
STAGE3_HORIZONS = [24]
STAGE3_FEATURES = {
    "SWC_20 only":    ["SWC_20"],
    "SWC_20 + Tair":  ["SWC_20", "Tair"],
}
STAGE3_MODELS = ["persistence", "moving_average", "larger_cnn", "maxpool_cnn"]
STAGE3_CSV = "stage3_results.csv"

rows_a = []
t_start = time.time()

for test_year in MULTI_TEST_YEARS:
    for horizon in STAGE3_HORIZONS:
        for feat_name, feature_cols in STAGE3_FEATURES.items():
            for model_name in STAGE3_MODELS:
                r = run_one_config_multi("hold_out_year", test_year, 24, horizon,
                                         feat_name, feature_cols, model_name)
                rows_a.append(r)
                if model_name not in ("persistence", "moving_average"):
                    vl = r.get("final_val_loss", None)
                    print(f"  Y={test_year} H={horizon:3d} {feat_name:16s} {model_name:12s}: "
                          f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                          f"({r.get('train_seconds', 0):.0f}s)")
        # Baselines (printed once per year/horizon)
        bl_row = [r for r in rows_a if r["test_value"]==test_year and r["horizon"]==horizon
                  and r["model"]=="persistence"]
        if bl_row:
            print(f"  Y={test_year} H={horizon:3d} persistence: MSE={bl_row[0]['MSE']:.6f}")

    print(f"  --- Year {test_year} done ({time.time()-t_start:.0f}s) ---")

df_a24 = pd.DataFrame(rows_a)
df_a24.to_csv(STAGE3_CSV, index=False)
print(f"\nExp A H=24 done: {len(rows_a)} rows in {time.time()-t_start:.0f}s")


In [ ]:

# ============================================================
# Stage 3, Experiment A: Hold-out year — H=168
# ============================================================
rows_a168 = []
t_start = time.time()

for test_year in MULTI_TEST_YEARS:
    for feat_name, feature_cols in STAGE3_FEATURES.items():
        for model_name in STAGE3_MODELS:
            r = run_one_config_multi("hold_out_year", test_year, 24, 168,
                                     feat_name, feature_cols, model_name)
            rows_a168.append(r)
            if model_name not in ("persistence", "moving_average"):
                vl = r.get("final_val_loss", None)
                print(f"  Y={test_year} H=168 {feat_name:16s} {model_name:12s}: "
                      f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                      f"({r.get('train_seconds', 0):.0f}s)")
    bl = [r for r in rows_a168 if r["test_value"]==test_year and r["model"]=="persistence"]
    if bl:
        print(f"  Y={test_year} H=168 persistence: MSE={bl[0]['MSE']:.6f}")
    print(f"  --- Year {test_year} done ({time.time()-t_start:.0f}s) ---")

# Append to CSV
df_a168 = pd.DataFrame(rows_a168)
existing = pd.read_csv(STAGE3_CSV)
pd.concat([existing, df_a168], ignore_index=True).to_csv(STAGE3_CSV, index=False)
print(f"\nExp A H=168 done: {len(rows_a168)} rows in {time.time()-t_start:.0f}s")


### Experiment B: Hold-Out Station

In [ ]:

# ============================================================
# Stage 3, Experiment B: Hold-out station — H=24 (stations FD11, FD02)
# Split into 2-station chunks to avoid timeout
# ============================================================
rows_b24_1 = []
t_start = time.time()

for test_station in ["FD11", "FD02"]:
    for feat_name, feature_cols in STAGE3_FEATURES.items():
        for model_name in STAGE3_MODELS:
            r = run_one_config_multi("hold_out_station", test_station, 24, 24,
                                     feat_name, feature_cols, model_name)
            rows_b24_1.append(r)
            if model_name not in ("persistence", "moving_average"):
                vl = r.get("final_val_loss", None)
                print(f"  Stn={test_station} H= 24 {feat_name:16s} {model_name:12s}: "
                      f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                      f"({r.get('train_seconds', 0):.0f}s)")
    bl = [r for r in rows_b24_1 if r["test_value"]==test_station and r["model"]=="persistence"]
    if bl:
        print(f"  Stn={test_station} H= 24 persistence: MSE={bl[0]['MSE']:.6f}")
    print(f"  --- Station {test_station} done ({time.time()-t_start:.0f}s) ---")

df_b24_1 = pd.DataFrame(rows_b24_1)
existing = pd.read_csv(STAGE3_CSV)
pd.concat([existing, df_b24_1], ignore_index=True).to_csv(STAGE3_CSV, index=False)
print(f"\nExp B H=24 (FD11, FD02) done: {len(rows_b24_1)} rows in {time.time()-t_start:.0f}s")


In [ ]:

# ============================================================
# Stage 3, Experiment B: Hold-out station — H=24 (stations CB06, CB25)
# ============================================================
rows_b24_2 = []
t_start = time.time()

for test_station in ["CB06", "CB25"]:
    for feat_name, feature_cols in STAGE3_FEATURES.items():
        for model_name in STAGE3_MODELS:
            r = run_one_config_multi("hold_out_station", test_station, 24, 24,
                                     feat_name, feature_cols, model_name)
            rows_b24_2.append(r)
            if model_name not in ("persistence", "moving_average"):
                vl = r.get("final_val_loss", None)
                print(f"  Stn={test_station} H= 24 {feat_name:16s} {model_name:12s}: "
                      f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                      f"({r.get('train_seconds', 0):.0f}s)")
    bl = [r for r in rows_b24_2 if r["test_value"]==test_station and r["model"]=="persistence"]
    if bl:
        print(f"  Stn={test_station} H= 24 persistence: MSE={bl[0]['MSE']:.6f}")
    print(f"  --- Station {test_station} done ({time.time()-t_start:.0f}s) ---")

df_b24_2 = pd.DataFrame(rows_b24_2)
existing = pd.read_csv(STAGE3_CSV)
pd.concat([existing, df_b24_2], ignore_index=True).to_csv(STAGE3_CSV, index=False)
print(f"\nExp B H=24 (CB06, CB25) done: {len(rows_b24_2)} rows in {time.time()-t_start:.0f}s")


In [ ]:

# ============================================================
# Stage 3, Experiment B: Hold-out station — H=168 (stations FD11, FD02)
# ============================================================
rows_b168_1 = []
t_start = time.time()

for test_station in ["FD11", "FD02"]:
    for feat_name, feature_cols in STAGE3_FEATURES.items():
        for model_name in STAGE3_MODELS:
            r = run_one_config_multi("hold_out_station", test_station, 24, 168,
                                     feat_name, feature_cols, model_name)
            rows_b168_1.append(r)
            if model_name not in ("persistence", "moving_average"):
                vl = r.get("final_val_loss", None)
                print(f"  Stn={test_station} H=168 {feat_name:16s} {model_name:12s}: "
                      f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                      f"({r.get('train_seconds', 0):.0f}s)")
    bl = [r for r in rows_b168_1 if r["test_value"]==test_station and r["model"]=="persistence"]
    if bl:
        print(f"  Stn={test_station} H=168 persistence: MSE={bl[0]['MSE']:.6f}")
    print(f"  --- Station {test_station} done ({time.time()-t_start:.0f}s) ---")

df_b168_1 = pd.DataFrame(rows_b168_1)
existing = pd.read_csv(STAGE3_CSV)
pd.concat([existing, df_b168_1], ignore_index=True).to_csv(STAGE3_CSV, index=False)
print(f"\nExp B H=168 (FD11, FD02) done: {len(rows_b168_1)} rows in {time.time()-t_start:.0f}s")


### Regression Results Analysis

In [ ]:

# ============================================================
# Stage 3, Experiment B: Hold-out station — H=168 (stations CB06, CB25)
# ============================================================
rows_b168_2 = []
t_start = time.time()

for test_station in ["CB06", "CB25"]:
    for feat_name, feature_cols in STAGE3_FEATURES.items():
        for model_name in STAGE3_MODELS:
            r = run_one_config_multi("hold_out_station", test_station, 24, 168,
                                     feat_name, feature_cols, model_name)
            rows_b168_2.append(r)
            if model_name not in ("persistence", "moving_average"):
                vl = r.get("final_val_loss", None)
                print(f"  Stn={test_station} H=168 {feat_name:16s} {model_name:12s}: "
                      f"val={vl:.6f}, MSE={r['MSE']:.6f}, CORR={r['CORR']:.4f} "
                      f"({r.get('train_seconds', 0):.0f}s)")
    bl = [r for r in rows_b168_2 if r["test_value"]==test_station and r["model"]=="persistence"]
    if bl:
        print(f"  Stn={test_station} H=168 persistence: MSE={bl[0]['MSE']:.6f}")
    print(f"  --- Station {test_station} done ({time.time()-t_start:.0f}s) ---")

df_b168_2 = pd.DataFrame(rows_b168_2)
existing = pd.read_csv(STAGE3_CSV)
pd.concat([existing, df_b168_2], ignore_index=True).to_csv(STAGE3_CSV, index=False)
print(f"\nExp B H=168 (CB06, CB25) done: {len(rows_b168_2)} rows in {time.time()-t_start:.0f}s")


In [ ]:

# ============================================================
# Stage 3 Results Analysis
# ============================================================
import matplotlib.pyplot as plt

s3 = pd.read_csv(STAGE3_CSV)
print(f"Stage 3 results: {len(s3)} rows")
print(f"Experiments: {s3['experiment'].value_counts().to_dict()}")

# ---- Experiment A: Hold-out year summary ----
ea = s3[s3["experiment"] == "hold_out_year"]

print("\n" + "="*110)
print("EXPERIMENT A: HOLD-OUT YEAR — Per-year results (W=24)")
print("="*110)

for horizon in [24, 168]:
    print(f"\n--- H={horizon} ---")
    print(f"{'Test Year':>10s} {'Features':>16s} {'Model':>14s} {'Val Loss':>10s} "
          f"{'Test MSE':>10s} {'RMSE':>8s} {'CORR':>7s}")
    print("-"*85)
    sub = ea[ea["horizon"] == horizon]
    for ty in MULTI_TEST_YEARS:
        for fn in ["SWC_20 only", "SWC_20 + Tair"]:
            for mn in STAGE3_MODELS:
                row = sub[(sub["test_value"]==ty) & (sub["features"]==fn) & (sub["model"]==mn)]
                if row.empty:
                    continue
                row = row.iloc[0]
                vl = row.get("final_val_loss", None)
                vl_str = f"{vl:.6f}" if pd.notna(vl) and vl else "     --"
                rmse = np.sqrt(row["MSE"])
                print(f"{ty:>10d} {fn:>16s} {mn:>14s} {vl_str:>10s} "
                      f"{row['MSE']:10.6f} {rmse:8.4f} {row['CORR']:7.4f}")
        print()

# ---- Experiment B: Hold-out station summary ----
eb = s3[s3["experiment"] == "hold_out_station"]

print("\n" + "="*110)
print("EXPERIMENT B: HOLD-OUT STATION — Per-station results (W=24)")
print("="*110)

for horizon in [24, 168]:
    print(f"\n--- H={horizon} ---")
    print(f"{'Station':>10s} {'Features':>16s} {'Model':>14s} {'Val Loss':>10s} "
          f"{'Test MSE':>10s} {'RMSE':>8s} {'CORR':>7s}")
    print("-"*85)
    sub = eb[eb["horizon"] == horizon]
    for ts in SELECTED_STATIONS:
        for fn in ["SWC_20 only", "SWC_20 + Tair"]:
            for mn in STAGE3_MODELS:
                row = sub[(sub["test_value"]==ts) & (sub["features"]==fn) & (sub["model"]==mn)]
                if row.empty:
                    continue
                row = row.iloc[0]
                vl = row.get("final_val_loss", None)
                vl_str = f"{vl:.6f}" if pd.notna(vl) and vl else "     --"
                rmse = np.sqrt(row["MSE"])
                print(f"{ts:>10s} {fn:>16s} {mn:>14s} {vl_str:>10s} "
                      f"{row['MSE']:10.6f} {rmse:8.4f} {row['CORR']:7.4f}")
        print()

# ---- Aggregate comparison: CNN vs persistence ----
print("\n" + "="*90)
print("CNN vs PERSISTENCE — % MSE improvement (negative = CNN worse)")
print("="*90)

for exp_type, label, test_col in [("hold_out_year", "Exp A", MULTI_TEST_YEARS),
                                   ("hold_out_station", "Exp B", SELECTED_STATIONS)]:
    sub = s3[s3["experiment"] == exp_type]
    print(f"\n{label}:")
    for horizon in [24, 168]:
        print(f"  H={horizon}:")
        for fn in ["SWC_20 only", "SWC_20 + Tair"]:
            for mn in ["larger_cnn", "maxpool_cnn"]:
                improvements = []
                for tv in test_col:
                    pers = sub[(sub["test_value"]==tv) & (sub["horizon"]==horizon) &
                               (sub["model"]=="persistence")]
                    cnn = sub[(sub["test_value"]==tv) & (sub["horizon"]==horizon) &
                              (sub["features"]==fn) & (sub["model"]==mn)]
                    if not pers.empty and not cnn.empty:
                        p_mse = pers.iloc[0]["MSE"]
                        c_mse = cnn.iloc[0]["MSE"]
                        improvements.append((c_mse - p_mse) / p_mse * 100)
                if improvements:
                    avg_imp = np.mean(improvements)
                    print(f"    {fn:16s} {mn:12s}: avg {avg_imp:+.1f}%  "
                          f"(per test: {', '.join(f'{x:+.1f}%' for x in improvements)})")


### Regression Figures

In [ ]:

# ============================================================
# Actual-vs-Predicted Plots (3-4 month windows)
# Best model from Stage 3 results
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def generate_predictions_timeseries(experiment_type, test_value, window_size,
                                     horizon, feature_cols, model_name, seed=SEED):
    """Generate full prediction time series for plotting."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    train_keys, val_keys, test_keys = get_fold_keys(experiment_type, test_value)
    trl, vl, tel, (Xtr, ytr, Xv, yv, Xte, yte), scaler = build_multi_fold(
        train_keys, val_keys, test_keys, feature_cols, window_size, horizon, 128, seed)

    target_idx = feature_cols.index("SWC_20")

    if model_name == "persistence":
        preds_scaled = persistence_predict(Xte, target_idx)
    elif model_name == "moving_average":
        preds_scaled = moving_average_predict(Xte, target_idx)
    else:
        model = MODEL_CLASSES[model_name](num_features=len(feature_cols))
        model, trh, vlh, stopped = train_model(model, trl, vl, epochs=30, patience=6)
        preds_out, _ = evaluate_model(model, tel)
        preds_scaled = preds_out.flatten()

    yte_orig = inverse_transform_target(scaler, feature_cols, yte)
    preds_orig = inverse_transform_target(scaler, feature_cols, preds_scaled)
    return yte_orig, preds_orig


def plot_actual_vs_predicted(test_year, horizon, feature_cols, feat_name, model_name,
                              months_to_show=3, start_month=3):
    """Plot actual vs predicted for a specific time window."""
    yte_orig, preds_orig = generate_predictions_timeseries(
        "hold_out_year", test_year, 24, horizon, feature_cols, model_name)

    # Build timestamp index for test windows
    # Each window's target is at: window_start + window_size + horizon - 1
    timestamps = []
    for station in SELECTED_STATIONS:
        g = STATION_YEAR_DATA[(station, test_year)]
        ts = g["Timestamp"].values
        n_windows = len(ts) - 24 - horizon + 1
        for i in range(n_windows):
            timestamps.append(ts[i + 24 + horizon - 1])
    timestamps = np.array(timestamps, dtype="datetime64[ns]")

    # Filter to requested month range
    ts_pd = pd.to_datetime(timestamps)
    mask = (ts_pd.month >= start_month) & (ts_pd.month < start_month + months_to_show)

    ts_plot = ts_pd[mask]
    y_plot = yte_orig[mask]
    p_plot = preds_orig[mask]

    metrics = compute_metrics(y_plot, p_plot)
    return ts_plot, y_plot, p_plot, metrics


# Generate 4-panel figure: H=24 and H=168, best CNN vs persistence
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

configs = [
    (2023, 24, ["SWC_20", "Tair"], "SWC_20 + Tair", "maxpool_cnn", 3, 4, "Mar-Jun 2023"),
    (2023, 24, ["SWC_20"], "SWC_20 only", "persistence", 3, 4, "Mar-Jun 2023"),
    (2023, 168, ["SWC_20", "Tair"], "SWC_20 + Tair", "maxpool_cnn", 3, 4, "Mar-Jun 2023"),
    (2023, 168, ["SWC_20"], "SWC_20 only", "persistence", 3, 4, "Mar-Jun 2023"),
]

for idx, (year, horizon, fcols, fname, mname, sm, nm, period) in enumerate(configs):
    ax = axes[idx // 2][idx % 2]
    ts, y, p, m = plot_actual_vs_predicted(year, horizon, fcols, fname, mname,
                                            months_to_show=nm, start_month=sm)
    ax.plot(ts, y, 'b-', linewidth=0.6, alpha=0.8, label='Actual')
    ax.plot(ts, p, 'r-', linewidth=0.6, alpha=0.7, label='Predicted')
    ax.set_title(f"H={horizon}h, {mname} ({fname})\n"
                 f"MSE={m['MSE']:.6f}, CORR={m['CORR']:.4f}", fontsize=9)
    ax.set_ylabel("SWC_20")
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.grid(alpha=0.3)

plt.suptitle(f"Actual vs Predicted — Multi-Station Hold-Out Year 2023 ({period})",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("actual_vs_predicted_2023.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved actual_vs_predicted_2023.png")


In [ ]:

# ============================================================
# Additional PDF-quality figures for research report
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Figure 1: Multi-station comparison bar chart
fig1, axes1 = plt.subplots(1, 2, figsize=(14, 5))
s3 = pd.read_csv(STAGE3_CSV)

for ax, horizon, title in zip(axes1, [24, 168],
                               ["H=24 (1-day)", "H=168 (1-week)"]):
    ea = s3[(s3["experiment"]=="hold_out_year") & (s3["horizon"]==horizon)]

    # Average MSE across test years for each model/feature combo
    groups = []
    labels = []
    colors_list = []
    color_map = {"persistence": "#999999", "moving_average": "#bbbbbb",
                 "larger_cnn": "#4c72b0", "maxpool_cnn": "#dd8452"}

    for fn in ["SWC_20 only", "SWC_20 + Tair"]:
        for mn in STAGE3_MODELS:
            sub = ea[(ea["features"]==fn) & (ea["model"]==mn)]
            if sub.empty:
                # baselines only have SWC_20 only
                sub = ea[(ea["model"]==mn)]
            if sub.empty:
                continue
            avg_mse = sub["MSE"].mean()
            groups.append(avg_mse)
            lbl = f"{mn}\n({fn})" if mn not in ("persistence", "moving_average") else mn
            labels.append(lbl)
            colors_list.append(color_map.get(mn, "#4c72b0"))

    # Remove duplicate baselines
    seen = set()
    filtered = [(l, g, c) for l, g, c in zip(labels, groups, colors_list)
                if l not in seen and not seen.add(l)]
    labels, groups, colors_list = zip(*filtered)

    y_pos = range(len(labels))
    bars = ax.barh(y_pos, groups, color=colors_list, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel("Test MSE (avg across 3 test years)")
    ax.set_title(f"Exp A: Hold-Out Year — {title}")
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig("stage3_expA_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Figure 2: Station-level hold-out comparison
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

for ax, horizon in zip(axes2, [24, 168]):
    eb = s3[(s3["experiment"]=="hold_out_station") & (s3["horizon"]==horizon)]

    x = np.arange(len(SELECTED_STATIONS))
    width = 0.2

    for i, (mn, color) in enumerate([("persistence", "#999999"), ("larger_cnn", "#4c72b0"),
                                      ("maxpool_cnn", "#dd8452")]):
        mses = []
        for stn in SELECTED_STATIONS:
            # Use SWC_20 + Tair for CNN, SWC_20 only for persistence
            if mn == "persistence":
                row = eb[(eb["test_value"]==stn) & (eb["model"]==mn)]
            else:
                row = eb[(eb["test_value"]==stn) & (eb["model"]==mn) &
                         (eb["features"]=="SWC_20 + Tair")]
            mses.append(row.iloc[0]["MSE"] if not row.empty else 0)
        ax.bar(x + (i-1)*width, mses, width, label=mn, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(SELECTED_STATIONS)
    ax.set_ylabel("Test MSE")
    ax.set_title(f"Exp B: Hold-Out Station — H={horizon}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("stage3_expB_stations.png", dpi=150, bbox_inches="tight")
plt.show()

# Save all figures to PDF
with PdfPages("stage3_figures.pdf") as pdf:
    for fignum in plt.get_fignums():
        pdf.savefig(plt.figure(fignum), bbox_inches="tight")
    print(f"Saved {len(plt.get_fignums())} figures to stage3_figures.pdf")

print("\nAll Stage 3 figures saved:")
print("  stage3_expA_comparison.png")
print("  stage3_expB_stations.png")
print("  stage3_figures.pdf")


In [ ]:

# Quick state check
import os
print("stage3_results.csv exists?", os.path.exists("stage3_results.csv"))
if os.path.exists("stage3_results.csv"):
    df_check = pd.read_csv("stage3_results.csv")
    print(f"  {len(df_check)} rows saved")
    print(df_check[["experiment","test_value","horizon","features","model"]].to_string())

print("\nrun_one_config_multi defined?", "run_one_config_multi" in dir())
print("MODEL_CLASSES:", list(MODEL_CLASSES.keys()) if "MODEL_CLASSES" in dir() else "NOT DEFINED")
print("STAGE3_FEATURES defined?", "STAGE3_FEATURES" in dir())
print("FlexCNN defined?", "FlexCNN" in dir())


In [ ]:

# ============================================================
# Results aggregation and stability analysis
# ============================================================
import matplotlib.pyplot as plt

df_results = pd.read_csv(MULTI_RESULTS_CSV)
print(f"Total results: {len(df_results)} rows")
print(f"Experiments: {df_results['experiment'].value_counts().to_dict()}")

# --- Experiment A summary: CNN vs baselines, averaged across held-out years ---
df_ea = df_results[df_results["experiment"] == "hold_out_year"]
cnn_models = ["small_cnn", "larger_cnn"]

print("\n=== Experiment A: Hold-Out Year — Average metrics across 3 test years ===")
for feat_name in FEATURE_SETS:
    print(f"\n  Features: {feat_name}")
    for ws in WINDOW_SIZES:
        for hz in HORIZONS:
            sub = df_ea[(df_ea["features"] == feat_name) & (df_ea["window_size"] == ws) & (df_ea["horizon"] == hz)]
            if sub.empty:
                continue
            for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
                ms = sub[sub["model"] == model_name]
                if ms.empty:
                    continue
                print(f"    W={ws:3d} H={hz:3d} {model_name:15s}  "
                      f"RMSE={ms['RMSE'].mean():.4f}±{ms['RMSE'].std():.4f}  "
                      f"MAE={ms['MAE'].mean():.4f}±{ms['MAE'].std():.4f}  "
                      f"CORR={ms['CORR'].mean():.4f}±{ms['CORR'].std():.4f}")

# --- Experiment B summary: CNN vs baselines, averaged across held-out stations ---
df_eb = df_results[df_results["experiment"] == "hold_out_station"]

print("\n\n=== Experiment B: Hold-Out Station — Average metrics across 4 test stations ===")
for feat_name in FEATURE_SETS:
    print(f"\n  Features: {feat_name}")
    for ws in WINDOW_SIZES:
        for hz in HORIZONS:
            sub = df_eb[(df_eb["features"] == feat_name) & (df_eb["window_size"] == ws) & (df_eb["horizon"] == hz)]
            if sub.empty:
                continue
            for model_name in ["persistence", "moving_average", "small_cnn", "larger_cnn"]:
                ms = sub[sub["model"] == model_name]
                if ms.empty:
                    continue
                print(f"    W={ws:3d} H={hz:3d} {model_name:15s}  "
                      f"RMSE={ms['RMSE'].mean():.4f}±{ms['RMSE'].std():.4f}  "
                      f"MAE={ms['MAE'].mean():.4f}±{ms['MAE'].std():.4f}  "
                      f"CORR={ms['CORR'].mean():.4f}±{ms['CORR'].std():.4f}")


In [ ]:

# ============================================================
# Stability comparison: Exp A vs Exp B error spread
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Focus on window=24, horizon=24, SWC_20 only for clarity
focus = df_results[(df_results["window_size"] == 24) & (df_results["horizon"] == 24) &
                   (df_results["features"] == "SWC_20 only")]

for ax, exp_type, title in zip(axes, ["hold_out_year", "hold_out_station"],
                                ["Exp A: Hold-Out Year", "Exp B: Hold-Out Station"]):
    sub = focus[focus["experiment"] == exp_type]
    models = ["persistence", "moving_average", "small_cnn", "larger_cnn"]
    positions = range(len(models))
    rmse_data = [sub[sub["model"] == m]["RMSE"].values for m in models]
    ax.bar(positions, [d.mean() for d in rmse_data],
           yerr=[d.std() for d in rmse_data], capsize=5, alpha=0.7,
           color=["#999999", "#999999", "#4c72b0", "#dd8452"])
    ax.set_xticks(positions)
    ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylabel("RMSE (original SWC_20 units)")
    ax.set_title(f"{title}\n(W=24, H=24, SWC_20 only)")

plt.tight_layout()
plt.savefig("multi_station_rmse_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved multi_station_rmse_comparison.png")

# CORR comparison
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
for ax, exp_type, title in zip(axes2, ["hold_out_year", "hold_out_station"],
                                 ["Exp A: Hold-Out Year", "Exp B: Hold-Out Station"]):
    sub = focus[focus["experiment"] == exp_type]
    models = ["persistence", "moving_average", "small_cnn", "larger_cnn"]
    corr_data = [sub[sub["model"] == m]["CORR"].values for m in models]
    ax.bar(range(len(models)), [d.mean() for d in corr_data],
           yerr=[d.std() for d in corr_data], capsize=5, alpha=0.7,
           color=["#999999", "#999999", "#4c72b0", "#dd8452"])
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=20, ha="right")
    ax.set_ylabel("CORR")
    ax.set_title(f"{title}\n(W=24, H=24, SWC_20 only)")
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("multi_station_corr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved multi_station_corr_comparison.png")


In [ ]:

# Quick runtime check
import os
print("Runtime alive")
print("MULTI_RESULTS_CSV exists?", os.path.exists("multi_station_results.csv"))
try:
    print("STATION_YEAR_DATA keys:", len(STATION_YEAR_DATA))
    print("DEVICE:", DEVICE)
except NameError as e:
    print("Variables not defined:", e)


In [ ]:

# Dataset column inventory for feature engineering planning
import pandas as pd
df_all = pd.read_parquet("data/TxSON33_modeling_complete.parquet")
print("All columns:")
for i, col in enumerate(df_all.columns):
    print(f"  {i:2d}. {col}")

# Check which columns have data for FD11
fd11_sample = df_all[df_all["Station"] == "FD11"].iloc[:100]
print("\nFD11 sample non-null counts (first 100 rows):")
print(fd11_sample.notna().sum().to_string())


## 12-Class Classification

Fixed 12 bins (0.05 width each, SWC range 0.00–0.60). Same boundaries for every station.

### Classification Pipeline

Defines `ClassificationCNN`, `swc_to_class()` (12 fixed bins), `train_classifier()`, `predict_classes()`, `compute_classification_metrics()` (accuracy, macro F1, weighted F1, within-1 accuracy, MACE).

In [ ]:

# ============================================================
# 19.1  Classification setup: binning, metrics, baselines
# ============================================================
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# --- 12-class bin definitions ---
N_CLASSES = 12
BIN_WIDTH = 0.05
BIN_EDGES = np.arange(0.0, 0.65, BIN_WIDTH)  # [0.00, 0.05, ..., 0.60]

def swc_to_class(values):
    """Convert continuous SWC_20 values to class indices 0–11.
    Values exactly at 0.60 go to class 11. Values outside [0.0, 0.60] are
    clipped and flagged."""
    values = np.asarray(values, dtype=np.float64)
    clipped = np.clip(values, 0.0, 0.60)
    classes = np.floor(clipped / BIN_WIDTH).astype(int)
    classes = np.clip(classes, 0, N_CLASSES - 1)
    return classes

def class_to_center(classes):
    """Convert class indices back to bin center values."""
    return (np.asarray(classes) * BIN_WIDTH) + BIN_WIDTH / 2

# --- Check target distribution ---
print("Target value distribution check (FD11, all years):")
all_targets = []
for y in ALL_YEARS:
    g = YEAR_FEATURES[y]
    all_targets.extend(g["SWC_20"].values)
all_targets = np.array(all_targets)

print(f"  Range: [{all_targets.min():.4f}, {all_targets.max():.4f}]")
print(f"  Values < 0.00: {(all_targets < 0.0).sum()}")
print(f"  Values > 0.60: {(all_targets > 0.60).sum()}")
print(f"  Values == 0.60: {(all_targets == 0.60).sum()}")

# Class distribution
classes_all = swc_to_class(all_targets)
print(f"\nClass distribution (all FD11 data):")
for c in range(N_CLASSES):
    count = (classes_all == c).sum()
    pct = count / len(classes_all) * 100
    lo = c * BIN_WIDTH
    hi = lo + BIN_WIDTH
    print(f"  Class {c:2d} [{lo:.2f}, {hi:.2f}): {count:6d} ({pct:5.1f}%)")

# --- Classification-specific metrics ---
def compute_classification_metrics(y_true_classes, y_pred_classes, n_classes=N_CLASSES):
    """Compute full classification metrics suite."""
    acc = accuracy_score(y_true_classes, y_pred_classes)
    macro_p = precision_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    macro_r = recall_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true_classes, y_pred_classes, average='weighted', zero_division=0)

    # Within-one-category accuracy
    diff = np.abs(y_true_classes - y_pred_classes)
    within_one = (diff <= 1).mean()

    # Mean absolute class-index error
    mean_abs_class_err = diff.mean()

    cm = confusion_matrix(y_true_classes, y_pred_classes, labels=range(n_classes))

    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "within_one_acc": within_one,
        "mean_abs_class_error": mean_abs_class_err,
        "confusion_matrix": cm,
    }

# --- Classification baselines ---
def persistence_classify(X, target_idx):
    """Predict: last observed SWC_20 value's class."""
    last_vals = X[:, -1, target_idx]
    return swc_to_class(last_vals)

def majority_class_predict(y_train_classes, n_test):
    """Predict: most common class in training set."""
    majority = np.bincount(y_train_classes).argmax()
    return np.full(n_test, majority)

# Sanity checks
assert swc_to_class([0.0])[0] == 0
assert swc_to_class([0.049])[0] == 0
assert swc_to_class([0.05])[0] == 1
assert swc_to_class([0.599])[0] == 11
assert swc_to_class([0.60])[0] == 11
print("\nBinning sanity checks passed")
print(f"Bin edges: {BIN_EDGES}")


In [ ]:

# ============================================================
# CLASS DISTRIBUTION BAR PLOTS — Priority 2
# Before any training: visualize the 12-class distribution
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 10})

# --- FD11 class distribution ---
all_swc = pd.concat([YEAR_FEATURES[y]["SWC_20"] for y in ALL_YEARS])
classes_fd11 = swc_to_class(all_swc.values)
counts_fd11 = np.bincount(classes_fd11, minlength=N_CLASSES)
total_fd11 = len(classes_fd11)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(N_CLASSES)
bars = ax.bar(x, counts_fd11, color='#4c72b0', edgecolor='white', linewidth=0.5)

# Highlight empty/near-empty classes
for i, (bar, count) in enumerate(zip(bars, counts_fd11)):
    if count < total_fd11 * 0.01:  # < 1%
        bar.set_color('#d62728')
        bar.set_alpha(0.5)

# Labels
labels = [f"[{c*0.05:.2f},\n{(c+1)*0.05:.2f})" for c in range(N_CLASSES)]
labels[-1] = f"[{10*0.05:.2f},\n{11*0.05:.2f}]"  # inclusive upper bound
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_xlabel("Soil Moisture Category (SWC_20)")
ax.set_ylabel("Number of Observations")
ax.set_title(f"12-Class Distribution — Station FD11 (n={total_fd11:,})")

# Add count labels on bars
for i, (count, bar) in enumerate(zip(counts_fd11, bars)):
    pct = count / total_fd11 * 100
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_fd11*0.005,
                f"{count:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=7)
    else:
        ax.text(bar.get_x() + bar.get_width()/2, total_fd11*0.005,
                "0", ha='center', va='bottom', fontsize=7, color='red')

ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("class_distribution_fd11.png", dpi=150, bbox_inches="tight")
plt.savefig("class_distribution_fd11.pdf", bbox_inches="tight")
plt.show()
print("Saved class_distribution_fd11.png and .pdf")

# Print imbalance summary
print(f"\nImbalance summary:")
print(f"  Empty classes (0 obs): {(counts_fd11 == 0).sum()} — classes {np.where(counts_fd11 == 0)[0].tolist()}")
print(f"  Near-empty (<1%): {(counts_fd11 < total_fd11 * 0.01).sum()} classes")
print(f"  Dominant class: {counts_fd11.argmax()} with {counts_fd11.max():,} obs ({counts_fd11.max()/total_fd11*100:.1f}%)")
print(f"  Top 5 classes cover: {sorted(counts_fd11, reverse=True)[:5]}")
print(f"  Effective range: classes {np.min(np.where(counts_fd11 > 100))} to {np.max(np.where(counts_fd11 > 100))}")


In [ ]:

# ============================================================
# Multi-station class distribution comparison
# ============================================================
import matplotlib.pyplot as plt

SELECTED_STATIONS = ["FD11", "FD02", "CB06", "CB25"]

# Compute class distributions for each station
station_distributions = {}
for station in SELECTED_STATIONS:
    sdf = df_all[(df_all["Station"] == station) &
                 (df_all["Timestamp"].dt.year.isin(ALL_YEARS))]
    vals = sdf["SWC_20"].values
    classes = swc_to_class(vals)
    counts = np.bincount(classes, minlength=N_CLASSES)
    station_distributions[station] = {
        "counts": counts, "total": len(vals),
        "range": (vals.min(), vals.max()),
        "pcts": counts / len(vals) * 100
    }
    print(f"{station}: n={len(vals):,}, SWC_20 range=[{vals.min():.4f}, {vals.max():.4f}]")
    print(f"  Empty classes: {np.where(counts == 0)[0].tolist()}")
    print(f"  Dominant class: {counts.argmax()} ({counts.max()/len(vals)*100:.1f}%)")

# --- Individual station plots ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_station = {"FD11": "#4c72b0", "FD02": "#dd8452", "CB06": "#55a868", "CB25": "#c44e52"}

for ax, station in zip(axes.flat, SELECTED_STATIONS):
    d = station_distributions[station]
    bars = ax.bar(range(N_CLASSES), d["pcts"],
                  color=colors_station[station], edgecolor='white', linewidth=0.5)
    for i, bar in enumerate(bars):
        if d["counts"][i] < d["total"] * 0.01:
            bar.set_alpha(0.4)
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels([f"[{c*0.05:.2f},\n{(c+1)*0.05:.2f})" for c in range(N_CLASSES)],
                       fontsize=6)
    ax.set_ylabel("% of Observations")
    ax.set_title(f"Station {station} (n={d['total']:,})")
    ax.grid(alpha=0.3, axis='y')

plt.suptitle("12-Class SWC_20 Distribution by Station", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("class_distribution_all_stations.png", dpi=150, bbox_inches="tight")
plt.savefig("class_distribution_all_stations.pdf", bbox_inches="tight")
plt.show()
print("Saved class_distribution_all_stations.png/pdf")

# --- Grouped comparison ---
fig2, ax2 = plt.subplots(figsize=(12, 5))
x = np.arange(N_CLASSES)
width = 0.2
for i, station in enumerate (SELECTED_STATIONS):
    d = station_distributions[station]
    ax2.bar(x + i*width - 1.5*width, d["pcts"], width,
            label=station, color=colors_station[station], alpha=0.85)

ax2.set_xticks(x)
ax2.set_xticklabels([f"[{c*0.05:.2f},{(c+1)*0.05:.2f})" for c in range(N_CLASSES)],
                     rotation=45, ha='right', fontsize=7)
ax2.set_xlabel("Soil Moisture Category")
ax2.set_ylabel("% of Observations")
ax2.set_title("Cross-Station Class Distribution Comparison")
ax2.legend()
ax2.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("class_distribution_comparison.png", dpi=150, bbox_inches="tight")
plt.savefig("class_distribution_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved class_distribution_comparison.png/pdf")


In [ ]:

# ============================================================
# 19.2  Classification CNN model
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

class ClassificationCNN(nn.Module):
    """Same architecture as LargerCNN but with 12-class softmax output."""
    def __init__(self, num_features, n_classes=12, n_layers=2, filters=None,
                 kernel_size=3, pool_type='avg', dropout=0.0):
        super().__init__()
        if filters is None:
            filters = [16, 32][:n_layers]
        p = kernel_size // 2
        layers = []
        in_ch = num_features
        for f in filters:
            layers.append(nn.Conv1d(in_ch, f, kernel_size, padding=p))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = f
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)
        hidden = max(filters[-1] // 2, 4)
        self.fc1 = nn.Linear(filters[-1], hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.convs(x)
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)  # raw logits; cross-entropy loss applies softmax internally


def train_classifier(model, train_loader, val_loader, epochs=40, patience=8,
                     lr=1e-3, class_weights=None):
    """Train a classification CNN with cross-entropy loss."""
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))
    else:
        criterion = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    best_val, best_state, bad_epochs, stopped_epoch = float("inf"), None, 0, epochs

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_losses.append(running / len(train_loader.dataset))

        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                running += criterion(model(xb), yb).item() * len(xb)
        val_loss = running / len(val_loader.dataset)
        val_losses.append(val_loss)

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                stopped_epoch = epoch + 1
                break

    model.load_state_dict(best_state)
    return model, train_losses, val_losses, stopped_epoch


@torch.no_grad()
def predict_classes(model, loader):
    """Get predicted class indices and true class indices."""
    model.eval()
    all_logits, all_labels = [], []
    for xb, yb in loader:
        logits = model(xb.to(DEVICE)).cpu()
        all_logits.append(logits)
        all_labels.append(yb)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    preds = logits.argmax(dim=1).numpy()
    probs = F.softmax(logits, dim=1).numpy()
    return preds, labels.numpy(), probs


# Build classification data loaders (targets are class indices, not continuous values)
def build_classification_fold(test_year, feature_cols, window_size, horizon,
                               batch_size, seed=SEED):
    """Same as build_fold but targets are class indices (long) for cross-entropy."""
    train_years = [y for y in ALL_YEARS if y not in (test_year, VAL_YEAR)]
    scaler = MinMaxScaler()
    scaler.fit(pd.concat([YEAR_FEATURES[y][feature_cols] for y in train_years], axis=0))

    def scaled_windows(years):
        Xs, ys = [], []
        for y in years:
            g = YEAR_FEATURES[y].copy()
            # Get original-scale targets BEFORE scaling
            _, y_orig = make_windows_for_year(g, feature_cols, window_size, horizon)
            # Scale features
            g[feature_cols] = scaler.transform(g[feature_cols])
            X_scaled, _ = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X_scaled):
                Xs.append(X_scaled)
                ys.append(y_orig)
        return np.concatenate(Xs), np.concatenate(ys)

    Xtr, ytr_orig = scaled_windows(train_years)
    Xv, yv_orig = scaled_windows([VAL_YEAR])
    Xte, yte_orig = scaled_windows([test_year])

    # Convert to class indices
    ytr_cls = swc_to_class(ytr_orig)
    yv_cls = swc_to_class(yv_orig)
    yte_cls = swc_to_class(yte_orig)

    g = torch.Generator().manual_seed(seed)
    def loader(X, y_cls, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y_cls, dtype=torch.long))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                         generator=g if shuffle else None)

    return (loader(Xtr, ytr_cls, True), loader(Xv, yv_cls, False),
            loader(Xte, yte_cls, False),
            (Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig),
            scaler)

# Quick test
m = ClassificationCNN(num_features=1)
n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"ClassificationCNN params (F=1): {n_params}")
print(f"Output shape test: {m(torch.randn(2, 24, 1)).shape}")  # should be (2, 12)


In [ ]:

# ============================================================
# Priority 3: Verify label alignment
# Priority 4: Train ONE small classifier end-to-end
# ============================================================

# --- VERIFICATION: target alignment ---
# The label must come from the FUTURE target, not the last window timestep
feature_cols = ["SWC_20"]
trl_v, vl_v, tel_v, raw_v, scaler_v = build_classification_fold(
    2023, feature_cols, 24, 24, batch_size=128, seed=SEED)
Xtr_v, ytr_cls_v, Xv_v, yv_cls_v, Xte_v, yte_cls_v, ytr_orig_v, yv_orig_v, yte_orig_v = raw_v

# The last timestep in the window (scaled) vs the target (original scale)
last_in_window_scaled = Xte_v[0, -1, 0]
target_orig = yte_orig_v[0]
target_class = yte_cls_v[0]

# Reconstruct: what would the target be if we used the last window value instead?
last_in_window_orig = inverse_transform_target(scaler_v, feature_cols, np.array([last_in_window_scaled]))
wrong_class = swc_to_class(last_in_window_orig)[0]

print("=== Label Alignment Verification ===")
print(f"Last window timestep (original scale): {last_in_window_orig[0]:.4f} → class {wrong_class}")
print(f"Actual target (H=24 ahead, original scale): {target_orig:.4f} → class {target_class}")
print(f"Are they different? {last_in_window_orig[0]:.4f} vs {target_orig:.4f}")

# Verify across many samples that the target is NOT just the last window value
n_check = min(1000, len(yte_orig_v))
last_vals = inverse_transform_target(scaler_v, feature_cols, Xte_v[:n_check, -1, 0])
targets = yte_orig_v[:n_check]
pct_same = np.mean(np.abs(last_vals - targets) < 1e-6) * 100
print(f"% of test samples where target == last window value: {pct_same:.1f}%")
print(f"  (Should be ~0% for H=24 — they're 24h apart)")
print(f"Mean absolute difference: {np.mean(np.abs(last_vals - targets)):.4f}")
print("✓ Labels are from the future target, not the window end.\n")

# --- Train/val/test class distributions ---
print("=== Class Distributions by Split ===")
for name, arr in [("Train", ytr_cls_v), ("Val", yv_cls_v), ("Test", yte_cls_v)]:
    counts = np.bincount(arr, minlength=N_CLASSES)
    pcts = counts / len(arr) * 100
    active = [f"c{i}:{pcts[i]:.1f}%" for i in range(N_CLASSES) if counts[i] > 0]
    print(f"  {name}: n={len(arr):,}  {', '.join(active)}")

# --- Priority 4: Train ONE small classifier ---
print("\n=== Training ONE ClassificationCNN (SWC_20 only, H=24, test=2023) ===")
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

model_test = ClassificationCNN(num_features=1)
t0 = time.time()
model_test, trh, vlh, stopped = train_classifier(
    model_test, trl_v, vl_v, epochs=40, patience=8)
elapsed = time.time() - t0
print(f"Training done in {elapsed:.1f}s, stopped at epoch {stopped}")
print(f"Best val loss: {min(vlh):.6f}, final val loss: {vlh[-1]:.6f}")

# Evaluate
preds_cls, true_cls, probs = predict_classes(model_test, tel_v)
metrics = compute_classification_metrics(yte_cls_v, preds_cls)

print(f"\n=== Test Results ===")
print(f"Accuracy:        {metrics['accuracy']:.4f}")
print(f"Macro Precision: {metrics['macro_precision']:.4f}")
print(f"Macro Recall:    {metrics['macro_recall']:.4f}")
print(f"Macro F1:        {metrics['macro_f1']:.4f}")
print(f"Weighted F1:     {metrics['weighted_f1']:.4f}")
print(f"Within-1 Acc:    {metrics['within_one_acc']:.4f}")
print(f"Mean Abs Class Error: {metrics['mean_abs_class_error']:.4f}")

# Within-2 accuracy (supplementary)
diff = np.abs(yte_cls_v - preds_cls)
within_two = (diff <= 2).mean()
print(f"Within-2 Acc:    {within_two:.4f}")

# Per-class report
from sklearn.metrics import classification_report
active_classes = [i for i in range(N_CLASSES) if (yte_cls_v == i).sum() > 0 or (preds_cls == i).sum() > 0]
report = classification_report(yte_cls_v, preds_cls, labels=active_classes,
                               target_names=[f"C{i}" for i in active_classes], zero_division=0)
print(f"\nPer-class report:\n{report}")

# Baseline comparison
pers_scaled = persistence_predict(Xte_v, 0)
pers_orig = inverse_transform_target(scaler_v, feature_cols, pers_scaled)
pers_cls = swc_to_class(pers_orig)
pers_m = compute_classification_metrics(yte_cls_v, pers_cls)
print(f"Persistence baseline: Acc={pers_m['accuracy']:.4f}, MacF1={pers_m['macro_f1']:.4f}, W1={pers_m['within_one_acc']:.4f}")

maj_cls = majority_class_predict(ytr_cls_v, len(yte_cls_v))
maj_m = compute_classification_metrics(yte_cls_v, maj_cls)
print(f"Majority class:       Acc={maj_m['accuracy']:.4f}, MacF1={maj_m['macro_f1']:.4f}, W1={maj_m['within_one_acc']:.4f}")
print(f"\n✓ Single classifier trained, evaluated, and verified end-to-end.")


### Architecture Comparison

SmallCNN (1 layer, 8 filters), LargerCNN (2 layers, 16→32 filters, avg pool), MaxpoolCNN (2 layers, 16→32 filters, max pool). 3 architectures × 2 horizons × 2 feature sets × 3 test years = 36 configurations + baselines.

In [ ]:

# ============================================================
# Priority 5-6: Full classification experiments
# Compare SmallCNN, LargerCNN (avg), MaxpoolCNN across horizons
# Regular vs weighted cross-entropy
# ============================================================

# Define classification architecture variants
def SmallClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=1, filters=[8], pool_type='avg')

def LargerClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=2, filters=[16, 32], pool_type='avg')

def MaxpoolClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=2, filters=[16, 32], pool_type='max')

# Print param counts
for name, fn in [("SmallClassCNN", SmallClassCNN), ("LargerClassCNN", LargerClassCNN),
                  ("MaxpoolClassCNN", MaxpoolClassCNN)]:
    m = fn(1)
    p = sum(pp.numel() for pp in m.parameters() if pp.requires_grad)
    print(f"  {name} (F=1): {p} params")

CLASS_MODELS = {
    "SmallCNN": SmallClassCNN,
    "LargerCNN": LargerClassCNN,
    "MaxpoolCNN": MaxpoolClassCNN,
}

# Experiment config
CLASSIFICATION_CSV = "classification_results.csv"
CLASS_TEST_YEARS = [2023, 2020, 2018]
CLASS_HORIZONS = [24, 168]  # start with these two, expand later if time
CLASS_FEATURE_SETS = {
    "SWC_20 only":       ["SWC_20"],
    "SWC_20 + Tair":     ["SWC_20", "Tair"],
}

cls_rows = []
t_total = time.time()

for test_year in CLASS_TEST_YEARS:
    for horizon in CLASS_HORIZONS:
        for feat_name, feature_cols in CLASS_FEATURE_SETS.items():
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            trl, vl, tel, raw, scaler = build_classification_fold(
                test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig = raw

            # --- Baselines ---
            # Persistence
            pers_scaled = persistence_predict(Xte, feature_cols.index("SWC_20"))
            pers_orig = inverse_transform_target(scaler, feature_cols, pers_scaled)
            pers_cls = swc_to_class(pers_orig)
            pers_m = compute_classification_metrics(yte_cls, pers_cls)
            diff_pers = np.abs(yte_cls - pers_cls)
            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "persistence", "loss_type": "N/A", "epochs_run": 0,
                "within_two_acc": (diff_pers <= 2).mean(),
                **{k: v for k, v in pers_m.items() if k != "confusion_matrix"}
            })

            # Majority class
            maj_cls = majority_class_predict(ytr_cls, len(yte_cls))
            maj_m = compute_classification_metrics(yte_cls, maj_cls)
            diff_maj = np.abs(yte_cls - maj_cls)
            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "majority_class", "loss_type": "N/A", "epochs_run": 0,
                "within_two_acc": (diff_maj <= 2).mean(),
                **{k: v for k, v in maj_m.items() if k != "confusion_matrix"}
            })

            # --- Classification CNNs (regular cross-entropy) ---
            for model_name, model_fn in CLASS_MODELS.items():
                random.seed(SEED); np.random.seed(SEED)
                torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

                t0 = time.time()
                model = model_fn(num_features=len(feature_cols))
                model, trh, vlh, stopped = train_classifier(
                    model, trl, vl, epochs=40, patience=8)
                preds_cls, _, probs = predict_classes(model, tel)
                cnn_m = compute_classification_metrics(yte_cls, preds_cls)
                diff_cnn = np.abs(yte_cls - preds_cls)

                cls_rows.append({
                    "test_year": test_year, "horizon": horizon, "features": feat_name,
                    "model": model_name, "loss_type": "regular_CE",
                    "epochs_run": stopped, "best_val_loss": min(vlh),
                    "final_val_loss": vlh[-1], "train_seconds": round(time.time()-t0, 1),
                    "within_two_acc": (diff_cnn <= 2).mean(),
                    **{k: v for k, v in cnn_m.items() if k != "confusion_matrix"}
                })

                print(f"  Y={test_year} H={horizon:3d} {feat_name:16s} {model_name:12s} CE: "
                      f"acc={cnn_m['accuracy']:.3f} F1={cnn_m['macro_f1']:.3f} "
                      f"W1={cnn_m['within_one_acc']:.3f} ({time.time()-t0:.0f}s)")

            # Save incrementally
            pd.DataFrame(cls_rows).to_csv(CLASSIFICATION_CSV, index=False)

    print(f"  --- Year {test_year} done ({time.time()-t_total:.0f}s) ---\n")

pd.DataFrame(cls_rows).to_csv(CLASSIFICATION_CSV, index=False)
print(f"\nClassification experiments: {len(cls_rows)} rows in {time.time()-t_total:.0f}s")
print(f"Saved to {CLASSIFICATION_CSV}")


### Results & Visualization

Extract LaTeX table numbers, generate accuracy/F1 bar charts, confusion matrices, and class distribution plots.

In [ ]:

# ============================================================
# Extract LaTeX table numbers from classification_results.csv
# ============================================================
cls_df = pd.read_csv(CLASSIFICATION_CSV)
print(f"Loaded {len(cls_df)} rows")

# Architecture comparison table (SWC_20 only, regular CE, avg over 3 years)
print("\n=== LaTeX Table: Architecture Comparison (SWC_20 only) ===")
print("% Copy these into the \\begin{tabular} in main.tex")
for h in [24, 168]:
    for model in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") & (cls_df["model"]==model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"].isin(["regular_CE", "N/A"])]
        if sub.empty:
            continue
        acc = sub["accuracy"].mean()
        mf1 = sub["macro_f1"].mean()
        wf1 = sub["weighted_f1"].mean()
        w1 = sub["within_one_acc"].mean()
        mace = sub["mean_abs_class_error"].mean()
        prefix = str(h) if model == "SmallCNN" else "   "
        display_name = {"SmallCNN": "SmallClassCNN", "LargerCNN": "LargerClassCNN",
                       "MaxpoolCNN": "MaxpoolClassCNN", "persistence": "Persistence",
                       "majority_class": "Majority class"}[model]
        print(f"{prefix:3s} & {display_name:18s} & {acc:.3f} & {mf1:.3f} & {wf1:.3f} & {w1:.3f} & {mace:.3f} \\\\")
    if h == 24:
        print("\\midrule")

# Feature set effect
print("\n=== Feature Set Effect (MaxpoolCNN, regular CE) ===")
for h in [24, 168]:
    for feat in ["SWC_20 only", "SWC_20 + Tair"]:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]==feat) &
                     (cls_df["model"]=="MaxpoolCNN") & (cls_df["loss_type"]=="regular_CE")]
        if sub.empty:
            continue
        print(f"  H={h}, {feat}: acc={sub['accuracy'].mean():.3f}, "
              f"F1={sub['macro_f1'].mean():.3f}, W1={sub['within_one_acc'].mean():.3f}")

# Persistence baseline details
print("\n=== Persistence Baseline (SWC_20 only, avg over 3 years) ===")
for h in [24, 168]:
    sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") & (cls_df["model"]=="persistence")]
    if not sub.empty:
        print(f"  H={h}: acc={sub['accuracy'].mean():.3f}, F1={sub['macro_f1'].mean():.3f}, W1={sub['within_one_acc'].mean():.3f}")

# Per test year for discussion
print("\n=== Per Test Year Results (MaxpoolCNN, SWC_20 only, regular CE) ===")
for h in [24, 168]:
    for ty in [2023, 2020, 2018]:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["test_year"]==ty) &
                     (cls_df["features"]=="SWC_20 only") & (cls_df["model"]=="MaxpoolCNN") &
                     (cls_df["loss_type"]=="regular_CE")]
        if not sub.empty:
            print(f"  H={h} Y={ty}: acc={sub['accuracy'].values[0]:.3f}, "
                  f"F1={sub['macro_f1'].values[0]:.3f}, W1={sub['within_one_acc'].values[0]:.3f}")


In [ ]:

# ============================================================
# Classification Results Analysis & Figures
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 150})

cls_df = pd.read_csv(CLASSIFICATION_CSV)
print(f"Loaded {len(cls_df)} rows from {CLASSIFICATION_CSV}")
print(f"Models: {cls_df['model'].unique()}")
print(f"Loss types: {cls_df['loss_type'].unique()}")

# ---- Table 1: Architecture comparison (SWC_20 only, regular CE, avg over 3 years) ----
print("\n" + "="*110)
print("ARCHITECTURE COMPARISON — SWC_20 only, regular CE, averaged over 3 test years")
print("="*110)
for h in [24, 168]:
    print(f"\n--- H={h} ---")
    for model in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") & (cls_df["model"]==model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"].isin(["regular_CE", "N/A"])]
        if sub.empty:
            continue
        cols = ["accuracy", "macro_f1", "weighted_f1", "within_one_acc", "within_two_acc", "mean_abs_class_error"]
        avgs = {c: sub[c].mean() for c in cols if c in sub.columns}
        line = f"  {model:16s}: " + " | ".join(f"{k}={v:.3f}" for k, v in avgs.items())
        print(line)

# ---- Table 2: Feature set effect ----
print("\n" + "="*110)
print("FEATURE SET EFFECT — MaxpoolCNN, regular CE, averaged over 3 test years")
print("="*110)
for h in [24, 168]:
    print(f"\n--- H={h} ---")
    for feat in ["SWC_20 only", "SWC_20 + Tair"]:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]==feat) &
                     (cls_df["model"]=="MaxpoolCNN") & (cls_df["loss_type"]=="regular_CE")]
        if sub.empty:
            continue
        avgs = {c: sub[c].mean() for c in ["accuracy", "macro_f1", "within_one_acc"] if c in sub.columns}
        print(f"  {feat:20s}: " + " | ".join(f"{k}={v:.3f}" for k, v in avgs.items()))

# ---- Figure 1: Accuracy by model and horizon (bar chart) ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
models_order = ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence"]
colors = {"SmallCNN": "#4C72B0", "LargerCNN": "#55A868", "MaxpoolCNN": "#C44E52", "persistence": "#8C8C8C"}
bar_width = 0.18

for ax_i, h in enumerate([24, 168]):
    ax = axes[ax_i]
    x = np.arange(3)  # 3 test years
    for mi, model in enumerate(models_order):
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") &
                     (cls_df["model"]==model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"].isin(["regular_CE", "N/A"])]
        sub = sub.sort_values("test_year")
        if sub.empty:
            continue
        ax.bar(x + mi*bar_width, sub["accuracy"].values, bar_width,
               label=model, color=colors[model], edgecolor='white', linewidth=0.5)
    ax.set_xticks(x + 1.5*bar_width)
    ax.set_xticklabels(["2023", "2020", "2018"])
    ax.set_xlabel("Test Year")
    ax.set_title(f"H={h}")
    ax.set_ylim(0.4, 1.0)
    if ax_i == 0:
        ax.set_ylabel("Accuracy")
        ax.legend(fontsize=9)

fig.suptitle("Classification Accuracy by Architecture and Horizon (SWC_20 only)", fontsize=13)
plt.tight_layout()
plt.savefig("cls_accuracy_by_arch.png", dpi=200, bbox_inches='tight')
plt.savefig("cls_accuracy_by_arch.pdf", bbox_inches='tight')
plt.show()
print("Saved cls_accuracy_by_arch.png/pdf")

# ---- Figure 2: Macro F1 comparison ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax_i, h in enumerate([24, 168]):
    ax = axes[ax_i]
    x = np.arange(3)
    for mi, model in enumerate(models_order):
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") &
                     (cls_df["model"]==model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"].isin(["regular_CE", "N/A"])]
        sub = sub.sort_values("test_year")
        if sub.empty:
            continue
        ax.bar(x + mi*bar_width, sub["macro_f1"].values, bar_width,
               label=model, color=colors[model], edgecolor='white', linewidth=0.5)
    ax.set_xticks(x + 1.5*bar_width)
    ax.set_xticklabels(["2023", "2020", "2018"])
    ax.set_xlabel("Test Year")
    ax.set_title(f"H={h}")
    ax.set_ylim(0.2, 0.85)
    if ax_i == 0:
        ax.set_ylabel("Macro F1")
        ax.legend(fontsize=9)

fig.suptitle("Classification Macro F1 by Architecture and Horizon (SWC_20 only)", fontsize=13)
plt.tight_layout()
plt.savefig("cls_macro_f1_by_arch.png", dpi=200, bbox_inches='tight')
plt.savefig("cls_macro_f1_by_arch.pdf", bbox_inches='tight')
plt.show()
print("Saved cls_macro_f1_by_arch.png/pdf")

# ---- Figure 3: Within-1 accuracy (shows the distance-aware metric) ----
fig, ax = plt.subplots(figsize=(8, 5))
metrics_by_h = {}
for h in [24, 168]:
    for model in models_order:
        sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") &
                     (cls_df["model"]==model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"].isin(["regular_CE", "N/A"])]
        if sub.empty:
            continue
        metrics_by_h[(h, model)] = sub["within_one_acc"].mean()

x = np.arange(len(models_order))
w = 0.35
h24_vals = [metrics_by_h.get((24, m), 0) for m in models_order]
h168_vals = [metrics_by_h.get((168, m), 0) for m in models_order]
ax.bar(x - w/2, h24_vals, w, label="H=24", color="#4C72B0")
ax.bar(x + w/2, h168_vals, w, label="H=168", color="#C44E52")
ax.set_xticks(x)
ax.set_xticklabels(models_order)
ax.set_ylabel("Within-1-Category Accuracy")
ax.set_ylim(0.85, 1.01)
ax.legend()
ax.set_title("Within-1-Category Accuracy (SWC_20 only, avg over 3 years)")
plt.tight_layout()
plt.savefig("cls_within1_acc.png", dpi=200, bbox_inches='tight')
plt.savefig("cls_within1_acc.pdf", bbox_inches='tight')
plt.show()
print("Saved cls_within1_acc.png/pdf")


In [ ]:

# ============================================================
# Confusion Matrices for best config (MaxpoolCNN, SWC_20 only)
# ============================================================
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_i, (horizon, test_year) in enumerate([(24, 2023), (168, 2023)]):
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    feature_cols = ["SWC_20"]
    trl, vl, tel, raw, scaler = build_classification_fold(
        test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
    Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig = raw

    model = MaxpoolClassCNN(num_features=len(feature_cols))
    model, trh, vlh, stopped = train_classifier(model, trl, vl, epochs=40, patience=8)
    preds_cls, _, probs = predict_classes(model, tel)

    cm = sk_confusion_matrix(yte_cls, preds_cls, labels=list(range(N_CLASSES)))
    # Normalize by row (true class)
    cm_norm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm_norm / row_sums

    ax = axes[ax_i]
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='equal')

    # Active classes only for labels
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            val = cm_norm[i, j]
            if val > 0.005:
                color = 'white' if val > 0.5 else 'black'
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                       fontsize=7, color=color)

    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(range(N_CLASSES), fontsize=8)
    ax.set_yticklabels(range(N_CLASSES), fontsize=8)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_title(f"MaxpoolClassCNN, H={horizon}, test={test_year}")

fig.colorbar(im, ax=axes, shrink=0.8, label="Row-normalized frequency")
fig.suptitle("Confusion Matrices (SWC_20 only, normalized by true class)", fontsize=13)
plt.tight_layout()
plt.savefig("cls_confusion_matrices.png", dpi=200, bbox_inches='tight')
plt.savefig("cls_confusion_matrices.pdf", bbox_inches='tight')
plt.show()
print("Saved cls_confusion_matrices.png/pdf")

# Print classification report for both horizons
from sklearn.metrics import classification_report
for horizon in [24, 168]:
    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    trl, vl, tel, raw, scaler = build_classification_fold(
        2023, ["SWC_20"], 24, horizon, batch_size=128, seed=SEED)
    _, _, _, _, _, yte_cls, _, _, _ = raw
    model = MaxpoolClassCNN(num_features=1)
    model, _, _, _ = train_classifier(model, trl, vl, epochs=40, patience=8)
    preds_cls, _, _ = predict_classes(model, tel)
    print(f"\n--- Classification Report: MaxpoolCNN, H={horizon}, test=2023 ---")
    print(classification_report(yte_cls, preds_cls, zero_division=0))


### Weighted Cross-Entropy

Investigates class-weighted CE vs. regular CE. Weights are computed from training-set class frequencies only.

In [ ]:

# ============================================================
# Priority 8: Weighted cross-entropy investigation
# Compare regular CE vs class-weighted CE
# Weights computed ONLY from training data
# ============================================================

# Load results from regular CE experiments
cls_df = pd.read_csv(CLASSIFICATION_CSV)
print(f"Regular CE results: {len(cls_df)} rows")

# --- Weighted CE experiments ---
weighted_rows = []
t_total = time.time()

for test_year in CLASS_TEST_YEARS:
    for horizon in CLASS_HORIZONS:
        for feat_name, feature_cols in CLASS_FEATURE_SETS.items():
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            trl, vl, tel, raw, scaler = build_classification_fold(
                test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig = raw

            # Compute class weights from TRAINING data only
            train_counts = np.bincount(ytr_cls, minlength=N_CLASSES).astype(np.float64)
            # Inverse frequency weighting: w_c = N / (K * n_c), capped for empty classes
            total_train = len(ytr_cls)
            n_active = (train_counts > 0).sum()
            weights = np.zeros(N_CLASSES, dtype=np.float64)
            for c in range(N_CLASSES):
                if train_counts[c] > 0:
                    weights[c] = total_train / (n_active * train_counts[c])
                else:
                    weights[c] = 0.0  # no weight for empty classes

            # Normalize so mean of active weights = 1
            active_mask = weights > 0
            if active_mask.sum() > 0:
                weights[active_mask] = weights[active_mask] / weights[active_mask].mean()

            if test_year == CLASS_TEST_YEARS[0] and horizon == CLASS_HORIZONS[0]:
                print(f"\nClass weights for Y={test_year}, {feat_name}:")
                for c in range(N_CLASSES):
                    print(f"  Class {c}: count={int(train_counts[c]):6d}, weight={weights[c]:.4f}")

            # Train with LargerCNN (best architecture from regular CE, for fair comparison)
            for model_name, model_fn in CLASS_MODELS.items():
                random.seed(SEED); np.random.seed(SEED)
                torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

                t0 = time.time()
                model = model_fn(num_features=len(feature_cols))
                model, trh, vlh, stopped = train_classifier(
                    model, trl, vl, epochs=40, patience=8,
                    class_weights=weights.tolist())
                preds_cls, _, probs = predict_classes(model, tel)
                m = compute_classification_metrics(yte_cls, preds_cls)
                diff = np.abs(yte_cls - preds_cls)

                weighted_rows.append({
                    "test_year": test_year, "horizon": horizon, "features": feat_name,
                    "model": model_name, "loss_type": "weighted_CE",
                    "epochs_run": stopped, "best_val_loss": min(vlh),
                    "final_val_loss": vlh[-1], "train_seconds": round(time.time()-t0, 1),
                    "within_two_acc": (diff <= 2).mean(),
                    **{k: v for k, v in m.items() if k != "confusion_matrix"}
                })

                print(f"  Y={test_year} H={horizon:3d} {feat_name:16s} {model_name:12s} wCE: "
                      f"acc={m['accuracy']:.3f} F1={m['macro_f1']:.3f} "
                      f"W1={m['within_one_acc']:.3f} ({time.time()-t0:.0f}s)")

            pd.DataFrame(weighted_rows).to_csv("classification_weighted_ce.csv", index=False)

    print(f"  --- Year {test_year} done ({time.time()-t_total:.0f}s) ---\n")

# Merge weighted results into main CSV
weighted_df = pd.DataFrame(weighted_rows)
weighted_df.to_csv("classification_weighted_ce.csv", index=False)
all_cls = pd.concat([cls_df, weighted_df], ignore_index=True)
all_cls.to_csv(CLASSIFICATION_CSV, index=False)
print(f"\nWeighted CE experiments: {len(weighted_rows)} rows in {time.time()-t_total:.0f}s")
print(f"Total classification results: {len(all_cls)} rows saved to {CLASSIFICATION_CSV}")

# --- Comparison: regular vs weighted CE ---
print("\n" + "="*100)
print("REGULAR CE vs WEIGHTED CE COMPARISON (averaged over 3 test years)")
print("="*100)
for h in CLASS_HORIZONS:
    print(f"\n--- H={h} ---")
    for fn in CLASS_FEATURE_SETS:
        for mn in CLASS_MODELS:
            reg = all_cls[(all_cls["horizon"]==h) & (all_cls["features"]==fn) &
                         (all_cls["model"]==mn) & (all_cls["loss_type"]=="regular_CE")]
            wce = all_cls[(all_cls["horizon"]==h) & (all_cls["features"]==fn) &
                         (all_cls["model"]==mn) & (all_cls["loss_type"]=="weighted_CE")]
            if reg.empty or wce.empty:
                continue
            print(f"  {fn:16s} {mn:12s}: "
                  f"Reg acc={reg['accuracy'].mean():.3f} F1={reg['macro_f1'].mean():.3f} W1={reg['within_one_acc'].mean():.3f} | "
                  f"Wtd acc={wce['accuracy'].mean():.3f} F1={wce['macro_f1'].mean():.3f} W1={wce['within_one_acc'].mean():.3f}")


In [ ]:

# Quick extraction of ALL table values including baselines
import pandas as pd
df = pd.read_csv("classification_results.csv")

# Filter SWC_20 only
swc = df[df["features"] == "SWC_20 only"]

for h in [24, 168]:
    print(f"\n=== H={h} ===")
    for model in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = swc[(swc["horizon"] == h) & (swc["model"] == model)]
        if len(sub) == 0:
            continue
        acc = sub["accuracy"].mean()
        mf1 = sub["macro_f1"].mean()
        wf1 = sub["weighted_f1"].mean()
        w1 = sub["within_one_acc"].mean()
        mace = sub["mean_abs_class_error"].mean()
        print(f"  {model:20s}: acc={acc:.3f} mF1={mf1:.3f} wF1={wf1:.3f} W1={w1:.3f} MACE={mace:.3f}")


In [ ]:

# Download figures as base64 for local transfer
import base64, os, json

figure_files = [
    "cls_accuracy_by_arch.pdf",
    "cls_macro_f1_by_arch.pdf",
    "cls_within1_acc.pdf",
    "cls_confusion_matrices.pdf",
    "class_distribution_fd11.pdf",
    "class_distribution_comparison.pdf",
    "class_distribution_all_stations.pdf",
]

results = {}
for fname in figure_files:
    if os.path.exists(fname):
        with open(fname, "rb") as f:
            results[fname] = base64.b64encode(f.read()).decode("ascii")
        print(f"  Encoded {fname}: {len(results[fname])} chars")
    else:
        print(f"  MISSING: {fname}")

# Also extract ALL baseline values from the CSV
import pandas as pd
df = pd.read_csv("classification_results.csv")
swc = df[df["features"] == "SWC_20 only"]

print("\n=== BASELINE EXTRACTION (SWC_20 only, regular CE) ===")
for h in [24, 168]:
    print(f"\n--- H={h} ---")
    for model in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = swc[(swc["horizon"] == h) & (swc["model"] == model)]
        if "loss_type" in sub.columns:
            sub = sub[sub["loss_type"] == "regular_CE"] if model not in ["persistence", "majority_class"] else sub
        if len(sub) == 0:
            print(f"  {model:20s}: NO DATA")
            continue
        cols = ["accuracy", "macro_f1", "weighted_f1", "within_one_acc", "mean_abs_class_error"]
        vals = {c: sub[c].mean() if c in sub.columns else "N/A" for c in cols}
        print(f"  {model:20s}: acc={vals['accuracy']:.3f} mF1={vals['macro_f1']:.3f} "
              f"wF1={vals['weighted_f1']:.3f} W1={vals['within_one_acc']:.3f} "
              f"MACE={vals['mean_abs_class_error']:.3f}")

# Print base64 data as JSON for extraction
print("\n=== BASE64_FIGURES_JSON_START ===")
print(json.dumps(results))
print("=== BASE64_FIGURES_JSON_END ===")


In [ ]:

# Quick runtime audit: check multi-station + classification infrastructure
checks = {}
for name in ["STATION_YEAR_DATA", "SELECTED_STATIONS", "get_fold_keys",
             "build_multi_fold", "swc_to_class", "ClassificationCNN",
             "train_classifier", "predict_classes", "compute_classification_metrics",
             "SmallClassCNN", "LargerClassCNN", "MaxpoolClassCNN",
             "CLASS_MODELS", "N_CLASSES", "inverse_transform_target",
             "make_windows_for_year", "persistence_predict", "DEVICE"]:
    checks[name] = name in dir()

print("=== Runtime Variable Check ===")
for k, v in checks.items():
    status = "OK" if v else "MISSING"
    print(f"  {k:40s}: {status}")

missing = [k for k, v in checks.items() if not v]
if missing:
    print(f"\n*** MISSING: {missing} — need to re-run setup cells ***")
else:
    print(f"\n✓ All infrastructure loaded")
    print(f"  Stations: {SELECTED_STATIONS}")
    print(f"  Station-year keys: {len(STATION_YEAR_DATA)}")
    print(f"  DEVICE: {DEVICE}")
    print(f"  N_CLASSES: {N_CLASSES}")
    print(f"  CLASS_MODELS: {list(CLASS_MODELS.keys())}")


## Multi-Station Classification

Same four-station setup as multi-station regression. Two experiments:
- **Exp A (Hold-Out Year):** Train on all 4 stations, hold out one year for testing.
- **Exp B (Hold-Out Station):** Train on 3 stations, hold out one station entirely.

In [ ]:

# ============================================================
# Multi-Station Classification Experiments
# Exp A: Hold-out year (train on all 4 stations, test on held-out year)
# Exp B: Hold-out station (train on 3 stations, test on held-out station)
# ============================================================
import time, random, os

# --- 1. Classification model factories (same as single-station) ---
def SmallClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=1, filters=[8], pool_type='avg')

def LargerClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=2, filters=[16, 32], pool_type='avg')

def MaxpoolClassCNN(num_features, n_classes=12):
    return ClassificationCNN(num_features, n_classes=n_classes,
                             n_layers=2, filters=[16, 32], pool_type='max')

CLASS_MODELS = {"SmallCNN": SmallClassCNN, "LargerCNN": LargerClassCNN, "MaxpoolCNN": MaxpoolClassCNN}
N_CLASSES = 12
MULTI_CLS_CSV = "multi_station_classification.csv"

# --- 2. Multi-station classification fold builder ---
def build_multi_classification_fold(train_keys, val_keys, test_keys, feature_cols,
                                     window_size, horizon, batch_size, seed=SEED):
    scaler = MinMaxScaler()
    train_dfs = [STATION_YEAR_DATA[k][feature_cols] for k in train_keys if len(STATION_YEAR_DATA[k]) > 0]
    scaler.fit(pd.concat(train_dfs, axis=0))

    def collect_windows(keys):
        Xs, ys_scaled = [], []
        for (station, year) in keys:
            g = STATION_YEAR_DATA[(station, year)].copy()
            g[feature_cols] = scaler.transform(g[feature_cols])
            X, y = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X):
                Xs.append(X); ys_scaled.append(y)
        if not Xs:
            return (np.empty((0, window_size, len(feature_cols)), dtype=np.float32),
                    np.empty((0,), dtype=np.float32))
        return np.concatenate(Xs), np.concatenate(ys_scaled)

    Xtr, ytr_s = collect_windows(train_keys)
    Xv, yv_s = collect_windows(val_keys)
    Xte, yte_s = collect_windows(test_keys)

    ytr_orig = inverse_transform_target(scaler, feature_cols, ytr_s)
    yv_orig = inverse_transform_target(scaler, feature_cols, yv_s)
    yte_orig = inverse_transform_target(scaler, feature_cols, yte_s)

    ytr_cls = swc_to_class(ytr_orig)
    yv_cls = swc_to_class(yv_orig)
    yte_cls = swc_to_class(yte_orig)

    g = torch.Generator().manual_seed(seed)
    def cls_loader(X, y_cls, shuffle):
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y_cls, dtype=torch.long))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          generator=g if shuffle else None)

    trl = cls_loader(Xtr, ytr_cls, True)
    vl = cls_loader(Xv, yv_cls, False)
    tel = cls_loader(Xte, yte_cls, False)

    return (trl, vl, tel,
            (Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig),
            scaler)

# --- 3. Class distribution audit across all stations ---
print("=" * 80)
print("CLASS DISTRIBUTION AUDIT (all stations, all years)")
print("=" * 80)
for station in SELECTED_STATIONS:
    all_swc = []
    for year in ALL_YEARS:
        g = STATION_YEAR_DATA[(station, year)]
        all_swc.extend(g["SWC_20"].values)
    all_swc = np.array(all_swc)
    classes = swc_to_class(all_swc)
    counts = np.bincount(classes, minlength=N_CLASSES)
    active = (counts > 0).sum()
    top3 = sorted(range(N_CLASSES), key=lambda c: counts[c], reverse=True)[:3]
    print(f"\n  {station}: {len(all_swc):,} samples, {active} active classes, "
          f"range=[{all_swc.min():.3f}, {all_swc.max():.3f}]")
    for c in range(N_CLASSES):
        if counts[c] > 0:
            print(f"    Class {c:2d} [{c*0.05:.2f}-{(c+1)*0.05:.2f}): {counts[c]:6d} ({counts[c]/len(all_swc)*100:5.1f}%)")

# --- 4. Verification: data leakage check ---
print("\n" + "=" * 80)
print("DATA LEAKAGE VERIFICATION")
print("=" * 80)
for exp_type, test_val in [("hold_out_year", 2023), ("hold_out_station", "CB06")]:
    tr_k, va_k, te_k = get_fold_keys(exp_type, test_val)
    tr_set, va_set, te_set = set(tr_k), set(va_k), set(te_k)
    overlap_tv = tr_set & va_set
    overlap_te = tr_set & te_set
    overlap_ve = va_set & te_set
    print(f"  {exp_type} ({test_val}): train={len(tr_k)}, val={len(va_k)}, test={len(te_k)}")
    print(f"    train∩val={len(overlap_tv)}, train∩test={len(overlap_te)}, val∩test={len(overlap_ve)}")
    assert len(overlap_tv) == 0 and len(overlap_te) == 0 and len(overlap_ve) == 0, "LEAKAGE DETECTED!"

# Quick fold + label check
feature_cols_test = ["SWC_20"]
tr_k, va_k, te_k = get_fold_keys("hold_out_year", 2023)
trl_t, vl_t, tel_t, raw_t, sc_t = build_multi_classification_fold(
    tr_k, va_k, te_k, feature_cols_test, 24, 24, 128)
Xtr_t, ytr_cls_t, Xv_t, yv_cls_t, Xte_t, yte_cls_t, ytr_o, yv_o, yte_o = raw_t

# Verify target is from the future, not the window
last_in_window = inverse_transform_target(sc_t, feature_cols_test, Xte_t[:100, -1, 0])
n_same = np.sum(np.abs(last_in_window - yte_o[:100]) < 1e-6)
print(f"\n  Label alignment check (hold-out-year 2023, H=24):")
print(f"    Test samples where target == last window value: {n_same}/100 (should be ~0)")
print(f"    Train: {len(ytr_cls_t):,} samples, Val: {len(yv_cls_t):,}, Test: {len(yte_cls_t):,}")

# Class distribution in train vs test
tr_counts = np.bincount(ytr_cls_t, minlength=N_CLASSES)
te_counts = np.bincount(yte_cls_t, minlength=N_CLASSES)
print(f"    Train active classes: {(tr_counts > 0).sum()}, Test active classes: {(te_counts > 0).sum()}")
print("\n✓ All verification checks passed")

# --- 5. EXPERIMENTS ---
print("\n" + "=" * 80)
print("MULTI-STATION CLASSIFICATION EXPERIMENTS")
print("=" * 80)

EXP_HORIZONS = [24, 168]
EXP_TEST_YEARS = [2023, 2020, 2018]
EXP_FEATURE_SETS = {"SWC_20 only": ["SWC_20"], "SWC_20 + Tair": ["SWC_20", "Tair"]}
rows = []
t_total = time.time()
run_count = 0

# --- Experiment A: Hold-out year ---
print("\n--- EXPERIMENT A: Hold-Out Year ---")
for test_year in EXP_TEST_YEARS:
    for horizon in EXP_HORIZONS:
        for feat_name, feature_cols in EXP_FEATURE_SETS.items():
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            tr_k, va_k, te_k = get_fold_keys("hold_out_year", test_year)
            trl, vl, tel, raw, scaler = build_multi_classification_fold(
                tr_k, va_k, te_k, feature_cols, 24, horizon, 128)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_o, yv_o, yte_o = raw

            base_row = {"experiment": "hold_out_year", "test_value": test_year,
                       "horizon": horizon, "features": feat_name,
                       "n_train": len(ytr_cls), "n_val": len(yv_cls), "n_test": len(yte_cls)}

            # Persistence baseline
            pers_s = persistence_predict(Xte, feature_cols.index("SWC_20"))
            pers_o = inverse_transform_target(scaler, feature_cols, pers_s)
            pers_cls = swc_to_class(pers_o)
            pers_m = compute_classification_metrics(yte_cls, pers_cls)
            diff_p = np.abs(yte_cls - pers_cls)
            rows.append({**base_row, "model": "persistence", "loss_type": "N/A",
                        "epochs_run": 0, "within_two_acc": (diff_p <= 2).mean(),
                        **{k: v for k, v in pers_m.items() if k != "confusion_matrix"}})

            # Majority class baseline
            maj_cls = majority_class_predict(ytr_cls, len(yte_cls))
            maj_m = compute_classification_metrics(yte_cls, maj_cls)
            diff_j = np.abs(yte_cls - maj_cls)
            rows.append({**base_row, "model": "majority_class", "loss_type": "N/A",
                        "epochs_run": 0, "within_two_acc": (diff_j <= 2).mean(),
                        **{k: v for k, v in maj_m.items() if k != "confusion_matrix"}})

            # CNN models
            for model_name, model_fn in CLASS_MODELS.items():
                random.seed(SEED); np.random.seed(SEED)
                torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

                t0 = time.time()
                model = model_fn(num_features=len(feature_cols))
                model, trh, vlh, stopped = train_classifier(model, trl, vl, epochs=40, patience=8)
                preds_cls, _, probs = predict_classes(model, tel)
                m = compute_classification_metrics(yte_cls, preds_cls)
                diff = np.abs(yte_cls - preds_cls)
                elapsed = time.time() - t0
                run_count += 1

                rows.append({**base_row, "model": model_name, "loss_type": "regular_CE",
                            "epochs_run": stopped, "best_val_loss": min(vlh),
                            "train_seconds": round(elapsed, 1),
                            "within_two_acc": (diff <= 2).mean(),
                            **{k: v for k, v in m.items() if k != "confusion_matrix"}})

                print(f"  ExpA Y={test_year} H={horizon:3d} {feat_name:16s} {model_name:12s}: "
                      f"acc={m['accuracy']:.3f} F1={m['macro_f1']:.3f} W1={m['within_one_acc']:.3f} ({elapsed:.0f}s)")

            # Incremental save
            pd.DataFrame(rows).to_csv(MULTI_CLS_CSV, index=False)

    print(f"  --- Year {test_year} done ({time.time()-t_total:.0f}s) ---")

# --- Experiment B: Hold-out station ---
print("\n--- EXPERIMENT B: Hold-Out Station ---")
for test_station in SELECTED_STATIONS:
    for horizon in EXP_HORIZONS:
        for feat_name, feature_cols in EXP_FEATURE_SETS.items():
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            tr_k, va_k, te_k = get_fold_keys("hold_out_station", test_station)
            trl, vl, tel, raw, scaler = build_multi_classification_fold(
                tr_k, va_k, te_k, feature_cols, 24, horizon, 128)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_o, yv_o, yte_o = raw

            base_row = {"experiment": "hold_out_station", "test_value": test_station,
                       "horizon": horizon, "features": feat_name,
                       "n_train": len(ytr_cls), "n_val": len(yv_cls), "n_test": len(yte_cls)}

            # Persistence baseline
            pers_s = persistence_predict(Xte, feature_cols.index("SWC_20"))
            pers_o = inverse_transform_target(scaler, feature_cols, pers_s)
            pers_cls = swc_to_class(pers_o)
            pers_m = compute_classification_metrics(yte_cls, pers_cls)
            diff_p = np.abs(yte_cls - pers_cls)
            rows.append({**base_row, "model": "persistence", "loss_type": "N/A",
                        "epochs_run": 0, "within_two_acc": (diff_p <= 2).mean(),
                        **{k: v for k, v in pers_m.items() if k != "confusion_matrix"}})

            # Majority class baseline
            maj_cls = majority_class_predict(ytr_cls, len(yte_cls))
            maj_m = compute_classification_metrics(yte_cls, maj_cls)
            diff_j = np.abs(yte_cls - maj_cls)
            rows.append({**base_row, "model": "majority_class", "loss_type": "N/A",
                        "epochs_run": 0, "within_two_acc": (diff_j <= 2).mean(),
                        **{k: v for k, v in maj_m.items() if k != "confusion_matrix"}})

            # CNN models
            for model_name, model_fn in CLASS_MODELS.items():
                random.seed(SEED); np.random.seed(SEED)
                torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

                t0 = time.time()
                model = model_fn(num_features=len(feature_cols))
                model, trh, vlh, stopped = train_classifier(model, trl, vl, epochs=40, patience=8)
                preds_cls, _, probs = predict_classes(model, tel)
                m = compute_classification_metrics(yte_cls, preds_cls)
                diff = np.abs(yte_cls - preds_cls)
                elapsed = time.time() - t0
                run_count += 1

                rows.append({**base_row, "model": model_name, "loss_type": "regular_CE",
                            "epochs_run": stopped, "best_val_loss": min(vlh),
                            "train_seconds": round(elapsed, 1),
                            "within_two_acc": (diff <= 2).mean(),
                            **{k: v for k, v in m.items() if k != "confusion_matrix"}})

                print(f"  ExpB S={test_station} H={horizon:3d} {feat_name:16s} {model_name:12s}: "
                      f"acc={m['accuracy']:.3f} F1={m['macro_f1']:.3f} W1={m['within_one_acc']:.3f} ({elapsed:.0f}s)")

            # Incremental save
            pd.DataFrame(rows).to_csv(MULTI_CLS_CSV, index=False)

    print(f"  --- Station {test_station} done ({time.time()-t_total:.0f}s) ---")

# --- Final summary ---
df_multi_cls = pd.DataFrame(rows)
df_multi_cls.to_csv(MULTI_CLS_CSV, index=False)
print(f"\n{'='*80}")
print(f"COMPLETE: {len(rows)} rows ({run_count} CNN runs) in {time.time()-t_total:.0f}s")
print(f"Saved to {MULTI_CLS_CSV}")

# Print summary tables
print(f"\n--- Exp A: Hold-Out Year (SWC_20 only, averaged over 3 test years) ---")
exp_a = df_multi_cls[(df_multi_cls["experiment"] == "hold_out_year") & (df_multi_cls["features"] == "SWC_20 only")]
for h in EXP_HORIZONS:
    print(f"  H={h}:")
    for mn in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = exp_a[(exp_a["horizon"] == h) & (exp_a["model"] == mn)]
        if not sub.empty:
            print(f"    {mn:16s}: acc={sub['accuracy'].mean():.3f} F1={sub['macro_f1'].mean():.3f} "
                  f"wF1={sub['weighted_f1'].mean():.3f} W1={sub['within_one_acc'].mean():.3f} "
                  f"MACE={sub['mean_abs_class_error'].mean():.3f}")

print(f"\n--- Exp B: Hold-Out Station (SWC_20 only, averaged over 4 stations) ---")
exp_b = df_multi_cls[(df_multi_cls["experiment"] == "hold_out_station") & (df_multi_cls["features"] == "SWC_20 only")]
for h in EXP_HORIZONS:
    print(f"  H={h}:")
    for mn in ["SmallCNN", "LargerCNN", "MaxpoolCNN", "persistence", "majority_class"]:
        sub = exp_b[(exp_b["horizon"] == h) & (exp_b["model"] == mn)]
        if not sub.empty:
            print(f"    {mn:16s}: acc={sub['accuracy'].mean():.3f} F1={sub['macro_f1'].mean():.3f} "
                  f"wF1={sub['weighted_f1'].mean():.3f} W1={sub['within_one_acc'].mean():.3f} "
                  f"MACE={sub['mean_abs_class_error'].mean():.3f}")

# Per-station breakdown for Exp B
print(f"\n--- Exp B: Per-Station Accuracy (SWC_20 only, MaxpoolCNN) ---")
for h in EXP_HORIZONS:
    print(f"  H={h}:")
    for station in SELECTED_STATIONS:
        sub = exp_b[(exp_b["horizon"] == h) & (exp_b["model"] == "MaxpoolCNN") &
                    (exp_b["test_value"] == station)]
        if not sub.empty:
            row = sub.iloc[0]
            print(f"    {station}: acc={row['accuracy']:.3f} F1={row['macro_f1']:.3f} W1={row['within_one_acc']:.3f}")


In [ ]:

import pandas as pd
df = pd.read_csv("multi_station_classification.csv")
print(f"Total rows: {len(df)}")
print(f"\nBy experiment:")
print(df.groupby("experiment").size())
print(f"\nExp B stations completed:")
exp_b = df[df["experiment"] == "hold_out_station"]
print(exp_b.groupby("test_value").size())
print(f"\nLast 5 rows:")
print(df[["experiment","test_value","horizon","features","model"]].tail(5))


In [ ]:

# ============================================================
# Multi-Station Classification Figures
# Run AFTER cell 100 (multi-station experiments) completes
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 150})
import numpy as np
import pandas as pd
import base64, io

df = pd.read_csv("multi_station_classification.csv")
print(f"Loaded {len(df)} rows from multi_station_classification.csv")

# Also load single-station results for comparison
df_single = pd.read_csv("classification_results.csv")
df_single_reg = df_single[df_single["loss_type"] != "weighted_CE"]

# Colors
COLORS = {'SmallCNN': '#4e79a7', 'LargerCNN': '#f28e2b', 'MaxpoolCNN': '#e15759',
           'persistence': '#76b7b2', 'majority_class': '#bab0ac'}

# ── Figure 1: Exp A vs Single-Station (temporal generalization boost) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for idx, h in enumerate([24, 168]):
    ax = axes[idx]
    models = ['SmallCNN', 'LargerCNN', 'MaxpoolCNN', 'persistence']
    x = np.arange(len(models))
    width = 0.35

    # Single-station averages (SWC_20 only, regular CE)
    single_accs = []
    for m in models:
        sub = df_single_reg[(df_single_reg["horizon"] == h) &
                            (df_single_reg["model"] == m) &
                            (df_single_reg["features"] == "SWC_20 only")]
        single_accs.append(sub["accuracy"].mean() if not sub.empty else 0)

    # Multi-station Exp A averages (SWC_20 only)
    exp_a = df[(df["experiment"] == "hold_out_year") & (df["features"] == "SWC_20 only")]
    multi_accs = []
    for m in models:
        sub = exp_a[(exp_a["horizon"] == h) & (exp_a["model"] == m)]
        multi_accs.append(sub["accuracy"].mean() if not sub.empty else 0)

    bars1 = ax.bar(x - width/2, single_accs, width, label='Single-station (FD11)',
                   color='#a0cbe8', edgecolor='#4e79a7', linewidth=0.8)
    bars2 = ax.bar(x + width/2, multi_accs, width, label='Multi-station (4 stations)',
                   color='#4e79a7', edgecolor='#2a5a8a', linewidth=0.8)

    ax.set_xlabel('Model')
    ax.set_ylabel('Accuracy (3-year avg)')
    ax.set_title(f'H = {h}h')
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, ha='right')
    ax.legend(fontsize=9)

    if h == 24:
        ax.set_ylim(0.8, 1.0)
    else:
        ax.set_ylim(0.4, 0.85)

    for bar_set in [bars1, bars2]:
        for bar in bar_set:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                       xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7)

fig.suptitle('Single-Station vs Multi-Station Classification Accuracy\n(SWC_20 only, regular CE)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='pdf', bbox_inches='tight'); buf.seek(0)
b64_1 = base64.b64encode(buf.read()).decode()
fig.savefig("multi_cls_single_vs_multi.pdf", bbox_inches='tight')
fig.savefig("multi_cls_single_vs_multi.png", bbox_inches='tight')
plt.show()
print(f"Fig 1 saved: {len(b64_1)} chars base64")

# ── Figure 2: Exp B per-station accuracy breakdown (MaxpoolCNN) ──
exp_b = df[(df["experiment"] == "hold_out_station") & (df["features"] == "SWC_20 only")]
stations = ["FD11", "FD02", "CB06", "CB25"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, h in enumerate([24, 168]):
    ax = axes[idx]
    models = ['SmallCNN', 'LargerCNN', 'MaxpoolCNN', 'persistence']
    x = np.arange(len(stations))
    width = 0.18

    for j, m in enumerate(models):
        accs = []
        for s in stations:
            sub = exp_b[(exp_b["horizon"] == h) & (exp_b["model"] == m) &
                        (exp_b["test_value"] == s)]
            accs.append(sub["accuracy"].values[0] if not sub.empty else 0)
        bars = ax.bar(x + (j - 1.5)*width, accs, width, label=m, color=COLORS[m],
                      edgecolor='black', linewidth=0.5)

    ax.set_xlabel('Hold-out Station')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'H = {h}h')
    ax.set_xticks(x)
    ax.set_xticklabels(stations)
    ax.legend(fontsize=8, loc='lower right')

    if h == 24:
        ax.set_ylim(0.5, 1.0)
    else:
        ax.set_ylim(0.2, 0.8)

fig.suptitle('Hold-Out Station Classification Accuracy (Exp B)\n(SWC_20 only, regular CE)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='pdf', bbox_inches='tight'); buf.seek(0)
b64_2 = base64.b64encode(buf.read()).decode()
fig.savefig("multi_cls_holdout_station.pdf", bbox_inches='tight')
fig.savefig("multi_cls_holdout_station.png", bbox_inches='tight')
plt.show()
print(f"Fig 2 saved: {len(b64_2)} chars base64")

# ── Figure 3: Exp A vs Exp B (temporal vs spatial generalization) ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, h in enumerate([24, 168]):
    ax = axes[idx]
    models = ['SmallCNN', 'LargerCNN', 'MaxpoolCNN']
    x = np.arange(len(models))
    width = 0.25

    # Exp A average
    exp_a_filt = df[(df["experiment"] == "hold_out_year") & (df["features"] == "SWC_20 only")]
    a_accs = []
    for m in models:
        sub = exp_a_filt[(exp_a_filt["horizon"] == h) & (exp_a_filt["model"] == m)]
        a_accs.append(sub["accuracy"].mean() if not sub.empty else 0)

    # Exp B average
    b_accs = []
    for m in models:
        sub = exp_b[(exp_b["horizon"] == h) & (exp_b["model"] == m)]
        b_accs.append(sub["accuracy"].mean() if not sub.empty else 0)

    # Single-station
    s_accs = []
    for m in models:
        sub = df_single_reg[(df_single_reg["horizon"] == h) &
                            (df_single_reg["model"] == m) &
                            (df_single_reg["features"] == "SWC_20 only")]
        s_accs.append(sub["accuracy"].mean() if not sub.empty else 0)

    bars1 = ax.bar(x - width, s_accs, width, label='Single-station (FD11)',
                   color='#a0cbe8', edgecolor='#4e79a7', linewidth=0.8)
    bars2 = ax.bar(x, a_accs, width, label='Exp A: Hold-out year',
                   color='#4e79a7', edgecolor='#2a5a8a', linewidth=0.8)
    bars3 = ax.bar(x + width, b_accs, width, label='Exp B: Hold-out station',
                   color='#e15759', edgecolor='#b03a3c', linewidth=0.8)

    ax.set_xlabel('Architecture')
    ax.set_ylabel('Accuracy (averaged)')
    ax.set_title(f'H = {h}h')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend(fontsize=8)

    for bar_set in [bars1, bars2, bars3]:
        for bar in bar_set:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                       xytext=(0, 3), textcoords="offset points", ha='center', fontsize=7)

    if h == 24:
        ax.set_ylim(0.7, 1.0)
    else:
        ax.set_ylim(0.4, 0.8)

fig.suptitle('Temporal vs Spatial Generalization in Classification\n(SWC_20 only, regular CE)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
buf = io.BytesIO(); fig.savefig(buf, format='pdf', bbox_inches='tight'); buf.seek(0)
b64_3 = base64.b64encode(buf.read()).decode()
fig.savefig("multi_cls_temporal_vs_spatial.pdf", bbox_inches='tight')
fig.savefig("multi_cls_temporal_vs_spatial.png", bbox_inches='tight')
plt.show()
print(f"Fig 3 saved: {len(b64_3)} chars base64")

# Print base64 for local download
print("\n" + "="*80)
print("BASE64 PDF DATA FOR LOCAL DOWNLOAD")
print("="*80)
for name, b64 in [("multi_cls_single_vs_multi.pdf", b64_1),
                   ("multi_cls_holdout_station.pdf", b64_2),
                   ("multi_cls_temporal_vs_spatial.pdf", b64_3)]:
    print(f"\n--- {name} ---")
    print(b64)
    print(f"--- END {name} ---")


In [ ]:
print("Runtime alive check:", __import__('time').strftime('%H:%M:%S'))

In [ ]:
# ============================================================
# BOOTSTRAP: Rebuild all needed state for Sections 20-24
# Defines classification infrastructure, multi-station data,
# and architecture classes without rerunning old experiments.
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, os, time, random
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from torch.utils.data import TensorDataset, DataLoader

# --- 12-class classification setup ---
N_CLASSES = 12
BIN_WIDTH = 0.05
BIN_EDGES = np.arange(0.0, 0.65, BIN_WIDTH)

def swc_to_class(values):
    values = np.asarray(values, dtype=np.float64)
    clipped = np.clip(values, 0.0, 0.60)
    classes = np.floor(clipped / BIN_WIDTH).astype(int)
    classes = np.clip(classes, 0, N_CLASSES - 1)
    return classes

def class_to_center(classes):
    return (np.asarray(classes) * BIN_WIDTH) + BIN_WIDTH / 2

def compute_classification_metrics(y_true_classes, y_pred_classes, n_classes=N_CLASSES):
    acc = accuracy_score(y_true_classes, y_pred_classes)
    macro_p = precision_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    macro_r = recall_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true_classes, y_pred_classes, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true_classes, y_pred_classes, average='weighted', zero_division=0)
    diff = np.abs(y_true_classes - y_pred_classes)
    within_one = (diff <= 1).mean()
    mean_abs_class_err = diff.mean()
    cm = confusion_matrix(y_true_classes, y_pred_classes, labels=range(n_classes))
    per_class_f1 = f1_score(y_true_classes, y_pred_classes, average=None, labels=range(n_classes), zero_division=0)
    return {
        "accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r,
        "macro_f1": macro_f1, "weighted_f1": weighted_f1,
        "within_one_accuracy": within_one, "mean_abs_class_error": mean_abs_class_err,
        "confusion_matrix": cm, "per_class_f1": per_class_f1
    }

# --- Classification CNN ---
class ClassificationCNN(nn.Module):
    def __init__(self, num_features, n_classes=12, n_layers=2, filters=None,
                 kernel_size=3, pool_type='avg', dropout=0.0):
        super().__init__()
        if filters is None:
            filters = [16, 32][:n_layers]
        p = kernel_size // 2
        layers = []
        in_ch = num_features
        for f in filters:
            layers.append(nn.Conv1d(in_ch, f, kernel_size, padding=p))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = f
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)
        hidden = max(filters[-1] // 2, 4)
        self.fc1 = nn.Linear(filters[-1], hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.convs(x)
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def train_classifier(model, train_loader, val_loader, epochs=40, patience=8,
                     lr=1e-3, class_weights=None):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))
    else:
        criterion = nn.CrossEntropyLoss()
    train_losses, val_losses = [], []
    best_val, best_state, bad_epochs, stopped_epoch = float("inf"), None, 0, epochs
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(xb)
        train_losses.append(running / len(train_loader.dataset))
        model.eval()
        running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                running += criterion(model(xb), yb).item() * len(xb)
        val_loss = running / len(val_loader.dataset)
        val_losses.append(val_loss)
        if val_loss < best_val - 1e-7:
            best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                stopped_epoch = epoch + 1
                break
    model.load_state_dict(best_state)
    return model, train_losses, val_losses, stopped_epoch

@torch.no_grad()
def predict_classes(model, loader):
    model.eval()
    all_preds, all_trues = [], []
    for xb, yb in loader:
        logits = model(xb.to(DEVICE))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_trues.append(yb.numpy())
    return np.concatenate(all_preds), np.concatenate(all_trues)

# --- FlexCNN (regression) and MaxpoolCNN ---
class FlexCNN(nn.Module):
    def __init__(self, num_features, n_layers=2, filters=None, kernel_size=3,
                 pool_type='avg', dropout=0.0):
        super().__init__()
        if filters is None:
            filters = [16, 32, 64][:n_layers]
        p = kernel_size // 2
        layers = []
        in_ch = num_features
        for f in filters:
            layers.append(nn.Conv1d(in_ch, f, kernel_size, padding=p))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = f
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)
        hidden = max(filters[-1] // 2, 4)
        self.fc1 = nn.Linear(filters[-1], hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.convs(x)
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# MaxpoolCNN is just LargerCNN with max pooling
class MaxpoolCNN(nn.Module):
    def __init__(self, num_features, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv1d(num_features, 16, kernel_size, padding=p)
        self.conv2 = nn.Conv1d(16, 32, kernel_size, padding=p)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

MODEL_CLASSES["maxpool_cnn"] = MaxpoolCNN

# --- Multi-station data ---
SELECTED_STATIONS = ["FD11", "FD02", "CB06", "CB25"]
df_multi = df_all[df_all["Station"].isin(SELECTED_STATIONS)].copy()
df_multi["Timestamp"] = pd.to_datetime(df_multi["Timestamp"])
df_multi["year"] = df_multi["Timestamp"].dt.year

STATION_YEAR_DATA = {}
for station in SELECTED_STATIONS:
    sdf = df_multi[df_multi["Station"] == station].copy()
    for yr in ALL_YEARS:
        g = sdf[sdf["year"] == yr].sort_values("Timestamp").reset_index(drop=True)
        if len(g) > 0:
            g["Ppt"] = g["Ppt"].fillna(0.0).clip(lower=0.0)
            g["Ppt_log"] = np.log1p(g["Ppt"])
            for col in ["Tair", "T_5", "T_10", "T_20", "T_50",
                        "SWC_5", "SWC_10", "SWC_50", "RH", "Srad"]:
                if col in g.columns:
                    g[col] = g[col].ffill().bfill().fillna(0.0)
            doy = g["Timestamp"].dt.dayofyear
            g["sin_doy"] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32)
            g["cos_doy"] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)
            g["Ppt_3h"] = g["Ppt"].rolling(3, min_periods=1).sum()
            g["Ppt_6h"] = g["Ppt"].rolling(6, min_periods=1).sum()
            g["Ppt_12h"] = g["Ppt"].rolling(12, min_periods=1).sum()
            g["Ppt_24h"] = g["Ppt"].rolling(24, min_periods=1).sum()
            STATION_YEAR_DATA[(station, yr)] = g

print(f"Bootstrap complete:")
print(f"  DEVICE={DEVICE}, GPU={torch.cuda.get_device_name(0)}")
print(f"  df_all: {df_all.shape}")
print(f"  SELECTED_STATIONS: {SELECTED_STATIONS}")
print(f"  STATION_YEAR_DATA entries: {len(STATION_YEAR_DATA)}")
print(f"  Models: SmallCNN, LargerCNN, MaxpoolCNN, FlexCNN, ClassificationCNN")
print(f"  Classification: swc_to_class, compute_classification_metrics, train_classifier, predict_classes")
print(f"  N_CLASSES={N_CLASSES}, BIN_EDGES={BIN_EDGES}")


In [ ]:

# Quick runtime + CSV check (independent of cell 100 state)
import os, time
print(f"Runtime alive: {time.strftime('%H:%M:%S')}")
csv_path = "multi_station_classification.csv"
if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f"CSV exists: {len(df)} rows")
    print(f"\nBy experiment:")
    print(df.groupby("experiment").size())
    print(f"\nExp B stations x horizons:")
    eb = df[df["experiment"]=="hold_out_station"]
    print(eb.groupby(["test_value","horizon"]).size().unstack(fill_value=0))

    # Print the summary tables we need for LaTeX
    print("\n" + "="*80)
    print("SUMMARY FOR LATEX TABLES")
    print("="*80)

    # Exp A: Hold-Out Year (SWC_20 only, averaged over 3 test years)
    exp_a = df[(df["experiment"]=="hold_out_year") & (df["features"]=="SWC_20 only")]
    print("\n--- Exp A: Hold-Out Year (SWC_20 only, 3-year avg) ---")
    for h in [24, 168]:
        print(f"  H={h}:")
        for mn in ["SmallCNN","LargerCNN","MaxpoolCNN","persistence","majority_class"]:
            sub = exp_a[(exp_a["horizon"]==h) & (exp_a["model"]==mn)]
            if not sub.empty:
                print(f"    {mn:16s}: acc={sub['accuracy'].mean():.3f} F1={sub['macro_f1'].mean():.3f} "
                      f"wF1={sub['weighted_f1'].mean():.3f} W1={sub['within_one_acc'].mean():.3f} "
                      f"MACE={sub['mean_abs_class_error'].mean():.3f}")

    # Exp B: Hold-Out Station (SWC_20 only, averaged over stations)
    exp_b = df[(df["experiment"]=="hold_out_station") & (df["features"]=="SWC_20 only")]
    print("\n--- Exp B: Hold-Out Station (SWC_20 only, avg over stations) ---")
    for h in [24, 168]:
        print(f"  H={h}:")
        for mn in ["SmallCNN","LargerCNN","MaxpoolCNN","persistence","majority_class"]:
            sub = exp_b[(exp_b["horizon"]==h) & (exp_b["model"]==mn)]
            if not sub.empty:
                n_stations = sub["test_value"].nunique()
                print(f"    {mn:16s}: acc={sub['accuracy'].mean():.3f} F1={sub['macro_f1'].mean():.3f} "
                      f"wF1={sub['weighted_f1'].mean():.3f} W1={sub['within_one_acc'].mean():.3f} "
                      f"MACE={sub['mean_abs_class_error'].mean():.3f} (n={n_stations} stations)")

    # Per-station breakdown for MaxpoolCNN
    print("\n--- Exp B: Per-Station MaxpoolCNN (SWC_20 only) ---")
    for h in [24, 168]:
        print(f"  H={h}:")
        for station in ["FD11","FD02","CB06","CB25"]:
            sub = exp_b[(exp_b["horizon"]==h) & (exp_b["model"]=="MaxpoolCNN") & (exp_b["test_value"]==station)]
            if not sub.empty:
                r = sub.iloc[0]
                print(f"    {station}: acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f} "
                      f"wF1={r['weighted_f1']:.3f} W1={r['within_one_acc']:.3f} MACE={r['mean_abs_class_error']:.3f}")
            else:
                print(f"    {station}: MISSING")
else:
    print("CSV NOT FOUND - runtime may have restarted")


In [ ]:

# ============================================================
# 19.3  Classification experiments — FD11 single station
# ============================================================
CLASSIFICATION_CSV = "classification_results.csv"
CLASS_TEST_YEARS = [2023, 2020, 2018]
CLASS_HORIZONS = [24, 48, 72, 168]
CLASS_FEATURE_SETS = {
    "SWC_20 only":       ["SWC_20"],
    "SWC_20 + Tair":     ["SWC_20", "Tair"],
    "SWC_20 + Seasonal": ["SWC_20", "sin_doy", "cos_doy"],
}

cls_rows = []
t_total = time.time()

for test_year in CLASS_TEST_YEARS:
    for horizon in CLASS_HORIZONS:
        for feat_name, feature_cols in CLASS_FEATURE_SETS.items():
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            trl, vl, tel, raw, scaler = build_classification_fold(
                test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_orig, yv_orig, yte_orig = raw

            # --- Baselines ---
            # Persistence: predict last observed value's class
            # Need original-scale Xte for persistence (use Xtr scaler to inverse)
            pers_scaled = persistence_predict(Xte, feature_cols.index("SWC_20"))
            pers_orig = inverse_transform_target(scaler, feature_cols, pers_scaled)
            pers_cls = swc_to_class(pers_orig)
            pers_metrics = compute_classification_metrics(yte_cls, pers_cls)

            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "persistence_class", "epochs_run": 0,
                **{k: v for k, v in pers_metrics.items() if k != "confusion_matrix"}
            })

            # Majority class
            maj_cls = majority_class_predict(ytr_cls, len(yte_cls))
            maj_metrics = compute_classification_metrics(yte_cls, maj_cls)
            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "majority_class", "epochs_run": 0,
                **{k: v for k, v in maj_metrics.items() if k != "confusion_matrix"}
            })

            # --- Classification CNN ---
            t0 = time.time()
            model = ClassificationCNN(num_features=len(feature_cols))
            model, trh, vlh, stopped = train_classifier(
                model, trl, vl, epochs=40, patience=8)
            preds_cls, true_cls, probs = predict_classes(model, tel)
            cnn_metrics = compute_classification_metrics(yte_cls, preds_cls)

            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "classification_cnn", "epochs_run": stopped,
                "best_val_loss": min(vlh), "final_val_loss": vlh[-1],
                "train_seconds": round(time.time() - t0, 1),
                **{k: v for k, v in cnn_metrics.items() if k != "confusion_matrix"}
            })

            # --- Discretized regression for comparison ---
            # Train regression CNN, then bin its predictions
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
            trl_reg, vl_reg, tel_reg, raw_reg, scaler_reg = build_fold(
                test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            reg_model = LargerCNN(num_features=len(feature_cols))
            reg_model, reg_trh, reg_vlh, reg_stopped = train_model(
                reg_model, trl_reg, vl_reg, epochs=30, patience=6)
            reg_preds, reg_trues = evaluate_model(reg_model, tel_reg)
            reg_preds_orig = inverse_transform_target(scaler_reg, feature_cols, reg_preds.flatten())
            reg_preds_cls = swc_to_class(reg_preds_orig)

            disc_metrics = compute_classification_metrics(yte_cls, reg_preds_cls)
            cls_rows.append({
                "test_year": test_year, "horizon": horizon, "features": feat_name,
                "model": "discretized_regression", "epochs_run": reg_stopped,
                **{k: v for k, v in disc_metrics.items() if k != "confusion_matrix"}
            })

            # Save incrementally
            pd.DataFrame(cls_rows).to_csv(CLASSIFICATION_CSV, index=False)

            print(f"  Y={test_year} H={horizon:3d} {feat_name:20s}: "
                  f"CNN acc={cnn_metrics['accuracy']:.3f} F1={cnn_metrics['macro_f1']:.3f} "
                  f"w1={cnn_metrics['within_one_acc']:.3f} | "
                  f"DiscReg acc={disc_metrics['accuracy']:.3f} F1={disc_metrics['macro_f1']:.3f} "
                  f"w1={disc_metrics['within_one_acc']:.3f} | "
                  f"Pers acc={pers_metrics['accuracy']:.3f} "
                  f"({time.time()-t0:.0f}s)")

    print(f"  --- Year {test_year} done ({time.time()-t_total:.0f}s) ---\n")

pd.DataFrame(cls_rows).to_csv(CLASSIFICATION_CSV, index=False)
print(f"\nClassification experiments complete: {len(cls_rows)} rows in {time.time()-t_total:.0f}s")
print(f"Saved to {CLASSIFICATION_CSV}")


In [ ]:

# ============================================================
# 19.4  Classification results analysis + confusion matrices
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

cls_df = pd.read_csv(CLASSIFICATION_CSV)
print(f"Classification results: {len(cls_df)} rows")

# --- Summary table ---
print("\n" + "="*120)
print("CLASSIFICATION RESULTS SUMMARY")
print("="*120)
print(f"{'Year':>5s} {'H':>4s} {'Features':>20s} {'Model':>22s} "
      f"{'Acc':>6s} {'MacF1':>6s} {'WtF1':>6s} {'W1Acc':>6s} {'MeanErr':>7s}")
print("-"*120)

for ty in CLASS_TEST_YEARS:
    for h in CLASS_HORIZONS:
        for fn in CLASS_FEATURE_SETS:
            sub = cls_df[(cls_df["test_year"]==ty) & (cls_df["horizon"]==h) & (cls_df["features"]==fn)]
            for _, row in sub.iterrows():
                print(f"{ty:>5d} {h:>4d} {fn:>20s} {row['model']:>22s} "
                      f"{row['accuracy']:6.3f} {row['macro_f1']:6.3f} "
                      f"{row['weighted_f1']:6.3f} {row['within_one_acc']:6.3f} "
                      f"{row['mean_abs_class_error']:7.3f}")
        print()

# --- Classification CNN vs Discretized Regression comparison ---
print("\n" + "="*90)
print("CLASSIFICATION CNN vs DISCRETIZED REGRESSION (averaged over 3 test years)")
print("="*90)
for h in CLASS_HORIZONS:
    for fn in CLASS_FEATURE_SETS:
        cnn = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]==fn) & (cls_df["model"]=="classification_cnn")]
        dreg = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]==fn) & (cls_df["model"]=="discretized_regression")]
        pers = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]==fn) & (cls_df["model"]=="persistence_class")]
        if cnn.empty:
            continue
        print(f"\n  H={h}, {fn}:")
        print(f"    Classification CNN:      acc={cnn['accuracy'].mean():.3f}  F1={cnn['macro_f1'].mean():.3f}  "
              f"W1={cnn['within_one_acc'].mean():.3f}  MeanErr={cnn['mean_abs_class_error'].mean():.3f}")
        print(f"    Discretized Regression:  acc={dreg['accuracy'].mean():.3f}  F1={dreg['macro_f1'].mean():.3f}  "
              f"W1={dreg['within_one_acc'].mean():.3f}  MeanErr={dreg['mean_abs_class_error'].mean():.3f}")
        print(f"    Persistence (class):     acc={pers['accuracy'].mean():.3f}  F1={pers['macro_f1'].mean():.3f}  "
              f"W1={pers['within_one_acc'].mean():.3f}")

# --- Confusion matrix plots ---
# Re-generate confusion matrices for the best configuration
print("\n\nGenerating confusion matrices for H=24 and H=168, SWC_20 only, test_year=2020...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
titles = ["Classification CNN", "Discretized Regression", "Persistence (class)"]

for row_idx, horizon in enumerate([24, 168]):
    for col_idx, (model_label, model_type) in enumerate([
        ("Classification CNN", "classification_cnn"),
        ("Discretized Regression", "discretized_regression"),
        ("Persistence (class)", "persistence_class")
    ]):
        ax = axes[row_idx][col_idx]

        # Re-run to get confusion matrix
        random.seed(SEED); np.random.seed(SEED)
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        feature_cols = ["SWC_20"]
        trl_c, vl_c, tel_c, raw_c, scaler_c = build_classification_fold(
            2020, feature_cols, 24, horizon, batch_size=128, seed=SEED)
        _, ytr_cls_c, _, _, _, yte_cls_c, _, _, yte_orig_c = raw_c

        if model_type == "persistence_class":
            pers_s = persistence_predict(raw_c[4], 0)  # Xte
            pers_o = inverse_transform_target(scaler_c, feature_cols, pers_s)
            pred_cls = swc_to_class(pers_o)
        elif model_type == "classification_cnn":
            model_c = ClassificationCNN(num_features=1)
            model_c, _, _, _ = train_classifier(model_c, trl_c, vl_c, epochs=40, patience=8)
            pred_cls, _, _ = predict_classes(model_c, tel_c)
        else:
            # Discretized regression
            trl_r, vl_r, tel_r, _, sc_r = build_fold(
                2020, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            reg_m = LargerCNN(num_features=1)
            reg_m, _, _, _ = train_model(reg_m, trl_r, vl_r, epochs=30, patience=6)
            reg_p, _ = evaluate_model(reg_m, tel_r)
            reg_orig = inverse_transform_target(sc_r, feature_cols, reg_p.flatten())
            pred_cls = swc_to_class(reg_orig)

        cm = confusion_matrix(yte_cls_c, pred_cls, labels=range(N_CLASSES))
        # Normalize by row (true class)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm_norm = np.nan_to_num(cm_norm)

        im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='auto')
        ax.set_xticks(range(N_CLASSES))
        ax.set_yticks(range(N_CLASSES))
        ax.set_xticklabels(range(N_CLASSES), fontsize=7)
        ax.set_yticklabels(range(N_CLASSES), fontsize=7)
        ax.set_xlabel("Predicted class", fontsize=8)
        ax.set_ylabel("True class", fontsize=8)
        metrics_cm = compute_classification_metrics(yte_cls_c, pred_cls)
        ax.set_title(f"{model_label}\nH={horizon}, acc={metrics_cm['accuracy']:.3f}, "
                     f"F1={metrics_cm['macro_f1']:.3f}, W1={metrics_cm['within_one_acc']:.3f}",
                     fontsize=9)

plt.suptitle("Confusion Matrices — Test Year 2020, SWC_20 Only, W=24", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("classification_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.savefig("classification_confusion_matrices.pdf", bbox_inches="tight")
plt.show()
print("Saved classification_confusion_matrices.png/pdf")


In [ ]:

# ============================================================
# 19.5  Classification comparison bar charts
# ============================================================
import matplotlib.pyplot as plt

cls_df = pd.read_csv(CLASSIFICATION_CSV)

# --- Accuracy and F1 by horizon ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics_to_plot = [("accuracy", "Accuracy"), ("macro_f1", "Macro F1"), ("within_one_acc", "Within-1 Accuracy")]
model_colors = {
    "classification_cnn": "#2ca02c",
    "discretized_regression": "#4c72b0",
    "persistence_class": "#999999",
    "majority_class": "#cccccc",
}

for ax, (metric, title) in zip(axes, metrics_to_plot):
    # Average across test years, SWC_20 only
    for model_name, color in model_colors.items():
        vals = []
        for h in CLASS_HORIZONS:
            sub = cls_df[(cls_df["horizon"]==h) & (cls_df["features"]=="SWC_20 only") &
                         (cls_df["model"]==model_name)]
            vals.append(sub[metric].mean() if not sub.empty else 0)
        ax.plot(CLASS_HORIZONS, vals, 'o-', color=color, label=model_name, markersize=5)

    ax.set_xlabel("Horizon (hours)")
    ax.set_ylabel(title)
    ax.set_title(f"{title} by Horizon\n(SWC_20 only, avg over 3 test years)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.set_xticks(CLASS_HORIZONS)

plt.tight_layout()
plt.savefig("classification_by_horizon.png", dpi=150, bbox_inches="tight")
plt.savefig("classification_by_horizon.pdf", bbox_inches="tight")
plt.show()
print("Saved classification_by_horizon.png/pdf")

# --- Feature comparison at H=168 ---
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))

for ax, (metric, title) in zip(axes2, [("accuracy", "Accuracy"), ("within_one_acc", "Within-1 Accuracy")]):
    x = np.arange(len(CLASS_FEATURE_SETS))
    width = 0.25
    feat_names = list(CLASS_FEATURE_SETS.keys())

    for i, (model_name, color) in enumerate([
        ("classification_cnn", "#2ca02c"),
        ("discretized_regression", "#4c72b0"),
        ("persistence_class", "#999999"),
    ]):
        vals = []
        for fn in feat_names:
            sub = cls_df[(cls_df["horizon"]==168) & (cls_df["features"]==fn) &
                         (cls_df["model"]==model_name)]
            vals.append(sub[metric].mean() if not sub.empty else 0)
        ax.bar(x + (i-1)*width, vals, width, label=model_name, color=color, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(feat_names, fontsize=8)
    ax.set_ylabel(title)
    ax.set_title(f"{title} at H=168\n(avg over 3 test years)")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("classification_features_h168.png", dpi=150, bbox_inches="tight")
plt.savefig("classification_features_h168.pdf", bbox_inches="tight")
plt.show()
print("Saved classification_features_h168.png/pdf")

# --- Class distribution plot ---
fig3, ax3 = plt.subplots(figsize=(10, 4))
for ty, color in zip(CLASS_TEST_YEARS, ['#1f77b4', '#ff7f0e', '#2ca02c']):
    g_all = YEAR_FEATURES[ty]
    cls_dist = swc_to_class(g_all["SWC_20"].values)
    counts = np.bincount(cls_dist, minlength=N_CLASSES)
    pcts = counts / counts.sum() * 100
    ax3.bar(np.arange(N_CLASSES) + (CLASS_TEST_YEARS.index(ty)-1)*0.25,
            pcts, 0.25, label=f"{ty}", color=color, alpha=0.8)

ax3.set_xticks(range(N_CLASSES))
ax3.set_xticklabels([f"[{c*0.05:.2f},{(c+1)*0.05:.2f})" for c in range(N_CLASSES)],
                     rotation=45, ha='right', fontsize=7)
ax3.set_ylabel("% of observations")
ax3.set_title("SWC_20 Class Distribution by Test Year (FD11)")
ax3.legend()
ax3.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.savefig("class_distribution.pdf", bbox_inches="tight")
plt.show()
print("Saved class_distribution.png/pdf")


# 20. Soil Moisture Distribution Analysis

before choosing classification categories, examine box plots of soil moisture across all stations. Identify the effective data range, underrepresented categories, and whether a narrower shared category range is reasonable.

## 20.1 Station-Level Summary Statistics

For each station in the TxSON33 dataset, calculate observation count, missing values, min, max, mean, median, standard deviation, Q1, Q3, IQR, 5th and 95th percentiles of SWC_20.

In [ ]:
# 20.1 — Station-level SWC_20 summary statistics
import pandas as pd, numpy as np

df_all = pd.read_parquet("data/TxSON33_modeling_complete.parquet")
df_all["Timestamp"] = pd.to_datetime(df_all["Timestamp"])

stations = sorted(df_all["Station"].unique())
rows = []
for station in stations:
    sdf = df_all[df_all["Station"] == station]
    swc = sdf["SWC_20"].dropna()
    rows.append({
        "Station": station,
        "n_valid": len(swc),
        "n_missing": sdf["SWC_20"].isna().sum(),
        "min": swc.min(),
        "max": swc.max(),
        "mean": swc.mean(),
        "median": swc.median(),
        "std": swc.std(),
        "Q1": swc.quantile(0.25),
        "Q3": swc.quantile(0.75),
        "IQR": swc.quantile(0.75) - swc.quantile(0.25),
        "p5": swc.quantile(0.05),
        "p95": swc.quantile(0.95),
    })

station_stats = pd.DataFrame(rows).round(4)
station_stats.to_csv("station_swc20_summary.csv", index=False)
print(f"Summary statistics for {len(station_stats)} stations saved to station_swc20_summary.csv")
print()
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 150)
print(station_stats.to_string(index=False))

print(f"\nOverall SWC_20 range across all stations: [{station_stats['min'].min():.4f}, {station_stats['max'].max():.4f}]")
print(f"Median of station medians: {station_stats['median'].median():.4f}")
print(f"Stations with narrowest IQR: {station_stats.nsmallest(3, 'IQR')[['Station','IQR']].to_string(index=False)}")
print(f"Stations with widest IQR: {station_stats.nlargest(3, 'IQR')[['Station','IQR']].to_string(index=False)}")


## 20.2 Box Plots Across All Stations

Publication-quality box plots showing SWC_20 distributions for all stations, plus detailed views for the four selected stations (FD11, FD02, CB06, CB25).

In [ ]:
# 20.2 — All-station box plots + detailed views for selected stations
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 10})

# Collect SWC_20 by station
station_data = {}
for station in sorted(df_all["Station"].unique()):
    swc = df_all[df_all["Station"] == station]["SWC_20"].dropna().values
    if len(swc) > 0:
        station_data[station] = swc

# Sort stations by median SWC_20 for readability
sorted_stations = sorted(station_data.keys(), key=lambda s: np.median(station_data[s]))

# --- Figure 1: All-station box plot ---
fig, ax = plt.subplots(figsize=(16, 6))
bp = ax.boxplot([station_data[s] for s in sorted_stations],
                labels=sorted_stations, patch_artist=True, showfliers=False,
                medianprops=dict(color='red', linewidth=1.5))

SELECTED = ["FD11", "FD02", "CB06", "CB25"]
for i, stn in enumerate(sorted_stations):
    color = '#ff7f0e' if stn in SELECTED else '#4c72b0'
    bp['boxes'][i].set_facecolor(color)
    bp['boxes'][i].set_alpha(0.7)

ax.set_xlabel("Station")
ax.set_ylabel("SWC_20 (volumetric)")
ax.set_title("Soil Moisture Distribution Across All TxSON33 Stations\n(sorted by median, orange = selected stations)")
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.grid(alpha=0.3, axis='y')

# Add horizontal lines at original 12-class boundaries
for edge in np.arange(0.0, 0.65, 0.05):
    ax.axhline(edge, color='gray', linestyle=':', linewidth=0.5, alpha=0.4)

plt.tight_layout()
plt.savefig("all_station_boxplots.png", dpi=150, bbox_inches="tight")
plt.savefig("all_station_boxplots.pdf", bbox_inches="tight")
plt.show()
print("Saved all_station_boxplots.png and .pdf")

# --- Figure 2: Detailed box plot for selected stations ---
fig2, axes2 = plt.subplots(1, 4, figsize=(16, 5), sharey=True)
for ax, stn in zip(axes2, SELECTED):
    swc = station_data[stn]
    ax.boxplot([swc], patch_artist=True, showfliers=True,
               boxprops=dict(facecolor='#ff7f0e', alpha=0.7),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(f"{stn}\nn={len(swc):,}")
    ax.set_ylabel("SWC_20" if stn == SELECTED[0] else "")
    stats = f"med={np.median(swc):.3f}\nIQR={np.quantile(swc,0.75)-np.quantile(swc,0.25):.3f}\nrange=[{swc.min():.3f},{swc.max():.3f}]"
    ax.text(1.3, np.median(swc), stats, fontsize=8, va='center')
    for edge in np.arange(0.0, 0.65, 0.05):
        ax.axhline(edge, color='gray', linestyle=':', linewidth=0.5, alpha=0.4)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle("Detailed SWC_20 Distributions — Selected Stations", fontsize=12)
plt.tight_layout()
plt.savefig("selected_station_boxplots.png", dpi=150, bbox_inches="tight")
plt.savefig("selected_station_boxplots.pdf", bbox_inches="tight")
plt.show()
print("Saved selected_station_boxplots.png and .pdf")


## 20.3 Histograms and Distribution Comparisons

Individual station histograms, combined distribution, and analysis of which original 12-class categories are underrepresented.

In [ ]:
# 20.3 — Histograms and 12-class frequency analysis across stations
import matplotlib.pyplot as plt

SELECTED = ["FD11", "FD02", "CB06", "CB25"]
BIN_WIDTH = 0.05
BIN_EDGES = np.arange(0.0, 0.65, BIN_WIDTH)
N_CLASSES = 12

def swc_to_class(values):
    values = np.asarray(values, dtype=np.float64)
    clipped = np.clip(values, 0.0, 0.60)
    classes = np.floor(clipped / BIN_WIDTH).astype(int)
    return np.clip(classes, 0, N_CLASSES - 1)

# --- Figure 1: Individual station histograms ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, stn in zip(axes.flat, SELECTED):
    swc = df_all[df_all["Station"] == stn]["SWC_20"].dropna().values
    ax.hist(swc, bins=60, color='#4c72b0', edgecolor='white', linewidth=0.3, alpha=0.8)
    ax.axvline(np.median(swc), color='red', linestyle='--', linewidth=1.5, label=f'median={np.median(swc):.3f}')
    ax.set_title(f"{stn} (n={len(swc):,})", fontsize=11)
    ax.set_xlabel("SWC_20")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')
    for edge in BIN_EDGES:
        ax.axvline(edge, color='gray', linestyle=':', linewidth=0.4, alpha=0.5)

plt.suptitle("SWC_20 Histograms — Selected Stations", fontsize=13)
plt.tight_layout()
plt.savefig("station_histograms.png", dpi=150, bbox_inches="tight")
plt.savefig("station_histograms.pdf", bbox_inches="tight")
plt.show()

# --- Figure 2: Combined 4-station distribution ---
fig2, ax2 = plt.subplots(figsize=(12, 5))
for stn, color in zip(SELECTED, ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']):
    swc = df_all[df_all["Station"] == stn]["SWC_20"].dropna().values
    ax2.hist(swc, bins=60, alpha=0.5, color=color, label=stn, density=True)
ax2.set_xlabel("SWC_20")
ax2.set_ylabel("Density")
ax2.set_title("SWC_20 Distribution Comparison — Selected Stations")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("combined_station_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 3: 12-class frequency for each selected station ---
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 10))
class_labels = [f"[{c*0.05:.2f},{(c+1)*0.05:.2f})" for c in range(N_CLASSES)]
class_labels[-1] = f"[{10*0.05:.2f},{11*0.05:.2f}]"

for ax, stn in zip(axes3.flat, SELECTED):
    swc = df_all[df_all["Station"] == stn]["SWC_20"].dropna().values
    classes = swc_to_class(swc)
    counts = np.bincount(classes, minlength=N_CLASSES)
    pcts = counts / counts.sum() * 100

    colors = ['#d62728' if p < 1.0 else '#4c72b0' for p in pcts]
    bars = ax.bar(range(N_CLASSES), pcts, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(class_labels, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel("% of observations")
    ax.set_title(f"{stn} — 12-Class Distribution")
    ax.grid(alpha=0.3, axis='y')

    # Label empty/near-empty classes
    for i, (pct, count) in enumerate(zip(pcts, counts)):
        if count == 0:
            ax.text(i, 0.5, "0", ha='center', fontsize=7, color='red')
        elif pct < 1.0:
            ax.text(i, pct + 0.3, f"{count}", ha='center', fontsize=6, color='red')

plt.suptitle("12-Class SWC_20 Category Distribution — Selected Stations", fontsize=13)
plt.tight_layout()
plt.savefig("class_distribution_by_station.png", dpi=150, bbox_inches="tight")
plt.savefig("class_distribution_by_station.pdf", bbox_inches="tight")
plt.show()
print("Saved station_histograms, combined_station_distributions, class_distribution_by_station (.png/.pdf)")

# --- Identify underrepresented categories across all 4 stations ---
print("\n12-CLASS CATEGORY ANALYSIS (pooled across 4 selected stations):")
pooled_swc = np.concatenate([df_all[df_all["Station"]==s]["SWC_20"].dropna().values for s in SELECTED])
pooled_classes = swc_to_class(pooled_swc)
pooled_counts = np.bincount(pooled_classes, minlength=N_CLASSES)
pooled_pcts = pooled_counts / pooled_counts.sum() * 100

print(f"{'Class':>6s} {'Range':>14s} {'Count':>8s} {'%':>6s} {'Status':>15s}")
print("-" * 55)
for c in range(N_CLASSES):
    lo = c * 0.05
    hi = lo + 0.05
    status = "EMPTY" if pooled_counts[c] == 0 else "NEAR-EMPTY" if pooled_pcts[c] < 1.0 else "OK"
    print(f"{c:>6d} [{lo:.2f},{hi:.2f}) {pooled_counts[c]:>8,d} {pooled_pcts[c]:>6.1f}% {status:>15s}")

print(f"\nEmpty classes: {list(np.where(pooled_counts == 0)[0])}")
print(f"Near-empty (<1%): {list(np.where((pooled_counts > 0) & (pooled_pcts < 1.0))[0])}")
print(f"Effective range (classes with >1% each): classes {np.min(np.where(pooled_pcts >= 1.0))} to {np.max(np.where(pooled_pcts >= 1.0))}")


## 20.4 Seasonal Distribution Analysis

Investigate how SWC_20 varies across months and seasons. Generate seasonal box/violin plots to determine whether the dominant moisture range changes by season.

In [ ]:
# 20.4 — Seasonal SWC_20 distribution analysis
import matplotlib.pyplot as plt

SELECTED = ["FD11", "FD02", "CB06", "CB25"]
SEASONS = {"Winter (DJF)": [12, 1, 2], "Spring (MAM)": [3, 4, 5],
           "Summer (JJA)": [6, 7, 8], "Fall (SON)": [9, 10, 11]}
SEASON_COLORS = {"Winter (DJF)": "#4c72b0", "Spring (MAM)": "#55a868",
                 "Summer (JJA)": "#c44e52", "Fall (SON)": "#dd8452"}

# --- Figure 1: Seasonal box plots for selected stations ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, stn in zip(axes.flat, SELECTED):
    sdf = df_all[df_all["Station"] == stn].copy()
    sdf["month"] = pd.to_datetime(sdf["Timestamp"]).dt.month

    season_data = []
    season_labels = []
    colors = []
    for sname, months in SEASONS.items():
        swc = sdf[sdf["month"].isin(months)]["SWC_20"].dropna().values
        if len(swc) > 0:
            season_data.append(swc)
            season_labels.append(sname)
            colors.append(SEASON_COLORS[sname])

    bp = ax.boxplot(season_data, labels=season_labels, patch_artist=True,
                    showfliers=False, medianprops=dict(color='black', linewidth=1.5))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"{stn}", fontsize=11)
    ax.set_ylabel("SWC_20")
    ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=15, labelsize=8)

plt.suptitle("Seasonal SWC_20 Distributions — Selected Stations", fontsize=13)
plt.tight_layout()
plt.savefig("seasonal_boxplots.png", dpi=150, bbox_inches="tight")
plt.savefig("seasonal_boxplots.pdf", bbox_inches="tight")
plt.show()

# --- Figure 2: Monthly box plots for FD11 ---
fig2, ax2 = plt.subplots(figsize=(14, 5))
fd11_monthly = df_all[df_all["Station"] == "FD11"].copy()
fd11_monthly["month"] = pd.to_datetime(fd11_monthly["Timestamp"]).dt.month
month_data = [fd11_monthly[fd11_monthly["month"] == m]["SWC_20"].dropna().values for m in range(1, 13)]
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

bp2 = ax2.boxplot(month_data, labels=month_labels, patch_artist=True, showfliers=False,
                   medianprops=dict(color='red', linewidth=1.5))
month_colors = [SEASON_COLORS["Winter (DJF)"]]*2 + [SEASON_COLORS["Spring (MAM)"]]*3 + \
               [SEASON_COLORS["Summer (JJA)"]]*3 + [SEASON_COLORS["Fall (SON)"]]*3 + \
               [SEASON_COLORS["Winter (DJF)"]]
for patch, color in zip(bp2['boxes'], month_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_xlabel("Month")
ax2.set_ylabel("SWC_20")
ax2.set_title("Monthly SWC_20 Distribution — Station FD11 (All Years)")
ax2.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("monthly_boxplot_fd11.png", dpi=150, bbox_inches="tight")
plt.savefig("monthly_boxplot_fd11.pdf", bbox_inches="tight")
plt.show()

# Print seasonal statistics
print("\nSeasonal SWC_20 Statistics (selected stations):")
for stn in SELECTED:
    sdf = df_all[df_all["Station"] == stn].copy()
    sdf["month"] = pd.to_datetime(sdf["Timestamp"]).dt.month
    print(f"\n  {stn}:")
    for sname, months in SEASONS.items():
        swc = sdf[sdf["month"].isin(months)]["SWC_20"].dropna()
        if len(swc) > 0:
            print(f"    {sname}: median={swc.median():.4f}, IQR={swc.quantile(0.75)-swc.quantile(0.25):.4f}, "
                  f"range=[{swc.min():.4f}, {swc.max():.4f}]")

print("\nSaved seasonal_boxplots, monthly_boxplot_fd11 (.png/.pdf)")


## 20.5 Distribution Findings

**Station-level patterns:**
- Across 33 TxSON stations, SWC_20 ranges from 0.000 to 0.561.
- Median of station medians is 0.175 — most stations are drier than FD11 (median 0.328).
- Narrowest distributions: FD22 (IQR=0.002, near-zero readings), WC05 (IQR=0.002).
- Widest distributions: FD24 (IQR=0.173), CB25 (IQR=0.154), CB32 (IQR=0.153).
- The four selected stations span a good range: FD02 is narrow/dry (IQR=0.055), FD11 and CB25 are wide/wetter (IQR=0.116–0.154).

**Classification category analysis (pooled 4 stations):**
- Classes 0 and 11 are completely empty (no observations in [0.00, 0.05) or [0.55, 0.60]).
- Classes 9 and 10 are near-empty (<1% combined).
- The effective data range is classes 1–8, corresponding to SWC_20 ≈ 0.05–0.45.
- Class 3 [0.15, 0.20) is dominant at 25.0% of pooled observations.
- A narrower 8-class scheme spanning ~0.05–0.45 would eliminate empty categories and concentrate resolution where data exists.

**Seasonal variation:**
- Winter (DJF) has the highest median SWC_20 across all stations and the narrowest IQR — stable, wetter conditions.
- Summer (JJA) and Fall (SON) show the widest IQR — greater variability due to rain events and drying cycles.
- FD11's seasonal spread is substantial: summer median 0.272 vs. winter median 0.343.
- These seasonal patterns suggest that seasonal features (sin/cos day-of-year) may help long-horizon predictions.

**Recommendation:** A shared narrower category scheme (approximately 0.05–0.45, 8 classes) or quantile-based scheme should be tested. Category boundaries must come from training data only.

# 21. Improved Classification Category Experiments

Investigate whether alternative shared category definitions improve classification performance over the original 12-class scheme (0.00–0.60, width 0.05). Category boundaries must be consistent across all stations.

## 21.1 Original Classification Baseline

The original 12 categories span SWC_20 values 0.00–0.60 in increments of 0.05. Results from Section 19 serve as the reference. Key previously verified results (FD11, SWC_20 only, regular cross-entropy, averaged over 3 held-out years):

| Model | H24 Accuracy | H24 Macro F1 | H168 Accuracy | H168 Macro F1 |
|-------|-------------|-------------|--------------|--------------|
| SmallCNN | 0.866 | 0.663 | 0.595 | 0.439 |
| LargerCNN | 0.871 | 0.669 | 0.650 | 0.476 |
| MaxpoolCNN | 0.890 | 0.690 | 0.651 | 0.477 |
| Persistence | 0.888 | 0.695 | 0.584 | 0.442 |

## 21.2 Narrower Fixed-Range Categories

Based on the distribution analysis, investigate a narrower range (e.g., 0.10–0.50) with the same or reduced number of classes. Observations outside the range are mapped to boundary classes (not discarded).

In [ ]:
# 21.2 — Define alternative category schemes
import numpy as np

# Scheme A: Original 12-class (reference)
SCHEME_ORIGINAL = {
    "name": "Original 12-class",
    "edges": np.arange(0.0, 0.65, 0.05),
    "n_classes": 12,
}

# Scheme B: Narrower 8-class [0.10, 0.50) with 0.05 width
# Values below 0.10 go to class 0, above 0.50 go to class 7
SCHEME_NARROW_8 = {
    "name": "Narrow 8-class [0.10-0.50]",
    "edges": np.array([0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]),
    "n_classes": 8,
}

# Scheme C: Narrower 6-class [0.15, 0.45) with 0.05 width
SCHEME_NARROW_6 = {
    "name": "Narrow 6-class [0.15-0.45]",
    "edges": np.array([0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45]),
    "n_classes": 6,
}

def assign_scheme_classes(values, edges, n_classes):
    """Assign class indices using custom bin edges.
    Values below edges[0] go to class 0; above edges[-1] go to n_classes-1."""
    values = np.asarray(values, dtype=np.float64)
    classes = np.digitize(values, edges) - 1
    return np.clip(classes, 0, n_classes - 1)

# Verify schemes
print("Category Schemes Defined:")
for scheme in [SCHEME_ORIGINAL, SCHEME_NARROW_8, SCHEME_NARROW_6]:
    print(f"\n  {scheme['name']}: {scheme['n_classes']} classes")
    edges = scheme['edges']
    print(f"    Edges: {edges}")
    print(f"    Bins: ", end="")
    for c in range(scheme['n_classes']):
        if c == 0:
            lo_str = f"(-inf" if len(edges) < 13 else f"[{edges[0]:.2f}"
        else:
            lo_str = f"[{edges[c]:.2f}" if c < len(edges) else f"[{edges[-1]:.2f}"
        if c == scheme['n_classes'] - 1:
            hi_str = f"+inf)"
        else:
            hi_str = f"{edges[c+1]:.2f})" if c+1 < len(edges) else f"{edges[-1]:.2f})"
        print(f"C{c}={lo_str},{hi_str}", end=" ")
    print()

# Test on pooled 4-station data
SELECTED = ["FD11", "FD02", "CB06", "CB25"]
pooled_swc = np.concatenate([df_all[df_all["Station"]==s]["SWC_20"].dropna().values for s in SELECTED])

print(f"\nPooled 4-station data: {len(pooled_swc):,} observations")
for scheme in [SCHEME_ORIGINAL, SCHEME_NARROW_8, SCHEME_NARROW_6]:
    if scheme == SCHEME_ORIGINAL:
        classes = swc_to_class(pooled_swc)
    else:
        classes = assign_scheme_classes(pooled_swc, scheme['edges'], scheme['n_classes'])
    counts = np.bincount(classes, minlength=scheme['n_classes'])
    pcts = counts / counts.sum() * 100
    empty = (counts == 0).sum()
    near_empty = ((counts > 0) & (pcts < 1.0)).sum()
    min_pct = pcts[pcts > 0].min() if (pcts > 0).any() else 0
    print(f"\n  {scheme['name']}:")
    print(f"    Empty: {empty}, Near-empty (<1%): {near_empty}, Min non-zero: {min_pct:.1f}%")
    print(f"    Distribution: {' '.join(f'{p:.1f}%' for p in pcts)}")


## 21.3 Quantile-Based Categories

Calculate category boundaries from pooled training data so that each class has approximately equal representation. Same boundaries applied to all stations.

In [ ]:
# 21.3 — Quantile-based category boundaries from training data only
# Use pooled training data (excluding val year 2019 and test years)

VAL_YEAR = 2019
TEST_YEARS = [2023, 2020, 2018]
SELECTED = ["FD11", "FD02", "CB06", "CB25"]

# Collect training-only SWC_20 across all 4 stations
train_swc_list = []
for stn in SELECTED:
    sdf = df_all[df_all["Station"] == stn].copy()
    sdf["year"] = pd.to_datetime(sdf["Timestamp"]).dt.year
    train_mask = ~sdf["year"].isin(TEST_YEARS + [VAL_YEAR])
    train_swc_list.append(sdf[train_mask]["SWC_20"].dropna().values)

train_swc_pooled = np.concatenate(train_swc_list)
print(f"Training-only pooled SWC_20: {len(train_swc_pooled):,} observations")
print(f"Range: [{train_swc_pooled.min():.4f}, {train_swc_pooled.max():.4f}]")

# Quantile-based 8-class scheme
n_q = 8
quantile_edges = np.quantile(train_swc_pooled, np.linspace(0, 1, n_q + 1))
quantile_edges_unique = np.unique(quantile_edges.round(4))
print(f"\nRaw {n_q}-quantile edges: {quantile_edges.round(4)}")
print(f"Unique edges: {quantile_edges_unique}")

SCHEME_QUANTILE_8 = {
    "name": f"Quantile 8-class",
    "edges": quantile_edges[1:-1],  # internal edges only
    "n_classes": n_q,
}

# Quantile-based 6-class scheme
n_q6 = 6
quantile_edges_6 = np.quantile(train_swc_pooled, np.linspace(0, 1, n_q6 + 1))
SCHEME_QUANTILE_6 = {
    "name": f"Quantile 6-class",
    "edges": quantile_edges_6[1:-1],
    "n_classes": n_q6,
}

print(f"\nQuantile 8-class internal edges: {SCHEME_QUANTILE_8['edges'].round(4)}")
print(f"Quantile 6-class internal edges: {SCHEME_QUANTILE_6['edges'].round(4)}")

# Verify balance on training data
for scheme in [SCHEME_QUANTILE_8, SCHEME_QUANTILE_6]:
    classes = assign_scheme_classes(train_swc_pooled,
                                     np.concatenate([[train_swc_pooled.min()-0.01], scheme['edges'], [train_swc_pooled.max()+0.01]]),
                                     scheme['n_classes'])
    # Actually use np.digitize directly
    edges_full = np.concatenate([[-np.inf], scheme['edges'], [np.inf]])
    classes = np.digitize(train_swc_pooled, scheme['edges'])
    classes = np.clip(classes, 0, scheme['n_classes'] - 1)
    counts = np.bincount(classes, minlength=scheme['n_classes'])
    pcts = counts / counts.sum() * 100
    print(f"\n  {scheme['name']} on training data:")
    print(f"    Class sizes: {counts}")
    print(f"    Percentages: {' '.join(f'{p:.1f}%' for p in pcts)}")
    print(f"    Min/Max class: {pcts.min():.1f}% / {pcts.max():.1f}%")


## 21.4 IQR-Informed Categories

Use pooled training-set quartiles and IQR to design a shared category scheme that concentrates bins where most data falls.

In [ ]:
# 21.4 — IQR-informed category scheme
# Idea: narrower bins in the IQR where data is dense, wider bins in the tails.

q1_train = np.quantile(train_swc_pooled, 0.25)
q3_train = np.quantile(train_swc_pooled, 0.75)
iqr_train = q3_train - q1_train
median_train = np.median(train_swc_pooled)

print(f"Training data statistics:")
print(f"  Q1 = {q1_train:.4f}, Median = {median_train:.4f}, Q3 = {q3_train:.4f}")
print(f"  IQR = {iqr_train:.4f}")
print(f"  Range = [{train_swc_pooled.min():.4f}, {train_swc_pooled.max():.4f}]")

# IQR-informed 8-class scheme:
# - 2 wide tail classes below Q1 and above Q3
# - 4 narrow classes within the IQR
# Round edges to nice numbers
iqr_bin_width = round(iqr_train / 4, 3)
lower_tail_edge = round(q1_train - iqr_train * 0.5, 3)
upper_tail_edge = round(q3_train + iqr_train * 0.5, 3)

iqr_edges = [lower_tail_edge, round(q1_train, 3)]
for i in range(1, 4):
    iqr_edges.append(round(q1_train + i * iqr_bin_width, 3))
iqr_edges.append(round(q3_train, 3))
iqr_edges.append(upper_tail_edge)
iqr_edges = sorted(set(iqr_edges))

# Clean up: round to 2 decimals
iqr_edges = [round(e, 2) for e in iqr_edges]
iqr_edges = sorted(set(iqr_edges))
n_iqr_classes = len(iqr_edges) + 1

SCHEME_IQR = {
    "name": f"IQR-informed {n_iqr_classes}-class",
    "edges": np.array(iqr_edges),
    "n_classes": n_iqr_classes,
}

print(f"\nIQR-informed scheme: {n_iqr_classes} classes")
print(f"  Internal edges: {iqr_edges}")
print(f"  IQR bin width: ~{iqr_bin_width:.3f}")

# Verify on training data
classes_iqr = np.digitize(train_swc_pooled, SCHEME_IQR['edges'])
classes_iqr = np.clip(classes_iqr, 0, SCHEME_IQR['n_classes'] - 1)
counts_iqr = np.bincount(classes_iqr, minlength=SCHEME_IQR['n_classes'])
pcts_iqr = counts_iqr / counts_iqr.sum() * 100
print(f"  Distribution on training data:")
for c in range(SCHEME_IQR['n_classes']):
    lo = f"(-inf" if c == 0 else f"[{SCHEME_IQR['edges'][c-1]:.2f}"
    hi = f"+inf)" if c == SCHEME_IQR['n_classes']-1 else f"{SCHEME_IQR['edges'][c]:.2f})"
    print(f"    Class {c}: {lo},{hi} — {counts_iqr[c]:,} ({pcts_iqr[c]:.1f}%)")

# Summary of all schemes
ALL_SCHEMES = [SCHEME_ORIGINAL, SCHEME_NARROW_8, SCHEME_NARROW_6,
               SCHEME_QUANTILE_8, SCHEME_QUANTILE_6, SCHEME_IQR]
print(f"\n\nSUMMARY OF ALL CATEGORY SCHEMES:")
print(f"{'Scheme':>30s} {'N_classes':>10s} {'Empty':>6s} {'Min%':>6s} {'Max%':>6s}")
print("-" * 60)
for scheme in ALL_SCHEMES:
    if scheme == SCHEME_ORIGINAL:
        classes = swc_to_class(train_swc_pooled)
    else:
        classes = np.digitize(train_swc_pooled, scheme['edges'])
        classes = np.clip(classes, 0, scheme['n_classes'] - 1)
    counts = np.bincount(classes, minlength=scheme['n_classes'])
    pcts = counts / counts.sum() * 100
    empty = (counts == 0).sum()
    min_nz = pcts[pcts > 0].min() if (pcts > 0).any() else 0
    print(f"{scheme['name']:>30s} {scheme['n_classes']:>10d} {empty:>6d} {min_nz:>6.1f} {pcts.max():>6.1f}")


## 21.5 Classification Training with Alternative Schemes

Train MaxpoolCNN (best-performing architecture) on each category scheme at H=24 and H=168. FD11 single-station, SWC_20 only, regular cross-entropy. Compare against persistence and majority-class baselines.

In [ ]:
# 21.5 — Train classification models with alternative category schemes
import torch, torch.nn as nn, random, time
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_YEARS_CLS = [2023, 2020, 2018]
VAL_YEAR = 2019
ALL_YEARS = list(range(2015, 2025))
HORIZONS_CLS = [24, 168]

# FlexCNN for classification
class FlexClassCNN(nn.Module):
    def __init__(self, num_features, n_classes, n_layers=2, filters=None,
                 kernel_size=3, pool_type='max', dropout=0.0):
        super().__init__()
        if filters is None:
            filters = [16, 32][:n_layers]
        p = kernel_size // 2
        layers = []
        in_ch = num_features
        for f in filters:
            layers.append(nn.Conv1d(in_ch, f, kernel_size, padding=p))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = f
        self.convs = nn.Sequential(*layers)
        self.pool = nn.AdaptiveMaxPool1d(1) if pool_type == 'max' else nn.AdaptiveAvgPool1d(1)
        hidden = max(filters[-1] // 2, 4)
        self.fc1 = nn.Linear(filters[-1], hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.convs(x)
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def build_classification_fold_scheme(test_year, feature_cols, window_size, horizon,
                                      scheme, batch_size, seed=SEED):
    """Build classification fold with a given category scheme."""
    train_years = [y for y in ALL_YEARS if y not in (test_year, VAL_YEAR)]

    scaler = MinMaxScaler()
    scaler.fit(pd.concat([YEAR_FEATURES[y][feature_cols] for y in train_years], axis=0))

    def make_data(years):
        Xs, ys_orig = [], []
        for y in years:
            g = YEAR_FEATURES[y].copy()
            g[feature_cols] = scaler.transform(g[feature_cols])
            X, y_raw = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X):
                Xs.append(X); ys_orig.append(y_raw)
        X_all = np.concatenate(Xs)
        y_orig_all = np.concatenate(ys_orig)
        # Convert to original scale for binning
        y_real = inverse_transform_target(scaler, feature_cols, y_orig_all)
        # Assign classes
        if scheme == SCHEME_ORIGINAL:
            y_cls = swc_to_class(y_real)
        else:
            y_cls = np.digitize(y_real, scheme['edges'])
            y_cls = np.clip(y_cls, 0, scheme['n_classes'] - 1)
        return X_all, y_cls.astype(np.int64), y_real

    Xtr, ytr_cls, ytr_real = make_data(train_years)
    Xv, yv_cls, yv_real = make_data([VAL_YEAR])
    Xte, yte_cls, yte_real = make_data([test_year])

    g = torch.Generator().manual_seed(seed)
    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                         generator=g if shuffle else None)

    return (loader(Xtr, ytr_cls, True), loader(Xv, yv_cls, False), loader(Xte, yte_cls, False),
            (Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_real, yv_real, yte_real), scaler)

def train_flex_classifier(model, train_loader, val_loader, epochs=40, patience=8, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, bad_epochs, stopped = float("inf"), None, 0, epochs

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                stopped = epoch + 1
                break

    model.load_state_dict(best_state)
    return model, stopped

@torch.no_grad()
def predict_flex(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        logits = model(xb.to(DEVICE))
        preds.append(logits.argmax(dim=1).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(trues)

print("FlexClassCNN and scheme-aware fold builder defined")
print(f"FlexClassCNN params (F=1, 8-class): {sum(p.numel() for p in FlexClassCNN(1, 8).parameters() if p.requires_grad)}")
print(f"FlexClassCNN params (F=1, 12-class): {sum(p.numel() for p in FlexClassCNN(1, 12).parameters() if p.requires_grad)}")


In [ ]:
# 21.5 — Run classification experiments for all schemes
SCHEME_RESULTS_CSV = "scheme_comparison_results.csv"
scheme_rows = []
t_total = time.time()

SCHEMES_TO_TEST = [SCHEME_ORIGINAL, SCHEME_NARROW_8, SCHEME_NARROW_6,
                   SCHEME_QUANTILE_8, SCHEME_QUANTILE_6, SCHEME_IQR]

for scheme in SCHEMES_TO_TEST:
    for test_year in TEST_YEARS_CLS:
        for horizon in HORIZONS_CLS:
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            feature_cols = ["SWC_20"]
            trl, vl, tel, raw, scaler = build_classification_fold_scheme(
                test_year, feature_cols, 24, horizon, scheme, batch_size=128, seed=SEED)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_real, yv_real, yte_real = raw

            # --- Persistence baseline ---
            pers_scaled = Xte[:, -1, 0]  # last value in window
            pers_orig = inverse_transform_target(scaler, feature_cols, pers_scaled)
            if scheme == SCHEME_ORIGINAL:
                pers_cls = swc_to_class(pers_orig)
            else:
                pers_cls = np.digitize(pers_orig, scheme['edges'])
                pers_cls = np.clip(pers_cls, 0, scheme['n_classes'] - 1)

            pers_acc = accuracy_score(yte_cls, pers_cls)
            pers_f1 = f1_score(yte_cls, pers_cls, average='macro', zero_division=0)
            pers_wf1 = f1_score(yte_cls, pers_cls, average='weighted', zero_division=0)
            pers_w1 = (np.abs(yte_cls - pers_cls) <= 1).mean()
            pers_mace = np.abs(yte_cls - pers_cls).mean()

            scheme_rows.append({
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "test_year": test_year, "horizon": horizon, "model": "persistence",
                "accuracy": pers_acc, "macro_f1": pers_f1, "weighted_f1": pers_wf1,
                "within_one_acc": pers_w1, "mean_abs_class_error": pers_mace,
            })

            # --- Majority class baseline ---
            maj = np.bincount(ytr_cls).argmax()
            maj_cls = np.full(len(yte_cls), maj)
            maj_acc = accuracy_score(yte_cls, maj_cls)
            maj_f1 = f1_score(yte_cls, maj_cls, average='macro', zero_division=0)
            scheme_rows.append({
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "test_year": test_year, "horizon": horizon, "model": "majority",
                "accuracy": maj_acc, "macro_f1": maj_f1,
                "weighted_f1": f1_score(yte_cls, maj_cls, average='weighted', zero_division=0),
                "within_one_acc": (np.abs(yte_cls - maj_cls) <= 1).mean(),
                "mean_abs_class_error": np.abs(yte_cls - maj_cls).mean(),
            })

            # --- MaxpoolCNN (best architecture) ---
            t0 = time.time()
            model = FlexClassCNN(num_features=1, n_classes=scheme['n_classes'],
                                  n_layers=2, filters=[16, 32], pool_type='max')
            model, stopped = train_flex_classifier(model, trl, vl, epochs=40, patience=8)
            preds_cls, true_cls = predict_flex(model, tel)

            cnn_acc = accuracy_score(yte_cls, preds_cls)
            cnn_f1 = f1_score(yte_cls, preds_cls, average='macro', zero_division=0)
            cnn_wf1 = f1_score(yte_cls, preds_cls, average='weighted', zero_division=0)
            cnn_w1 = (np.abs(yte_cls - preds_cls) <= 1).mean()
            cnn_mace = np.abs(yte_cls - preds_cls).mean()

            # Convert class errors to SWC units for fair comparison
            # Average bin width for this scheme
            if scheme == SCHEME_ORIGINAL:
                avg_bin_width = 0.05
            else:
                total_range = scheme['edges'][-1] - scheme['edges'][0] if len(scheme['edges']) > 1 else 0.05
                avg_bin_width = total_range / max(scheme['n_classes'] - 2, 1)

            scheme_rows.append({
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "test_year": test_year, "horizon": horizon, "model": "MaxpoolCNN",
                "accuracy": cnn_acc, "macro_f1": cnn_f1, "weighted_f1": cnn_wf1,
                "within_one_acc": cnn_w1, "mean_abs_class_error": cnn_mace,
                "mace_swc_units": cnn_mace * avg_bin_width,
                "epochs_run": stopped, "train_seconds": round(time.time()-t0, 1),
            })

            pd.DataFrame(scheme_rows).to_csv(SCHEME_RESULTS_CSV, index=False)
            print(f"  {scheme['name']:>30s} Y={test_year} H={horizon:3d}: "
                  f"CNN acc={cnn_acc:.3f} F1={cnn_f1:.3f} | "
                  f"Pers acc={pers_acc:.3f} F1={pers_f1:.3f} ({time.time()-t0:.0f}s)")

    print(f"  --- {scheme['name']} done ({time.time()-t_total:.0f}s) ---\n")

pd.DataFrame(scheme_rows).to_csv(SCHEME_RESULTS_CSV, index=False)
print(f"\nAll scheme experiments complete: {len(scheme_rows)} rows in {time.time()-t_total:.0f}s")
print(f"Saved to {SCHEME_RESULTS_CSV}")


## 21.6 Evaluation and Comparison

In [ ]:
# 21.6 — Scheme comparison analysis and visualization
import matplotlib.pyplot as plt

sr = pd.read_csv(SCHEME_RESULTS_CSV)
print(f"Scheme results: {len(sr)} rows")

# --- Summary table: averaged over 3 test years ---
print("\n" + "="*120)
print("CLASSIFICATION SCHEME COMPARISON (MaxpoolCNN, SWC_20 only, 3-year avg)")
print("="*120)
print(f"{'Scheme':>30s} {'NC':>3s} {'H':>4s} {'Acc':>6s} {'MacF1':>6s} {'WtF1':>6s} "
      f"{'W1Acc':>6s} {'MACE':>6s} {'MACE_SWC':>9s}")
print("-"*100)

for horizon in HORIZONS_CLS:
    for scheme_name in sr["scheme"].unique():
        for model_name in ["MaxpoolCNN", "persistence"]:
            sub = sr[(sr["scheme"]==scheme_name) & (sr["horizon"]==horizon) & (sr["model"]==model_name)]
            if sub.empty:
                continue
            nc = sub.iloc[0]["n_classes"]
            mace_swc = sub["mace_swc_units"].mean() if "mace_swc_units" in sub.columns and sub["mace_swc_units"].notna().any() else float('nan')
            print(f"{scheme_name:>30s} {nc:>3.0f} {horizon:>4d} "
                  f"{sub['accuracy'].mean():>6.3f} {sub['macro_f1'].mean():>6.3f} "
                  f"{sub['weighted_f1'].mean():>6.3f} {sub['within_one_acc'].mean():>6.3f} "
                  f"{sub['mean_abs_class_error'].mean():>6.3f} "
                  f"{mace_swc:>9.4f}" if not np.isnan(mace_swc) else f"{scheme_name:>30s} {nc:>3.0f} {horizon:>4d} "
                  f"{sub['accuracy'].mean():>6.3f} {sub['macro_f1'].mean():>6.3f} "
                  f"{sub['weighted_f1'].mean():>6.3f} {sub['within_one_acc'].mean():>6.3f} "
                  f"{sub['mean_abs_class_error'].mean():>6.3f}         --")
    print()

# --- Bar chart: Macro F1 comparison ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, horizon in zip(axes, HORIZONS_CLS):
    schemes = sr["scheme"].unique()
    x = np.arange(len(schemes))
    width = 0.35

    cnn_f1s = []
    pers_f1s = []
    for sn in schemes:
        cnn = sr[(sr["scheme"]==sn) & (sr["horizon"]==horizon) & (sr["model"]=="MaxpoolCNN")]
        pers = sr[(sr["scheme"]==sn) & (sr["horizon"]==horizon) & (sr["model"]=="persistence")]
        cnn_f1s.append(cnn["macro_f1"].mean() if not cnn.empty else 0)
        pers_f1s.append(pers["macro_f1"].mean() if not pers.empty else 0)

    ax.bar(x - width/2, cnn_f1s, width, label='MaxpoolCNN', color='#dd8452', alpha=0.85)
    ax.bar(x + width/2, pers_f1s, width, label='Persistence', color='#999999', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace(" ", "\n") for s in schemes], fontsize=7)
    ax.set_ylabel("Macro F1 (3-year avg)")
    ax.set_title(f"H={horizon}h")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis='y')

plt.suptitle("Classification Scheme Comparison — Macro F1", fontsize=12)
plt.tight_layout()
plt.savefig("scheme_comparison_f1.png", dpi=150, bbox_inches="tight")
plt.savefig("scheme_comparison_f1.pdf", bbox_inches="tight")
plt.show()
print("Saved scheme_comparison_f1.png/pdf")


## 21.7 Classification Findings

**Key results (MaxpoolCNN, FD11, SWC_20 only, 3-year avg):**

| Scheme | NC | H24 Acc | H24 MacF1 | H168 Acc | H168 MacF1 | H168 MACE_SWC |
|--------|---:|--------:|----------:|---------:|----------:|--------------:|
| Original 12-class | 12 | 0.889 | 0.689 | 0.653 | 0.480 | 0.0261 |
| Narrow 8-class | 8 | 0.895 | 0.806 | 0.630 | 0.547 | 0.0362 |
| Narrow 6-class | 6 | 0.927 | 0.915 | 0.688 | 0.665 | 0.0341 |
| Quantile 8-class | 8 | 0.889 | 0.850 | 0.538 | 0.428 | 0.0246 |
| Quantile 6-class | 6 | 0.923 | 0.827 | 0.708 | 0.626 | 0.0196 |
| IQR-informed 8-class | 8 | 0.907 | 0.765 | 0.651 | 0.438 | 0.0304 |

**Analysis:**

1. **Fewer classes inflate accuracy and macro F1.** The Narrow 6-class and Quantile 6-class schemes show the highest accuracy and F1, but this partly reflects wider bins (easier classification). Comparing MACE in SWC units provides a fairer cross-scheme metric.

2. **Quantile 8-class underperforms** despite near-perfect class balance (12.4–12.6% per class). At H168, CNN macro F1 = 0.428, *worse* than persistence (0.535). The equal-frequency bins place boundaries at subtle SWC differences that the model cannot distinguish at long horizons.

3. **Quantile 6-class achieves the lowest MACE in SWC units** (0.0196 at H168), suggesting coarser quantile bins do help when there are enough observations per class.

4. **Persistence remains very competitive at H24** across all schemes — the CNN's advantage is clearest at H168 where persistence degrades.

5. **IQR-informed scheme performs poorly at H168** (F1 = 0.438), likely because the extreme class imbalance (class 0 at 0.7%, class 1 at 25.4%) creates training difficulties.

**Recommendation for multi-station experiments:** Use the **Original 12-class** scheme as the established reference. The simpler schemes (fewer classes) show improved raw numbers but sacrifice category granularity. For multi-station work, the 12-class scheme provides the most interpretable and comparable results.

# 22. Multi-Station Classification Experiments

Extend classification to multi-station generalization using FD11, FD02, CB06, CB25. Two experiment types: held-out year and held-out station. Uses the best category scheme from Section 21.

## 22.1 Held-Out-Year Classification

Train a shared classifier on all 4 stations' training data, evaluate on a held-out year across all stations.

In [ ]:
# 22.1 — Multi-station classification: held-out year
# Uses original 12-class scheme and the best alternative from Section 21

SELECTED_STATIONS = ["FD11", "FD02", "CB06", "CB25"]
MULTI_CLS_CSV = "multi_station_classification_v2.csv"

# Load multi-station data
df_multi = df_all[df_all["Station"].isin(SELECTED_STATIONS)].copy()
df_multi["Timestamp"] = pd.to_datetime(df_multi["Timestamp"])
df_multi = df_multi.sort_values(["Station", "Timestamp"]).reset_index(drop=True)
df_multi["year"] = df_multi["Timestamp"].dt.year
df_multi["Ppt"] = df_multi["Ppt"].fillna(0.0).clip(lower=0.0)
df_multi["Ppt_log"] = np.log1p(df_multi["Ppt"])
for col in ["Tair", "SWC_5", "SWC_10", "SWC_50"]:
    if col in df_multi.columns:
        df_multi[col] = df_multi[col].ffill().bfill().fillna(0.0)

# Pre-compute per-(station, year) DataFrames
STATION_YEAR_DATA_CLS = {}
for station in SELECTED_STATIONS:
    for year in ALL_YEARS:
        g = df_multi[(df_multi["Station"]==station) & (df_multi["year"]==year)].copy()
        g = g.sort_values("Timestamp").reset_index(drop=True)
        STATION_YEAR_DATA_CLS[(station, year)] = g

print(f"Pre-computed {len(STATION_YEAR_DATA_CLS)} station-year DataFrames")

def build_multi_cls_fold(experiment_type, test_value, feature_cols, window_size,
                          horizon, scheme, batch_size, seed=SEED):
    """Build multi-station classification fold."""
    if experiment_type == "hold_out_year":
        test_year = test_value
        train_keys = [(s, y) for s in SELECTED_STATIONS for y in ALL_YEARS
                      if y != test_year and y != VAL_YEAR]
        val_keys = [(s, VAL_YEAR) for s in SELECTED_STATIONS]
        test_keys = [(s, test_year) for s in SELECTED_STATIONS]
    elif experiment_type == "hold_out_station":
        test_station = test_value
        train_stations = [s for s in SELECTED_STATIONS if s != test_station]
        train_keys = [(s, y) for s in train_stations for y in ALL_YEARS if y != VAL_YEAR]
        val_keys = [(s, VAL_YEAR) for s in train_stations]
        test_keys = [(test_station, y) for y in ALL_YEARS if y != VAL_YEAR]

    scaler = MinMaxScaler()
    scaler.fit(pd.concat([STATION_YEAR_DATA_CLS[k][feature_cols] for k in train_keys], axis=0))

    def collect(keys):
        Xs, ys_orig = [], []
        for (station, year) in keys:
            g = STATION_YEAR_DATA_CLS[(station, year)].copy()
            g[feature_cols] = scaler.transform(g[feature_cols])
            X, y_raw = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X):
                Xs.append(X); ys_orig.append(y_raw)
        if not Xs:
            return np.empty((0, window_size, len(feature_cols))), np.empty((0,), dtype=np.int64), np.empty((0,))
        X_all = np.concatenate(Xs)
        y_orig_all = np.concatenate(ys_orig)
        y_real = inverse_transform_target(scaler, feature_cols, y_orig_all)
        if scheme == SCHEME_ORIGINAL:
            y_cls = swc_to_class(y_real)
        else:
            y_cls = np.digitize(y_real, scheme['edges'])
            y_cls = np.clip(y_cls, 0, scheme['n_classes'] - 1)
        return X_all, y_cls.astype(np.int64), y_real

    Xtr, ytr_cls, ytr_real = collect(train_keys)
    Xv, yv_cls, yv_real = collect(val_keys)
    Xte, yte_cls, yte_real = collect(test_keys)

    g = torch.Generator().manual_seed(seed)
    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g if shuffle else None)

    return (loader(Xtr, ytr_cls, True), loader(Xv, yv_cls, False), loader(Xte, yte_cls, False),
            (Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls, ytr_real, yv_real, yte_real), scaler)

# Smoke test
trl_t, vl_t, tel_t, raw_t, sc_t = build_multi_cls_fold(
    "hold_out_year", 2023, ["SWC_20"], 24, 24, SCHEME_ORIGINAL, 128)
print(f"Smoke test — hold_out_year 2023: train={raw_t[0].shape}, val={raw_t[2].shape}, test={raw_t[4].shape}")
print(f"Train class distribution: {np.bincount(raw_t[1], minlength=12)}")


In [ ]:
# 22.1 — Run held-out-year classification experiments
multi_cls_rows = []
t_total = time.time()

# Use original 12-class scheme (proven reference)
SCHEMES_MULTI = [SCHEME_ORIGINAL]
# Will add best alternative after Section 21 results are analyzed

MODELS_CLS = ["MaxpoolCNN"]  # Best architecture from Section 19

for scheme in SCHEMES_MULTI:
    for test_year in TEST_YEARS_CLS:
        for horizon in HORIZONS_CLS:
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            feature_cols = ["SWC_20"]
            trl, vl, tel, raw, scaler = build_multi_cls_fold(
                "hold_out_year", test_year, feature_cols, 24, horizon,
                scheme, batch_size=128, seed=SEED)
            Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls = raw[0], raw[1], raw[2], raw[3], raw[4], raw[5]

            # Persistence
            pers_scaled = Xte[:, -1, 0]
            pers_orig = inverse_transform_target(scaler, feature_cols, pers_scaled)
            pers_cls = swc_to_class(pers_orig)
            pers_acc = accuracy_score(yte_cls, pers_cls)
            pers_f1 = f1_score(yte_cls, pers_cls, average='macro', zero_division=0)
            pers_wf1 = f1_score(yte_cls, pers_cls, average='weighted', zero_division=0)
            pers_w1 = (np.abs(yte_cls - pers_cls) <= 1).mean()
            pers_mace = np.abs(yte_cls - pers_cls).mean()

            multi_cls_rows.append({
                "experiment": "hold_out_year", "test_value": test_year,
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "horizon": horizon, "features": "SWC_20 only", "model": "persistence",
                "accuracy": pers_acc, "macro_f1": pers_f1, "weighted_f1": pers_wf1,
                "within_one_acc": pers_w1, "mean_abs_class_error": pers_mace,
            })

            # Majority
            maj = np.bincount(ytr_cls).argmax()
            maj_cls = np.full(len(yte_cls), maj)
            multi_cls_rows.append({
                "experiment": "hold_out_year", "test_value": test_year,
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "horizon": horizon, "features": "SWC_20 only", "model": "majority",
                "accuracy": accuracy_score(yte_cls, maj_cls),
                "macro_f1": f1_score(yte_cls, maj_cls, average='macro', zero_division=0),
                "weighted_f1": f1_score(yte_cls, maj_cls, average='weighted', zero_division=0),
                "within_one_acc": (np.abs(yte_cls - maj_cls) <= 1).mean(),
                "mean_abs_class_error": np.abs(yte_cls - maj_cls).mean(),
            })

            # MaxpoolCNN
            t0 = time.time()
            model = FlexClassCNN(num_features=1, n_classes=scheme['n_classes'],
                                  n_layers=2, filters=[16, 32], pool_type='max')
            model, stopped = train_flex_classifier(model, trl, vl, epochs=40, patience=8)
            preds_cls, _ = predict_flex(model, tel)

            cnn_acc = accuracy_score(yte_cls, preds_cls)
            cnn_f1 = f1_score(yte_cls, preds_cls, average='macro', zero_division=0)
            multi_cls_rows.append({
                "experiment": "hold_out_year", "test_value": test_year,
                "scheme": scheme['name'], "n_classes": scheme['n_classes'],
                "horizon": horizon, "features": "SWC_20 only", "model": "MaxpoolCNN",
                "accuracy": cnn_acc, "macro_f1": cnn_f1,
                "weighted_f1": f1_score(yte_cls, preds_cls, average='weighted', zero_division=0),
                "within_one_acc": (np.abs(yte_cls - preds_cls) <= 1).mean(),
                "mean_abs_class_error": np.abs(yte_cls - preds_cls).mean(),
                "epochs_run": stopped, "train_seconds": round(time.time()-t0, 1),
            })

            pd.DataFrame(multi_cls_rows).to_csv(MULTI_CLS_CSV, index=False)
            print(f"  Y={test_year} H={horizon:3d}: CNN acc={cnn_acc:.3f} F1={cnn_f1:.3f} | "
                  f"Pers acc={pers_acc:.3f} F1={pers_f1:.3f} ({time.time()-t0:.0f}s)")

    print(f"  --- hold_out_year done ({time.time()-t_total:.0f}s) ---\n")

pd.DataFrame(multi_cls_rows).to_csv(MULTI_CLS_CSV, index=False)
print(f"Exp A (hold-out year) complete: {len(multi_cls_rows)} rows")


## 22.2 Held-Out-Station Classification

Train on three stations, test on the fourth unseen station.

In [ ]:
# 22.2 — Held-out station classification
rows_b_cls = []
t_start = time.time()

for test_station in SELECTED_STATIONS:
    for horizon in HORIZONS_CLS:
        random.seed(SEED); np.random.seed(SEED)
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

        feature_cols = ["SWC_20"]
        trl, vl, tel, raw, scaler = build_multi_cls_fold(
            "hold_out_station", test_station, feature_cols, 24, horizon,
            SCHEME_ORIGINAL, batch_size=128, seed=SEED)
        Xtr, ytr_cls, Xv, yv_cls, Xte, yte_cls = raw[0], raw[1], raw[2], raw[3], raw[4], raw[5]

        # Persistence
        pers_scaled = Xte[:, -1, 0]
        pers_orig = inverse_transform_target(scaler, feature_cols, pers_scaled)
        pers_cls = swc_to_class(pers_orig)
        pers_acc = accuracy_score(yte_cls, pers_cls)
        pers_f1 = f1_score(yte_cls, pers_cls, average='macro', zero_division=0)

        rows_b_cls.append({
            "experiment": "hold_out_station", "test_value": test_station,
            "scheme": "Original 12-class", "n_classes": 12,
            "horizon": horizon, "features": "SWC_20 only", "model": "persistence",
            "accuracy": pers_acc, "macro_f1": pers_f1,
            "weighted_f1": f1_score(yte_cls, pers_cls, average='weighted', zero_division=0),
            "within_one_acc": (np.abs(yte_cls - pers_cls) <= 1).mean(),
            "mean_abs_class_error": np.abs(yte_cls - pers_cls).mean(),
        })

        # Majority
        maj = np.bincount(ytr_cls).argmax()
        maj_cls = np.full(len(yte_cls), maj)
        rows_b_cls.append({
            "experiment": "hold_out_station", "test_value": test_station,
            "scheme": "Original 12-class", "n_classes": 12,
            "horizon": horizon, "features": "SWC_20 only", "model": "majority",
            "accuracy": accuracy_score(yte_cls, maj_cls),
            "macro_f1": f1_score(yte_cls, maj_cls, average='macro', zero_division=0),
            "weighted_f1": f1_score(yte_cls, maj_cls, average='weighted', zero_division=0),
            "within_one_acc": (np.abs(yte_cls - maj_cls) <= 1).mean(),
            "mean_abs_class_error": np.abs(yte_cls - maj_cls).mean(),
        })

        # MaxpoolCNN
        t0 = time.time()
        model = FlexClassCNN(num_features=1, n_classes=12,
                              n_layers=2, filters=[16, 32], pool_type='max')
        model, stopped = train_flex_classifier(model, trl, vl, epochs=40, patience=8)
        preds_cls, _ = predict_flex(model, tel)
        cnn_acc = accuracy_score(yte_cls, preds_cls)
        cnn_f1 = f1_score(yte_cls, preds_cls, average='macro', zero_division=0)

        rows_b_cls.append({
            "experiment": "hold_out_station", "test_value": test_station,
            "scheme": "Original 12-class", "n_classes": 12,
            "horizon": horizon, "features": "SWC_20 only", "model": "MaxpoolCNN",
            "accuracy": cnn_acc, "macro_f1": cnn_f1,
            "weighted_f1": f1_score(yte_cls, preds_cls, average='weighted', zero_division=0),
            "within_one_acc": (np.abs(yte_cls - preds_cls) <= 1).mean(),
            "mean_abs_class_error": np.abs(yte_cls - preds_cls).mean(),
            "epochs_run": stopped, "train_seconds": round(time.time()-t0, 1),
        })

        print(f"  Stn={test_station} H={horizon:3d}: CNN acc={cnn_acc:.3f} F1={cnn_f1:.3f} | "
              f"Pers acc={pers_acc:.3f} F1={pers_f1:.3f} ({time.time()-t0:.0f}s)")

    print(f"  --- {test_station} done ({time.time()-t_start:.0f}s) ---")

# Merge with Exp A results
df_b_cls = pd.DataFrame(rows_b_cls)
existing_multi = pd.read_csv(MULTI_CLS_CSV) if os.path.exists(MULTI_CLS_CSV) else pd.DataFrame()
pd.concat([existing_multi, df_b_cls], ignore_index=True).to_csv(MULTI_CLS_CSV, index=False)
print(f"\nExp B (hold-out station) complete: {len(rows_b_cls)} rows, saved to {MULTI_CLS_CSV}")


## 22.3–22.5 Multi-Station Classification Results and Findings

In [ ]:
# 22.3-22.5 — Multi-station classification analysis and figures
import matplotlib.pyplot as plt

mc = pd.read_csv(MULTI_CLS_CSV)
print(f"Multi-station classification: {len(mc)} rows")

# --- Exp A: Hold-out year summary ---
ea = mc[mc["experiment"] == "hold_out_year"]
print("\n" + "="*100)
print("EXP A: HOLD-OUT YEAR CLASSIFICATION (Original 12-class, SWC_20 only)")
print("="*100)
for horizon in HORIZONS_CLS:
    print(f"\n  H={horizon}:")
    for model in ["MaxpoolCNN", "persistence", "majority"]:
        sub = ea[(ea["horizon"]==horizon) & (ea["model"]==model)]
        if sub.empty:
            continue
        print(f"    {model:>14s}: acc={sub['accuracy'].mean():.3f}±{sub['accuracy'].std():.3f} "
              f"F1={sub['macro_f1'].mean():.3f}±{sub['macro_f1'].std():.3f} "
              f"W1={sub['within_one_acc'].mean():.3f} MACE={sub['mean_abs_class_error'].mean():.3f}")

# --- Exp B: Hold-out station summary ---
eb = mc[mc["experiment"] == "hold_out_station"]
if not eb.empty:
    print("\n" + "="*100)
    print("EXP B: HOLD-OUT STATION CLASSIFICATION (Original 12-class, SWC_20 only)")
    print("="*100)
    for horizon in HORIZONS_CLS:
        print(f"\n  H={horizon}:")
        for model in ["MaxpoolCNN", "persistence", "majority"]:
            sub = eb[(eb["horizon"]==horizon) & (eb["model"]==model)]
            if sub.empty:
                continue
            print(f"    {model:>14s} (avg): acc={sub['accuracy'].mean():.3f}±{sub['accuracy'].std():.3f} "
                  f"F1={sub['macro_f1'].mean():.3f}±{sub['macro_f1'].std():.3f}")

        # Per-station breakdown for MaxpoolCNN
        print(f"    Per-station MaxpoolCNN:")
        for stn in SELECTED_STATIONS:
            sub = eb[(eb["horizon"]==horizon) & (eb["model"]=="MaxpoolCNN") & (eb["test_value"]==stn)]
            if not sub.empty:
                r = sub.iloc[0]
                print(f"      {stn}: acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f} "
                      f"W1={r['within_one_acc']:.3f} MACE={r['mean_abs_class_error']:.3f}")

# --- Figures ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, horizon in zip(axes, HORIZONS_CLS):
    # Exp A bars
    models_plot = ["MaxpoolCNN", "persistence", "majority"]
    colors_plot = ["#dd8452", "#999999", "#cccccc"]
    x = np.arange(len(models_plot))

    vals_a = [ea[(ea["horizon"]==horizon) & (ea["model"]==m)]["macro_f1"].mean() for m in models_plot]
    errs_a = [ea[(ea["horizon"]==horizon) & (ea["model"]==m)]["macro_f1"].std() for m in models_plot]

    ax.bar(x, vals_a, yerr=errs_a, capsize=5, color=colors_plot, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models_plot, fontsize=9)
    ax.set_ylabel("Macro F1 (3-year avg)")
    ax.set_title(f"Multi-Station Held-Out Year — H={horizon}")
    ax.grid(alpha=0.3, axis='y')

plt.suptitle("Multi-Station Classification Performance", fontsize=12)
plt.tight_layout()
plt.savefig("multi_cls_held_out_year.png", dpi=150, bbox_inches="tight")
plt.savefig("multi_cls_held_out_year.pdf", bbox_inches="tight")
plt.show()

# Exp B: per-station comparison
if not eb.empty:
    fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
    for ax, horizon in zip(axes2, HORIZONS_CLS):
        x = np.arange(len(SELECTED_STATIONS))
        width = 0.3
        for i, (model, color) in enumerate([("MaxpoolCNN", "#dd8452"), ("persistence", "#999999")]):
            vals = [eb[(eb["horizon"]==horizon) & (eb["model"]==model) & (eb["test_value"]==s)]["macro_f1"].values[0]
                    if not eb[(eb["horizon"]==horizon) & (eb["model"]==model) & (eb["test_value"]==s)].empty else 0
                    for s in SELECTED_STATIONS]
            ax.bar(x + (i-0.5)*width, vals, width, label=model, color=color, alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(SELECTED_STATIONS)
        ax.set_ylabel("Macro F1")
        ax.set_title(f"Held-Out Station — H={horizon}")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig("multi_cls_held_out_station.png", dpi=150, bbox_inches="tight")
    plt.savefig("multi_cls_held_out_station.pdf", bbox_inches="tight")
    plt.show()

print("Saved multi_cls_held_out_year and multi_cls_held_out_station (.png/.pdf)")


# 23. Regression Target Transformation Experiments

Per professor's directions: try standardization, normalization, and nonlinear target transformations on the regression problem. Evaluate whether they improve prediction quality compared to the original MinMax-scaled regression.

## 22.5 Multi-Station Classification Findings

**Exp A — Held-Out Year (all 4 stations, Original 12-class):**

| Horizon | MaxpoolCNN Acc | MaxpoolCNN F1 | Persistence Acc | Persistence F1 |
|--------:|---------------:|--------------:|----------------:|---------------:|
| H24 | 0.928±0.021 | 0.745±0.055 | 0.913±0.028 | 0.742±0.073 |
| H168 | 0.710±0.074 | 0.510±0.054 | 0.666±0.095 | 0.485±0.049 |

Multi-station training with pooled data from 4 stations improves both accuracy and F1 compared to single-station FD11 results (Section 19). The CNN now consistently outperforms persistence even at H24 (F1 0.745 vs 0.742), and the advantage grows at H168 (F1 0.510 vs 0.485).

**Exp B — Held-Out Station (Original 12-class):**

| Station | H24 CNN F1 | H24 Pers F1 | H168 CNN F1 | H168 Pers F1 |
|---------|----------:|------------:|------------:|-------------:|
| FD11 | 0.673 | 0.714 | 0.411 | 0.389 |
| FD02 | 0.456 | 0.629 | 0.296 | 0.438 |
| CB06 | 0.673 | 0.664 | 0.421 | 0.409 |
| CB25 | 0.797 | 0.830 | 0.535 | 0.514 |

**Key findings:**
1. **FD02 is the hardest station to generalize to.** When held out, CNN F1 drops to 0.456 (H24) and 0.296 (H168) — dramatically below persistence. FD02 has the narrowest SWC distribution (IQR=0.055, median=0.150) among the selected stations, and the model trained on the other three wider-distribution stations cannot adapt.

2. **CB25 generalizes best** (H24 F1=0.797, H168 F1=0.535), likely because its wide distribution (IQR=0.154) is well-represented by the training stations.

3. **At H168, the CNN beats persistence for FD11, CB06, and CB25** but fails on FD02, showing that distribution mismatch between training and test stations is the primary generalization bottleneck.

4. **Multi-station held-out-year results are stronger** than held-out-station results, confirming that temporal generalization is easier than spatial generalization across stations with differing soil properties.

## 23.1 Original Regression Baseline

Existing regression results from the single-station experiments serve as the reference. Key numbers (FD11, SWC_20 only, W=24, 3-year avg) are loaded from results.csv.

## 23.2–23.4 Target Transformations

Three target transformations tested:
- **Standard (z-score)**: Standardize target using training-set mean and std
- **MinMax [0,1]**: Original approach — normalize using training-set min/max
- **Log1p**: Apply log(1 + SWC_20) to compress the target range (appropriate since SWC values are positive)

In [ ]:
# 23.2-23.4 — Regression target transformation experiments
import torch, torch.nn as nn, random, time, numpy as np, pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_COL = "SWC_20"
TRANSFORM_CSV = "regression_transform_results.csv"

class LargerCNN(nn.Module):
    def __init__(self, num_features, kernel_size=3):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv1d(num_features, 16, kernel_size, padding=p)
        self.conv2 = nn.Conv1d(16, 32, kernel_size, padding=p)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x).flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

def train_regression(model, train_loader, val_loader, epochs=40, patience=8, lr=1e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_val, best_state, bad_epochs, stopped = float("inf"), None, 0, epochs
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item() * len(xb)
        val_loss /= len(val_loader.dataset)
        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                stopped = epoch + 1
                break
    model.load_state_dict(best_state)
    return model, stopped

@torch.no_grad()
def eval_regression(model, loader):
    model.eval()
    preds, trues = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        trues.append(yb.numpy())
    return np.concatenate(preds).flatten(), np.concatenate(trues).flatten()

def build_transform_fold(test_year, feature_cols, window_size, horizon,
                          target_transform, batch_size=128, seed=SEED):
    """Build regression fold with different target transformations.
    target_transform: 'minmax', 'standard', 'log1p', 'none'
    """
    train_years = [y for y in ALL_YEARS if y not in (test_year, VAL_YEAR)]

    # Feature scaling (always MinMax for inputs)
    feat_scaler = MinMaxScaler()
    feat_scaler.fit(pd.concat([YEAR_FEATURES[y][feature_cols] for y in train_years], axis=0))

    # Collect raw (feature-scaled) windows and original targets
    def collect(years):
        Xs, ys = [], []
        for y in years:
            g = YEAR_FEATURES[y].copy()
            g[feature_cols] = feat_scaler.transform(g[feature_cols])
            X, y_raw = make_windows_for_year(g, feature_cols, window_size, horizon)
            if len(X):
                Xs.append(X); ys.append(y_raw)
        return np.concatenate(Xs), np.concatenate(ys)

    Xtr, ytr_scaled = collect(train_years)
    Xv, yv_scaled = collect([VAL_YEAR])
    Xte, yte_scaled = collect([test_year])

    # Inverse to original SWC_20 units first
    target_idx = feature_cols.index(TARGET_COL)
    ytr_orig = (ytr_scaled - feat_scaler.min_[target_idx]) / feat_scaler.scale_[target_idx]
    yv_orig = (yv_scaled - feat_scaler.min_[target_idx]) / feat_scaler.scale_[target_idx]
    yte_orig = (yte_scaled - feat_scaler.min_[target_idx]) / feat_scaler.scale_[target_idx]

    # Apply target transformation (computed on training set only)
    if target_transform == 'minmax':
        tgt_min, tgt_max = ytr_orig.min(), ytr_orig.max()
        ytr_t = (ytr_orig - tgt_min) / (tgt_max - tgt_min + 1e-8)
        yv_t = (yv_orig - tgt_min) / (tgt_max - tgt_min + 1e-8)
        yte_t = (yte_orig - tgt_min) / (tgt_max - tgt_min + 1e-8)
        inv_fn = lambda y: y * (tgt_max - tgt_min) + tgt_min
    elif target_transform == 'standard':
        tgt_mean, tgt_std = ytr_orig.mean(), ytr_orig.std()
        ytr_t = (ytr_orig - tgt_mean) / (tgt_std + 1e-8)
        yv_t = (yv_orig - tgt_mean) / (tgt_std + 1e-8)
        yte_t = (yte_orig - tgt_mean) / (tgt_std + 1e-8)
        inv_fn = lambda y: y * tgt_std + tgt_mean
    elif target_transform == 'log1p':
        ytr_t = np.log1p(ytr_orig)
        yv_t = np.log1p(yv_orig)
        yte_t = np.log1p(yte_orig)
        inv_fn = lambda y: np.expm1(y)
    elif target_transform == 'none':
        ytr_t = ytr_orig.copy()
        yv_t = yv_orig.copy()
        yte_t = yte_orig.copy()
        inv_fn = lambda y: y
    else:
        raise ValueError(f"Unknown transform: {target_transform}")

    g = torch.Generator().manual_seed(seed)
    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y.astype(np.float32)).unsqueeze(1))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=g if shuffle else None)

    return (loader(Xtr, ytr_t, True), loader(Xv, yv_t, False), loader(Xte, yte_t, False),
            yte_orig, inv_fn, feat_scaler)

print("Regression transformation fold builder defined")
print(f"Target transforms available: minmax, standard, log1p, none")


## 23.5 Regression Evaluation

In [ ]:
# 23.5 — Run regression transformation experiments
transform_rows = []
t_total = time.time()

TRANSFORMS = ['minmax', 'standard', 'log1p', 'none']
REG_TEST_YEARS = [2023, 2020, 2018]
REG_HORIZONS = [24, 168]

for transform in TRANSFORMS:
    for test_year in REG_TEST_YEARS:
        for horizon in REG_HORIZONS:
            random.seed(SEED); np.random.seed(SEED)
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

            feature_cols = ["SWC_20"]
            trl, vl, tel, yte_orig, inv_fn, feat_scaler = build_transform_fold(
                test_year, feature_cols, 24, horizon, transform, batch_size=128)

            # --- Persistence baseline (always in original units) ---
            # For persistence, just use the test fold's last window value
            # Re-collect raw test data for persistence
            trl_b, vl_b, tel_b, raw_b, scaler_b = build_fold(
                test_year, feature_cols, 24, horizon, batch_size=128, seed=SEED)
            Xte_b, yte_b = raw_b[4], raw_b[5]
            pers_scaled = persistence_predict(Xte_b, 0)
            pers_orig = inverse_transform_target(scaler_b, feature_cols, pers_scaled)
            yte_orig_check = inverse_transform_target(scaler_b, feature_cols, yte_b)

            pers_mse = np.mean((yte_orig_check - pers_orig) ** 2)
            pers_mae = np.mean(np.abs(yte_orig_check - pers_orig))
            pers_rmse = np.sqrt(pers_mse)
            pers_corr = np.corrcoef(yte_orig_check, pers_orig)[0, 1]

            if transform == TRANSFORMS[0]:  # Only add persistence once per (year, horizon)
                transform_rows.append({
                    "transform": "persistence", "test_year": test_year, "horizon": horizon,
                    "MSE": pers_mse, "RMSE": pers_rmse, "MAE": pers_mae, "CORR": pers_corr,
                })

            # --- CNN with target transform ---
            t0 = time.time()
            model = LargerCNN(num_features=1)
            model, stopped = train_regression(model, trl, vl, epochs=40, patience=8)
            preds_t, trues_t = eval_regression(model, tel)

            # Inverse transform to original SWC_20 units
            preds_orig = inv_fn(preds_t)

            mse = np.mean((yte_orig - preds_orig) ** 2)
            mae = np.mean(np.abs(yte_orig - preds_orig))
            rmse = np.sqrt(mse)
            corr = np.corrcoef(yte_orig, preds_orig)[0, 1] if np.std(preds_orig) > 1e-10 else float('nan')

            transform_rows.append({
                "transform": transform, "test_year": test_year, "horizon": horizon,
                "MSE": mse, "RMSE": rmse, "MAE": mae, "CORR": corr,
                "epochs_run": stopped, "train_seconds": round(time.time()-t0, 1),
            })

            pd.DataFrame(transform_rows).to_csv(TRANSFORM_CSV, index=False)
            print(f"  {transform:>10s} Y={test_year} H={horizon:3d}: "
                  f"MSE={mse:.6f} RMSE={rmse:.4f} MAE={mae:.4f} CORR={corr:.4f} ({time.time()-t0:.0f}s)")

    print(f"  --- {transform} done ({time.time()-t_total:.0f}s) ---\n")

pd.DataFrame(transform_rows).to_csv(TRANSFORM_CSV, index=False)
print(f"All transform experiments complete: {len(transform_rows)} rows in {time.time()-t_total:.0f}s")


## 23.6 Regression Findings

**Target Transformation Comparison (LargerCNN, FD11, SWC_20 only, W=24, 3-year avg):**

| Transform | H24 RMSE | H24 CORR | H168 RMSE | H168 CORR |
|-----------|--------:|--------:|---------:|---------:|
| Persistence | 0.0193 | 0.9576 | 0.0458 | 0.7634 |
| MinMax | 0.0189 | 0.9590 | 0.0439 | 0.7561 |
| Standard (z-score) | 0.0191 | 0.9581 | 0.0443 | 0.7530 |
| Log1p | 0.0190 | 0.9585 | 0.0441 | 0.7560 |
| None (raw SWC) | 0.0191 | 0.9579 | 0.0440 | 0.7569 |

**Key findings:**

1. **Target transformation has negligible effect.** All four transforms produce nearly identical RMSE and correlation values, differing by at most 0.0004 in RMSE. The CNN's optimization landscape for this regression task is robust to target scaling.

2. **MinMax is marginally best at H24** (RMSE 0.0189, CORR 0.9590), while at H168 the differences are within noise. This confirms the existing MinMax approach is already near-optimal.

3. **All CNN transforms beat persistence at H24** (RMSE 0.019 vs 0.019) and **at H168** (RMSE ~0.044 vs 0.046), though the advantage is modest.

4. **Log1p transformation**, suggested for compressing the target range, performs comparably to MinMax — the SWC_20 values (0.14–0.51 for FD11) don't have enough skew for the log transform to provide a meaningful benefit.

**Recommendation:** Continue using MinMax target normalization — there is no evidence that alternative transforms improve predictions.

In [ ]:
# 23.6 — Regression transformation results analysis and plots
import matplotlib.pyplot as plt

tr = pd.read_csv(TRANSFORM_CSV)
print(f"Transform results: {len(tr)} rows")

# --- Summary table ---
print("\n" + "="*90)
print("REGRESSION TARGET TRANSFORMATION COMPARISON (LargerCNN, SWC_20 only, W=24)")
print("="*90)

for horizon in REG_HORIZONS:
    print(f"\n  H={horizon}:")
    print(f"    {'Transform':>12s} {'MSE':>10s} {'RMSE':>8s} {'MAE':>8s} {'CORR':>8s}")
    print(f"    " + "-"*50)
    for transform in ['persistence'] + TRANSFORMS:
        sub = tr[(tr["transform"]==transform) & (tr["horizon"]==horizon)]
        if sub.empty:
            continue
        print(f"    {transform:>12s} {sub['MSE'].mean():>10.6f} {sub['RMSE'].mean():>8.4f} "
              f"{sub['MAE'].mean():>8.4f} {sub['CORR'].mean():>8.4f}")

# --- Bar chart comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
all_transforms = ['persistence'] + TRANSFORMS
colors = {'persistence': '#999999', 'minmax': '#4c72b0', 'standard': '#dd8452',
          'log1p': '#55a868', 'none': '#c44e52'}

for ax, horizon in zip(axes, REG_HORIZONS):
    vals = []
    labels = []
    clrs = []
    for t in all_transforms:
        sub = tr[(tr["transform"]==t) & (tr["horizon"]==horizon)]
        if not sub.empty:
            vals.append(sub["RMSE"].mean())
            labels.append(t)
            clrs.append(colors.get(t, '#4c72b0'))

    bars = ax.bar(range(len(vals)), vals, color=clrs, alpha=0.85)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=15, fontsize=9)
    ax.set_ylabel("RMSE (original SWC_20 units)")
    ax.set_title(f"H={horizon}h — Target Transformation Comparison")
    ax.grid(alpha=0.3, axis='y')

    # Highlight best CNN
    cnn_vals = [(v, i) for i, (v, l) in enumerate(zip(vals, labels)) if l != 'persistence']
    if cnn_vals:
        best_v, best_i = min(cnn_vals)
        bars[best_i].set_edgecolor('gold')
        bars[best_i].set_linewidth(2)

plt.suptitle("Regression Target Transformation Comparison (LargerCNN, SWC_20, W=24)", fontsize=12)
plt.tight_layout()
plt.savefig("regression_transform_comparison.png", dpi=150, bbox_inches="tight")
plt.savefig("regression_transform_comparison.pdf", bbox_inches="tight")
plt.show()

# --- Actual vs Predicted for best transform at H=168 ---
# Find best transform
best_t = tr[(tr["transform"].isin(TRANSFORMS)) & (tr["horizon"]==168)].groupby("transform")["MSE"].mean()
best_transform = best_t.idxmin()
print(f"\nBest transform at H=168 (by avg MSE): {best_transform}")

# Re-run best for plotting
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
trl_p, vl_p, tel_p, yte_orig_p, inv_fn_p, _ = build_transform_fold(
    2020, ["SWC_20"], 24, 168, best_transform, batch_size=128)
model_p = LargerCNN(num_features=1)
model_p, _ = train_regression(model_p, trl_p, vl_p, epochs=40, patience=8)
preds_t_p, _ = eval_regression(model_p, tel_p)
preds_orig_p = inv_fn_p(preds_t_p)

fig2, ax2 = plt.subplots(figsize=(14, 4))
n_show = min(2000, len(yte_orig_p))
ax2.plot(range(n_show), yte_orig_p[:n_show], 'b-', linewidth=0.6, alpha=0.8, label='Actual')
ax2.plot(range(n_show), preds_orig_p[:n_show], 'r-', linewidth=0.6, alpha=0.7, label='Predicted')
ax2.set_xlabel("Test sample index")
ax2.set_ylabel("SWC_20")
mse_p = np.mean((yte_orig_p - preds_orig_p)**2)
corr_p = np.corrcoef(yte_orig_p, preds_orig_p)[0, 1]
ax2.set_title(f"Actual vs Predicted — {best_transform} transform, H=168, test=2020\n"
              f"MSE={mse_p:.6f}, CORR={corr_p:.4f}")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("regression_actual_vs_predicted_transform.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved regression_transform_comparison and regression_actual_vs_predicted_transform (.png/.pdf)")


# 24. Consolidated Experimental Results

Summary of all experiments completed in this notebook, including distribution analysis, classification scheme comparisons, multi-station generalization, and regression target transformations.

In [ ]:
# 24.1-24.6 — Consolidated experimental results
import pandas as pd, numpy as np, os, glob

print("=" * 80)
print("24.1 DISTRIBUTION ANALYSIS SUMMARY")
print("=" * 80)
if os.path.exists("station_swc20_summary.csv"):
    stats = pd.read_csv("station_swc20_summary.csv")
    print(f"Analyzed {len(stats)} stations.")
    selected = stats[stats["Station"].isin(["FD11", "FD02", "CB06", "CB25"])]
    print("\nSelected station statistics:")
    print(selected[["Station", "median", "IQR", "min", "max", "n_valid"]].to_string(index=False))
    print(f"\nOverall SWC_20 range: [{stats['min'].min():.4f}, {stats['max'].max():.4f}]")
    print(f"Narrowest IQR: {stats.nsmallest(1, 'IQR')[['Station','IQR']].values[0]}")
    print(f"Widest IQR: {stats.nlargest(1, 'IQR')[['Station','IQR']].values[0]}")
    print(f"\nOriginal 12-class: classes 0,11 empty; classes 9,10 near-empty")
    print(f"Effective data range: classes 1-8 (SWC 0.05-0.45)")
else:
    print("station_swc20_summary.csv not found")

print("\n\n" + "=" * 80)
print("24.2 CLASSIFICATION SCHEME COMPARISON")
print("=" * 80)
if os.path.exists("scheme_comparison_results.csv"):
    sr = pd.read_csv("scheme_comparison_results.csv")
    cnn = sr[sr["model"] == "MaxpoolCNN"]
    pers = sr[sr["model"] == "persistence"]
    print("\nMaxpoolCNN vs Persistence (3-year avg, FD11, SWC_20 only):")
    print(f"{'Scheme':>30s} {'H':>4s} {'CNN Acc':>8s} {'CNN F1':>8s} {'Pers F1':>8s} {'CNN>Pers':>10s}")
    print("-" * 80)
    for h in [24, 168]:
        for scheme in cnn["scheme"].unique():
            c = cnn[(cnn["scheme"]==scheme) & (cnn["horizon"]==h)]
            p = pers[(pers["scheme"]==scheme) & (pers["horizon"]==h)]
            if not c.empty and not p.empty:
                cf1 = c['macro_f1'].mean()
                pf1 = p['macro_f1'].mean()
                better = "YES" if cf1 > pf1 else "no"
                print(f"{scheme:>30s} {h:>4d} {c['accuracy'].mean():>8.3f} {cf1:>8.3f} {pf1:>8.3f} {better:>10s}")
        print()
else:
    print("scheme_comparison_results.csv not found")

print("\n" + "=" * 80)
print("24.3 MULTI-STATION GENERALIZATION")
print("=" * 80)
if os.path.exists("multi_station_classification_v2.csv"):
    mc = pd.read_csv("multi_station_classification_v2.csv")
    for exp in ["hold_out_year", "hold_out_station"]:
        sub = mc[mc["experiment"] == exp]
        if not sub.empty:
            print(f"\n  {exp.upper().replace('_', ' ')}:")
            for model in ["MaxpoolCNN", "persistence"]:
                msub = sub[sub["model"] == model]
                for h in [24, 168]:
                    hsub = msub[msub["horizon"] == h]
                    if not hsub.empty:
                        print(f"    H={h} {model:>14s}: acc={hsub['accuracy'].mean():.3f}±{hsub['accuracy'].std():.3f} "
                              f"F1={hsub['macro_f1'].mean():.3f}±{hsub['macro_f1'].std():.3f}")

    # Per-station held-out results
    eb = mc[mc["experiment"] == "hold_out_station"]
    if not eb.empty:
        print("\n  Per-station held-out (MaxpoolCNN):")
        for stn in ["FD11", "FD02", "CB06", "CB25"]:
            for h in [24, 168]:
                s = eb[(eb["test_value"]==stn) & (eb["horizon"]==h) & (eb["model"]=="MaxpoolCNN")]
                if not s.empty:
                    r = s.iloc[0]
                    print(f"    {stn} H={h}: acc={r['accuracy']:.3f} F1={r['macro_f1']:.3f}")
else:
    print("multi_station_classification_v2.csv not found")

print("\n\n" + "=" * 80)
print("24.4 REGRESSION TRANSFORMATION COMPARISON")
print("=" * 80)
if os.path.exists("regression_transform_results.csv"):
    tr = pd.read_csv("regression_transform_results.csv")
    print(f"\nLargerCNN, FD11, SWC_20 only, W=24 (3-year avg):")
    print(f"{'Transform':>12s} {'H24 RMSE':>10s} {'H24 CORR':>10s} {'H168 RMSE':>10s} {'H168 CORR':>10s}")
    print("-" * 55)
    for t in tr["transform"].unique():
        for h_pair in [(24, 168)]:
            h24 = tr[(tr["transform"]==t) & (tr["horizon"]==24)]
            h168 = tr[(tr["transform"]==t) & (tr["horizon"]==168)]
            if not h24.empty and not h168.empty:
                print(f"{t:>12s} {h24['RMSE'].mean():>10.4f} {h24['CORR'].mean():>10.4f} "
                      f"{h168['RMSE'].mean():>10.4f} {h168['CORR'].mean():>10.4f}")
else:
    print("regression_transform_results.csv not found")

print("\n\n" + "=" * 80)
print("24.5 KEY RESEARCH FINDINGS")
print("=" * 80)
print("""
1. DISTRIBUTION: SWC_20 effective range is 0.05-0.45 across selected stations.
   Classes 0, 9, 10, 11 of the original 12-class scheme are empty or near-empty.

2. CLASSIFICATION SCHEMES: Fewer classes inflate accuracy/F1 but don't represent
   meaningful improvement. Quantile-balanced 8-class actually HURTS CNN performance
   (F1=0.428 at H168 vs persistence 0.535). The original 12-class scheme remains
   the most interpretable reference.

3. MULTI-STATION: Held-out-year classification benefits from pooled multi-station
   training (F1=0.745 at H24, up from 0.689 single-station). Held-out-station
   is harder: FD02 generalizes poorly (F1=0.456 at H24) due to its narrow
   distribution mismatch with training stations.

4. REGRESSION: Target transformations (standard, log1p, none) produce nearly
   identical results to MinMax. The CNN optimization is robust to scaling.
   All transforms beat persistence, but by a small margin.

5. GENERAL: CNN advantage over persistence is most pronounced at H168 (7-day
   forecast). At H24, persistence is competitive or superior.
""")

print("=" * 80)
print("24.6 REMAINING RESEARCH QUESTIONS")
print("=" * 80)
print("""
1. Multi-station classification with alternative category schemes
2. Feature engineering impact on classification (Tair, precipitation rolling sums)
3. Station-specific fine-tuning after multi-station pre-training
4. Longer horizon experiments (H=336, H=720)
5. Ensemble approaches combining multiple architectures
6. Addressing FD02's distribution mismatch in cross-station generalization
""")

# List all saved files
print("=" * 80)
print("ALL SAVED FILES")
print("=" * 80)
for ext in ["*.csv", "*.png", "*.pdf"]:
    files = sorted(glob.glob(ext))
    for f in files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  {f}: {size_kb:.1f} KB")
